# ADOT Pavement Management Simulation

This notebook consolidates the computational logic from three policy-specific source notebooks without modifying those source files:

1. **Worst-first** — ranks candidate segments by a composite pavement-distress score.
2. **Preservation** — ranks treatments by area-under-the-performance-curve benefit divided by agency cost.
3. **Preservation (Considered AADT)** — applies the same benefit-cost calculation with additional traffic weighting.

The workflow loads and validates the pavement network, builds performance projections, applies the ADOT treatment decision tree, runs Monte Carlo simulations, evaluates convergence, performs multi-budget analysis, and compares the three prioritization strategies.

Set the `RUN_*` flags in the configuration cell, then run the notebook from top to bottom. The comparison section requires results from all three strategy blocks.


In [ ]:
# ============================================================
# USER CONFIGURATION
# Edit the values in this cell before running the notebook.
# All other cells can be executed without modification.
# ============================================================

# ── Prioritization strategies (1 = run, 0 = skip) ────────────────────────────
RUN_WORST_FIRST       = 1   # Worst-first: fund the most distressed segments first
RUN_BENEFIT_COST      = 1   # Preservation: rank by benefit-cost ratio (AUC method)
RUN_BENEFIT_COST_AADT = 1   # Preservation with AADT weighting on benefit score

# ── Treatment cost uncertainty ────────────────────────────────────────────────
# 1 = sample treatment unit costs from a correlated lognormal distribution
# 0 = sample each treatment cost independently
USE_COST_CORRELATION  = 1

# ── User-cost baseline ────────────────────────────────────────────────────────
# IRI representing acceptable pavement condition (in/mi). Excess vehicle
# operating cost is computed relative to this reference IRI.
USER_COST_IRI_BASE    = 60

# ── Performance model uncertainty (coefficient of variation) ─────────────────
# Multiplicative noise applied to the annual deterioration increment each year.
# Set near zero (e.g. 0.0001) to approximate the deterministic baseline.
COV_IRI     = 0.0001   # IRI (roughness) deterioration CoV
COV_RUTTING =  0.0001  # Rutting deterioration CoV
COV_FATIGUE =  0.0001  # Fatigue cracking deterioration CoV

# ── Single-budget simulation ──────────────────────────────────────────────────
N_MC   = 400    # Monte Carlo iterations
SEED   = 123      # random seed for reproducibility
YEARS  = 10       # simulation horizon (years)
BUDGET = 440_000_000  # annual agency budget ($)

# ── Multi-budget simulation ───────────────────────────────────────────────────
BUDGETS_MULTI = [20_000_000, 100_000_000, 300_000_000, 500_000_000, 800_000_000]
N_MC_MULTI    = 400   # Monte Carlo iterations per budget level

# ── Composite condition index ─────────────────────────────────────────────────
# Weighted score combining normalized IRI, fatigue cracking, and rutting.
# Higher values indicate worse pavement condition.
#   scaled_iri     = IRI / COMPOSITE_IRI_SCALE          (weight 0.25)
#   scaled_crack   = Cracking / COMPOSITE_CRACK_SCALE   (weight 0.40)
#   scaled_rutting = Rutting * COMPOSITE_RUT_FACTOR      (weight 0.10)
#   composite = scaled_iri*0.25 + scaled_crack*0.40 + scaled_rutting*0.10
COMPOSITE_IRI_SCALE   = 8.0    # IRI normalization divisor
COMPOSITE_CRACK_SCALE = 4.0    # Cracking normalization divisor
COMPOSITE_RUT_FACTOR  = 25.0   # Rutting normalization multiplier
COMPOSITE_W_IRI       = 0.25   # IRI weight
COMPOSITE_W_CRACK     = 0.40   # Cracking weight
COMPOSITE_W_RUT       = 0.10   # Rutting weight

# ── Runtime scaling analysis ──────────────────────────────────────────────────
# Sample sizes used to profile simulation run time versus Monte Carlo iterations.
# All values must be <= N_MC.
TIMING_N_MC_VALUES = [1, 50, 250, 500, 750]

# ── Segment-level output (optional) ──────────────────────────────────────────
# Set SAVE_SEGMENT_RESULTS = 1 to export per-run Parquet files containing
# segment-level IRI, cracking, and rutting for each Monte Carlo realization.
SAVE_SEGMENT_RESULTS        = 0
SEGMENT_RESULTS_DIR         = "segment_results"
SEGMENT_RESULTS_COMPRESSION = "snappy"

# ── Output folder ─────────────────────────────────────────────────────────────
# Determined by USE_COST_CORRELATION; created automatically on first CSV export.
OUTPUT_DIR = "Correlation_Results" if USE_COST_CORRELATION else "Without_Correlation_Results"

## Section 1: Imports and Plotting Standard

This section imports the shared scientific Python libraries and defines the plotting constants used throughout the notebook. The figure dimensions, font scaling, colors, line widths, markers, percentile bands, and legend formatting follow `Code.ipynb`.


In [ ]:
# ============================================================
# Standard imports used throughout the notebook
# ============================================================
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import copy
from joblib import Parallel, delayed

# ============================================================
# Public plotting standard copied from Code.ipynb
# ============================================================
# Keeping these settings in one place ensures that every figure in the
# notebook uses the same dimensions, typography, colors, and line styling.
sns.set_theme(style="darkgrid", font_scale=1.6)

PLOT_PERCENTILE_BAND = (10, 90)
PLOT_FIGSIZE = (12, 7)
PLOT_FIGSIZE_VIOLIN = (22, 8)
PLOT_LINEWIDTH = 3
PLOT_MARKERSIZE = 12
PLOT_BAND_ALPHA = 0.20
PLOT_LABEL_SIZE = 20
PLOT_TICK_SIZE = 18
PLOT_TITLE_SIZE = 20
PLOT_LEGEND_SIZE = 15
PLOT_LEGEND_TITLE_SIZE = 20
PLOT_PALETTE = sns.color_palette("deep")
MULTI_BUDGET_MARKERS = ["o", "X", "s", "D", "^", "v", "P", "*"]

STRATEGY_STYLES = {
    "Worst-first": {
        "color": PLOT_PALETTE[0],
        "marker": "o",
        "label": "Worst-first",
    },
    "Benefit-Cost / AUC": {
        "color": PLOT_PALETTE[1],
        "marker": "X",
        "label": "Preservation",
    },
    "Benefit-Cost / AUC with AADT": {
        "color": "green",
        "marker": "s",
        "label": "Preservation (Considered AADT)",
    },
}

STRATEGY_ORDER = [
    "Benefit-Cost / AUC",
    "Benefit-Cost / AUC with AADT",
    "Worst-first",
]


## Section 2: Data Loading and Preprocessing

Load the treatment workbook, the 0.1-mile base-segment network, and the merged 5-mile decision network. The preprocessing cells normalize identifiers and construct the base-to-merged mapping used throughout the simulation.


In [ ]:
# ============================================================
# File paths and data loading
# Paths are relative to the notebook directory — no edits needed
# after cloning the repository.
# ============================================================
import pandas as pd
import geopandas as gpd
from pathlib import Path

# GIS shapefile (before merge — 0.1-mile base segments)
shapefile_path = Path("adot_data 2") / "gis" / "before_merge_0p1miles_arizona" / "Arizona.shp"
gdf = gpd.read_file(shapefile_path)

# GIS shapefile (merged — 5-mile merged segments for decision-making)
shapefile_merged_path = Path("adot_data 2") / "gis" / "merged_5miles_arizona" / "Merged_Arizona.shp"
gdf_merged = gpd.read_file(shapefile_merged_path)

print(f"Base segments loaded: {len(gdf)}")
print(f"Merged segments loaded: {len(gdf_merged)}")

In [ ]:
# ============================================================
# Fix data types: parse Original_I from string to list,
# ensure FID_N integers match between gdf and gdf_merged
# ============================================================
import pandas as pd
import ast

print("Fixing data types...")

def parse_list_string(x):
    """
    Parse a serialized Python list used in the Original_I column.

    Existing list values are returned unchanged. Invalid serialized values
    are converted to empty lists so the subsequent explode operation is safe.
    """
    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return x

gdf_merged['Original_I'] = gdf_merged['Original_I'].apply(parse_list_string)

temp_exploded = gdf_merged[['Original_I']].explode('Original_I')
temp_exploded['Original_I'] = pd.to_numeric(temp_exploded['Original_I'], errors='coerce').fillna(-1).astype(int)
gdf_merged['Original_I'] = temp_exploded.groupby(level=0)['Original_I'].agg(list)

gdf['FID_N'] = pd.to_numeric(gdf['FID_N'], errors='coerce').fillna(-1).astype(int)

exploded_ids = set(gdf_merged['Original_I'].explode())
base_ids = set(gdf['FID_N'])
overlap = len(exploded_ids.intersection(base_ids))

print(f"Data fixed. Matching IDs found: {overlap}")
if overlap == 0:
    print("WARNING: IDs still do not match.")

print(f"Merged segments: {len(gdf_merged)}")
print(f"Base segments: {len(gdf)}")

### Data QA: IRI and Cracking Before and After Merging

These plots verify that lane-mile-weighted aggregation from base segments to merged decision segments preserves reasonable IRI and cracking distributions. They are data-quality checks and do not participate in the simulation calculations.


In [ ]:
# ============================================================
# Data QA: Aggregated AvgIRI distribution before vs after merging.
# This is a data-quality check.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

gdf['lane_miles_cal'] = (gdf['ToMeasure'] - gdf['FromMeasur']) * gdf['Number_of_']

exploded = gdf_merged.explode('Original_I')
mapping = (
    exploded
    .reset_index()
    .rename(columns={'index': 'merged_idx', 'Original_I': 'FID_N'})
)
mapping = mapping.dropna(subset=['FID_N'])
mapping['FID_N'] = mapping['FID_N'].astype(gdf['FID_N'].dtype)

agg_df = mapping.merge(gdf[['FID_N', 'AvgIRI', 'lane_miles_cal']], on='FID_N', how='inner')
agg_df['Weighted_IRI'] = agg_df['AvgIRI'] * agg_df['lane_miles_cal']

groups = agg_df.groupby('merged_idx')
weighted_sum = groups['Weighted_IRI'].sum()
weight_sum   = groups['lane_miles_cal'].sum().replace(0, np.nan)
agg_vals = weighted_sum / weight_sum

new_col_name = 'Aggregated_AvgIRI'
gdf_merged[new_col_name] = np.nan
gdf_merged.loc[agg_vals.index, new_col_name] = agg_vals

if gdf_merged[new_col_name].isnull().any():
    fill_val = gdf['AvgIRI'].median()
    gdf_merged[new_col_name] = gdf_merged[new_col_name].fillna(fill_val)

base_mask   = gdf['AvgIRI'].notna()
merged_mask = gdf_merged[new_col_name].notna()
base_iri   = gdf.loc[base_mask, 'AvgIRI']
merged_iri = gdf_merged.loc[merged_mask, new_col_name]

print(f"Base segments: {base_iri.shape[0]}, Merged segments: {merged_iri.shape[0]}")

min_val = min(base_iri.min(), merged_iri.min())
max_val = max(base_iri.max(), merged_iri.max())
bins = np.linspace(min_val, max_val, 50)

plt.figure(figsize=PLOT_FIGSIZE)
plt.hist(base_iri, bins=bins, edgecolor='black', alpha=0.6,
         label=f'Raw AvgIRI (Base, n={base_iri.shape[0]})', density=True)
plt.hist(merged_iri, bins=bins, edgecolor='black', alpha=0.6,
         label=f'Aggregated AvgIRI (Merged, n={merged_iri.shape[0]})', density=True)
plt.title('Distribution of AvgIRI Before and After Merging', fontsize=PLOT_TITLE_SIZE)
plt.xlabel('AvgIRI Value', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.ylabel('Number of Segments', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.xticks(fontsize=PLOT_TICK_SIZE)
plt.yticks(fontsize=PLOT_TICK_SIZE)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(fontsize=PLOT_LEGEND_SIZE)
plt.tight_layout()
plt.show()

print("\n--- First 5 Merged Segments with Aggregated IRI ---")
print(gdf_merged[[new_col_name, 'Length_Mil']].head())

In [ ]:
# ============================================================
# Data QA: Aggregated HPMS_Crack distribution before vs after merging.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

gdf['lane_miles_cal'] = (gdf['ToMeasure'] - gdf['FromMeasur']) * gdf['Number_of_']

exploded = gdf_merged.explode('Original_I')
mapping = (
    exploded
    .reset_index()
    .rename(columns={'index': 'merged_idx', 'Original_I': 'FID_N'})
)
mapping = mapping.dropna(subset=['FID_N'])
mapping['FID_N'] = mapping['FID_N'].astype(gdf['FID_N'].dtype)

agg_df_crack = mapping.merge(gdf[['FID_N', 'HPMS_Crack', 'lane_miles_cal']], on='FID_N', how='inner')
agg_df_crack['Weighted_Crack'] = agg_df_crack['HPMS_Crack'] * agg_df_crack['lane_miles_cal']

groups_crack = agg_df_crack.groupby('merged_idx')
weighted_sum_crack = groups_crack['Weighted_Crack'].sum()
weight_sum_crack   = groups_crack['lane_miles_cal'].sum().replace(0, np.nan)
agg_vals_crack = weighted_sum_crack / weight_sum_crack

crack_col_name = 'Aggregated_HPMS_Crack'
gdf_merged[crack_col_name] = np.nan
gdf_merged.loc[agg_vals_crack.index, crack_col_name] = agg_vals_crack

if gdf_merged[crack_col_name].isnull().any():
    fill_val_crack = gdf['HPMS_Crack'].median()
    gdf_merged[crack_col_name] = gdf_merged[crack_col_name].fillna(fill_val_crack)

base_crack   = gdf.loc[gdf['HPMS_Crack'].notna(), 'HPMS_Crack']
merged_crack = gdf_merged.loc[gdf_merged[crack_col_name].notna(), crack_col_name]

print(f"Base segments: {base_crack.shape[0]}, Merged segments: {merged_crack.shape[0]}")

min_val_crack = min(base_crack.min(), merged_crack.min())
max_val_crack = max(base_crack.max(), merged_crack.max())
bins_crack = np.linspace(min_val_crack, max_val_crack, 50)

plt.figure(figsize=PLOT_FIGSIZE)
plt.hist(base_crack, bins=bins_crack, edgecolor='black', alpha=0.6,
         label=f'Raw HPMS_Crack (Base, n={base_crack.shape[0]})', density=True)
plt.hist(merged_crack, bins=bins_crack, edgecolor='black', alpha=0.6,
         label=f'Aggregated HPMS_Crack (Merged, n={merged_crack.shape[0]})', density=True)
plt.title('Distribution of HPMS_Crack Before and After Merging', fontsize=PLOT_TITLE_SIZE)
plt.xlabel('HPMS_Crack Value', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.ylabel('Number of Segments', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.xticks(fontsize=PLOT_TICK_SIZE)
plt.yticks(fontsize=PLOT_TICK_SIZE)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(fontsize=PLOT_LEGEND_SIZE)
plt.tight_layout()
plt.show()

print("\n--- First 5 Merged Segments with Aggregated Cracking ---")
print(gdf_merged[[crack_col_name, 'Length_Mil']].head())

## Section 3: Treatment Cost Distributions

Read treatment definitions and agency unit costs from the workbook. Treatment costs are sampled once per Monte Carlo realization and are later used by each budget-allocation strategy.


In [ ]:
# ============================================================
# Treatment definitions — hardcoded from the ADOT treatment workbook.
# All fields from the original structure are preserved.
# ============================================================
_nan = float('nan')

treatments = {
    'CPR': {
        'Description': 'Concrete Pavement Repair',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 5000,
        'Roughness': _nan,
        'Rutting': _nan,
        'Cracking': _nan,
        'Faulting': _nan,
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'CRACKSEAL': {
        'Description': 'Crack Seal',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 9200,
        'Roughness': _nan,
        'Rutting': _nan,
        'Cracking': 'Reset to new and held constant for the next two years, then deteriorates back to previous predicted value during the next two years',
        'Faulting': _nan,
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'CRACKSEAL_AND_CHIPSEAL': {
        'Description': 'Crack Seal and Chip Seal',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 51400,
        'Roughness': 'Increase 10% and then deteriorates as normal',
        'Rutting': _nan,
        'Cracking': 'Reset to new and held constant for the next two years, then deteriorates back to previous predicted value during the next four years',
        'Faulting': _nan,
        'Years_to_pretreatment_crack': 4,
        'years_crack_at_zero': 2,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'DIAMOND_GRIND': {
        'Description': 'Diamond Grinding of Concrete Pavement',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 104000,
        'Roughness': _nan,
        'Rutting': _nan,
        'Cracking': _nan,
        'Faulting': _nan,
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'FOG_COAT': {
        'Description': 'Fog Coat',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 4400,
        'Roughness': _nan,
        'Rutting': _nan,
        'Cracking': _nan,
        'Faulting': _nan,
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'MAJOR_REHAB_OR_RECONSTRUCTION': {
        'Description': 'Major Rehab or Reconstruction',
        'Budget_Category': 'Major_Projects',
        'Cost_lane_mi': _nan,
        'Roughness': _nan,
        'Rutting': _nan,
        'Cracking': _nan,
        'Faulting': _nan,
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'MILL_FR_AND_MICRO_CAPE_SEAL': {
        'Description': 'Mill FR and Micro Cape Seal',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 129900,
        'Roughness': _nan,
        'Rutting': 'Reset to new and held constant for the next one year, then deteriorates back to pre-existing condition during the next two years',
        'Cracking': 'Reset to new and held constant for the next three years, then deteriorates back to previous predicted value during the next five years',
        'Faulting': _nan,
        'Years_to_pretreatment_crack': 5,
        'years_crack_at_zero': 3,
        'years_to_pretreatment_rutting': 2,
        'years_rutting_at_zero': 1,
    },
    'MS_1_PASS': {
        'Description': '1 Pass Micro Surface',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 52100,
        'Roughness': _nan,
        'Rutting': _nan,
        'Cracking': 'Reset to new and held constant for two years, then deteriorates back to previous predicted value during the next four years',
        'Faulting': _nan,
        'Years_to_pretreatment_crack': 4,
        'years_crack_at_zero': 2,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'MS_2_PASS': {
        'Description': '2 Pass Micro Surface',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 52100,
        'Roughness': _nan,
        'Rutting': 'Reset to new, held constant for one year, then deteriorates back to pre-existing condition during the next two years',
        'Cracking': 'Reset to new and held constant for two years, then deteriorates back to previous predicted value during the next five years',
        'Faulting': _nan,
        'Years_to_pretreatment_crack': 5,
        'years_crack_at_zero': 2,
        'years_to_pretreatment_rutting': 2,
        'years_rutting_at_zero': 1,
    },
    'RECONSTRUCTION': {
        'Description': 'Reconstruction for Worst First Analysis only',
        'Budget_Category': 'Reconstruction',
        'Cost_lane_mi': 1033600,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'RR_0p5INCH_FR': {
        'Description': 'Remove and Replace 0.5-inch plus FR',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 125600,
        'Roughness': '60% of existing and then deteriorates as normal',
        'Rutting': 'Reset to new, held constant for five years, then deteriorates back to pre-existing condition during the next five years',
        'Cracking': 'Reset to new and held constant for five years, then deteriorates back to previous predicted value during the next five years',
        'Faulting': _nan,
        'Years_to_pretreatment_crack': 5,
        'years_crack_at_zero': 5,
        'years_to_pretreatment_rutting': 5,
        'years_rutting_at_zero': 5,
    },
    'RR_1INCH_FR': {
        'Description': 'Remove and Replace 1-inch plus FR',
        'Budget_Category': 'Preservation',
        'Cost_lane_mi': 162500,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'RR_2INCH_AC_FR': {
        'Description': 'Remove and Replace 2-inch AC + FR',
        'Budget_Category': 'Major_Projects',
        'Cost_lane_mi': 236100,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'RR_2p5INCH_AC_FR': {
        'Description': 'Remove and Replace 2.5-inch AC + FR',
        'Budget_Category': 'Major_Projects',
        'Cost_lane_mi': 273200,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'RR_3INCH_AC_FR': {
        'Description': 'Remove and Replace 3-inch AC + FR',
        'Budget_Category': 'Major_Projects',
        'Cost_lane_mi': 309900,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'RR_4INCH_AC_FR': {
        'Description': 'Remove and Replace 4-inch AC + FR',
        'Budget_Category': 'Major_Projects',
        'Cost_lane_mi': 384000,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'RR_5INCH_AC_FR': {
        'Description': 'Remove and Replace 5-inch AC + FR',
        'Budget_Category': 'Major_Projects',
        'Cost_lane_mi': 457500,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
    'SR_3INCH_AC_MS': {
        'Description': 'Spot Repair 3-inch AC with Micro Surfacing',
        'Budget_Category': 'Major_Projects',
        'Cost_lane_mi': 63200,
        'Roughness': 'Reset to new',
        'Rutting': 'Reset to new',
        'Cracking': 'Reset to new',
        'Faulting': 'Reset to new',
        'Years_to_pretreatment_crack': _nan,
        'years_crack_at_zero': _nan,
        'years_to_pretreatment_rutting': _nan,
        'years_rutting_at_zero': _nan,
    },
}

# Fix MAJOR_REHAB_OR_RECONSTRUCTION cost (copy from RECONSTRUCTION)
reconstruction_cost = None
if 'RECONSTRUCTION' in treatments:
    reconstruction_cost = treatments['RECONSTRUCTION'].get('Cost_lane_mi')

for treatment_name, treatment_data in treatments.items():
    cost_lane_mi = treatment_data.get('Cost_lane_mi')
    if treatment_name == 'MAJOR_REHAB_OR_RECONSTRUCTION' and (cost_lane_mi is None or (isinstance(cost_lane_mi, float) and np.isnan(cost_lane_mi))):
        treatments[treatment_name]['Cost_lane_mi'] = reconstruction_cost

for treatment_name, treatment_data in treatments.items():
    print(f"Treatment: {treatment_name}, Cost_lane_mi: {treatments[treatment_name].get('Cost_lane_mi')}")

In [ ]:
# ============================================================
# Treatment cost distributions: attach lognormal parameters
# (mean + std) to each treatment type. These are sampled
# once per MC run to reflect cost uncertainty.
# ============================================================

# Cost standard deviation table ($/lane-mile)
COST_STD_TABLE = {
    'CRACKSEAL':                    2_900,
    'CRACKSEAL_AND_CHIPSEAL':      11_300,
    'MILL_FR_AND_MICRO_CAPE_SEAL': 21_700,
    'MS_1_PASS':                   13_000,
    'MS_1_PASS_OR_CHIPSEAL':       13_000,
    'MS_2_PASS':                   13_000,
    'FOG_COAT':                     2_400,
    'RR_0p5INCH_FR':               20_300,
    'RR_0p5INCH_FR_OR_MS_1_PASS':  20_300,
    'RR_1INCH_FR':                 26_100,
    'RR_2INCH_AC_FR':              41_000,
    'RR_2p5INCH_AC_FR':            49_600,
    'RR_3INCH_AC_FR':              57_800,
    'RR_4INCH_AC_FR':              75_300,
    'RR_5INCH_AC_FR':              92_600,
    'SR_3INCH_AC_MS':              13_400,
    'CPR':                            500,
    'DIAMOND_GRIND':               11_000,
    'RECONSTRUCTION':             189_100,
    'RECONSTRUCTION_OR_RR_1INCH_FR':               189_100,
    'RECONSTRUCTION_OR_RR_4INCH_AC_FR_OR_RR_3INCH_AC_FR': 189_100,
    'RECONSTRUCTION_OR_RR_5INCH_AC_FR_OR_RR_3INCH_AC_FR': 189_100,
    'RR_4INCH_AC_FR_OR_RR_3INCH_AC_FR':            75_300,
    'RR_5INCH_AC_FR_OR_RR_3INCH_AC_FR':            92_600,
}

# Compute lognormal parameters for each treatment
for t_name, t_data in treatments.items():
    mean_cost = t_data.get('Cost_lane_mi')
    if mean_cost is None or (isinstance(mean_cost, float) and np.isnan(mean_cost)):
        mean_cost = 0.0

    std_cost = COST_STD_TABLE.get(t_name, mean_cost * 0.1)  # fallback: 10% of mean

    if mean_cost > 0 and std_cost > 0:
        # Lognormal parameterization: mean and std of underlying normal
        cv2 = (std_cost / mean_cost) ** 2
        ln_sigma = np.sqrt(np.log(1 + cv2))
        ln_mu = np.log(mean_cost) - 0.5 * ln_sigma**2
        t_data['ln_mu']    = ln_mu
        t_data['ln_sigma'] = ln_sigma
    else:
        t_data['ln_mu']    = None
        t_data['ln_sigma'] = None

# Alias for use inside simulation functions
treatments_cost = treatments

print('Lognormal cost parameters attached.')
for t_name, t_data in treatments_cost.items():
    print(f"  {t_name}: mu={t_data.get('ln_mu','N/A'):.3f}, sigma={t_data.get('ln_sigma','N/A'):.3f}" if t_data.get('ln_mu') else f"  {t_name}: no lognormal params")

In [ ]:
# ============================================================
# Correlated treatment unit-cost sampling helpers
# ============================================================

# Mapping: correlation CSV column names -> PMS treatment dict keys
CORR_CSV_TO_PMS = {
    "R&R 0.5-inch AC + FR": "RR_0p5INCH_FR",
    "R&R 1-inch AC + FR":   "RR_1INCH_FR",
    "R&R 2-inch AC + FR":   "RR_2INCH_AC_FR",
    "R&R 2.5-inch AC + FR": "RR_2p5INCH_AC_FR",
    "R&R 3-inch AC + FR":   "RR_3INCH_AC_FR",
    "R&R 4-inch AC + FR":   "RR_4INCH_AC_FR",
    "R&R 5-inch AC + FR":   "RR_5INCH_AC_FR",
    "Reconstruction":       "RECONSTRUCTION",
}

def _load_and_validate_corr_csv(csv_path):
    """Load correlation CSV; enforce symmetry, diagonal=1, and repair to PSD."""
    df = pd.read_csv(csv_path, index_col=0)
    df = df.apply(pd.to_numeric, errors="coerce")
    arr = df.to_numpy(dtype=float)
    arr = (arr + arr.T) / 2.0
    np.fill_diagonal(arr, 1.0)
    eigvals, eigvecs = np.linalg.eigh(arr)
    eigvals = np.maximum(eigvals, 1e-8)
    arr = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(arr))
    arr = arr / np.outer(d, d)
    np.fill_diagonal(arr, 1.0)
    return pd.DataFrame(arr, index=df.index, columns=df.columns)

def _cost_corr_to_normal_space(rho_cost_mat, sigmas):
    """
    Convert final-cost correlation matrix to underlying normal-space correlation.
    rho_z[i,j] = log(1 + rho_cost[i,j]*sqrt((exp(si^2)-1)*(exp(sj^2)-1))) / (si*sj)
    """
    n = len(sigmas)
    rho_z = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            si, sj = sigmas[i], sigmas[j]
            inner = rho_cost_mat[i, j] * np.sqrt(
                (np.exp(si**2) - 1.0) * (np.exp(sj**2) - 1.0)
            )
            inner = np.clip(inner, -0.9999, None)
            rho_z[i, j] = np.log1p(inner) / (si * sj)
            rho_z[j, i] = rho_z[i, j]
    return rho_z

def _repair_psd(mat):
    """Clip negative eigenvalues and renormalise to a valid correlation matrix."""
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.maximum(eigvals, 1e-8)
    mat = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(mat))
    mat = mat / np.outer(d, d)
    np.fill_diagonal(mat, 1.0)
    return mat

def sample_fixed_unit_costs_with_correlation(treatments_dict, rng, corr_cost_df=None):
    """
    Sample one unit cost per treatment for a single MC realization.

    Treatments whose names appear in corr_cost_df (via CORR_CSV_TO_PMS) and have
    valid ln_mu / ln_sigma are sampled jointly from a correlated multivariate
    lognormal.  All other treatments are sampled independently, exactly as before.
    Zero-cost treatments (DO_NOTHING, CONTINUE_TRACKING, REPAIR, DIAMOND_GRIND)
    are always 0.
    """
    ZERO_COST = {"DO_NOTHING", "CONTINUE_TRACKING", "REPAIR", "DIAMOND_GRIND"}

    # Step 1 -- independent baseline draw for every treatment
    fixed = {}
    for t, info in treatments_dict.items():
        if t in ZERO_COST:
            fixed[t] = 0.0
            continue
        ln_mu    = info.get("ln_mu",    None)
        ln_sigma = info.get("ln_sigma", None)
        if ln_mu is not None and ln_sigma is not None and float(ln_sigma) > 0:
            fixed[t] = float(rng.lognormal(mean=float(ln_mu), sigma=float(ln_sigma)))
        else:
            fixed[t] = float(info.get("Cost_lane_mi", 0.0) or 0.0)
    for t in ZERO_COST:
        fixed.setdefault(t, 0.0)

    if corr_cost_df is None:
        return fixed

    # Step 2 -- find treatments present in both PMS dict and correlation matrix
    matched_csv = [
        c for c in corr_cost_df.columns
        if CORR_CSV_TO_PMS.get(c) in treatments_dict
        and treatments_dict[CORR_CSV_TO_PMS[c]].get("ln_mu") is not None
        and float(treatments_dict[CORR_CSV_TO_PMS[c]].get("ln_sigma", 0)) > 0
    ]
    matched_pms = [CORR_CSV_TO_PMS[c] for c in matched_csv]

    if len(matched_pms) < 2:
        return fixed

    sigmas = np.array([float(treatments_dict[p]["ln_sigma"]) for p in matched_pms])
    ln_mus = np.array([float(treatments_dict[p]["ln_mu"])    for p in matched_pms])

    # Step 3 -- convert cost-space submatrix to normal-space, repair PSD
    rho_cost = corr_cost_df.loc[matched_csv, matched_csv].to_numpy(dtype=float)
    rho_z    = _cost_corr_to_normal_space(rho_cost, sigmas)
    rho_z    = _repair_psd(rho_z)

    # Step 4 -- draw correlated standard normals; overwrite independent samples
    z_corr = rng.multivariate_normal(np.zeros(len(matched_pms)), rho_z)
    for i, pms_name in enumerate(matched_pms):
        fixed[pms_name] = float(np.exp(ln_mus[i] + sigmas[i] * z_corr[i]))

    return fixed

# Load and validate the treatment cost correlation matrix
_corr_csv_path = "paired_rr_ac_fr_correlation_matrix.csv"
try:
    treatment_cost_corr_df = _load_and_validate_corr_csv(_corr_csv_path)
    _matched_names = [CORR_CSV_TO_PMS[c] for c in treatment_cost_corr_df.columns
                      if CORR_CSV_TO_PMS.get(c) in treatments_cost]
    print(f"Loaded treatment cost correlation matrix ({treatment_cost_corr_df.shape[0]}x{treatment_cost_corr_df.shape[1]})")
    print(f"  Matched PMS treatments ({len(_matched_names)}): {_matched_names}")
except Exception as _e:
    print(f"WARNING: Could not load correlation CSV -- falling back to independent sampling. ({_e})")
    treatment_cost_corr_df = None

print("sample_fixed_unit_costs_with_correlation defined.")

## Section 4: ESAL, Pavement Family, and Structural Variability

Calculate 20-year equivalent single axle loads (ESAL), assign pavement families, cap ESAL values according to the source logic, and define the structural variability factor used by the performance models.


In [ ]:
# ============================================================
# Truck fraction (Number_of_Trucks) by functional class.
# ESAL 20-year calculation using VCD, TF, DD, LD, TGF.
# ============================================================

# Number_of_Trucks Calculation
gdf['Number_of_Trucks'] = np.select(
    [
        gdf['Functional'].isin([1, 2]),
        gdf['Functional'].isin([11, 12]),
        gdf['Functional'].isin([3, 4]),
        gdf['Functional'].isin([14]),
        gdf['Functional'].isin([6, 7, 8, 9]),
        gdf['Functional'].isin([16, 17, 18, 19])
    ],
    [0.2391, 0.0837, 0.0740, 0.0224, 0.0339, 0.0054],
    default=0
)

if gdf['Functional'].isin([1, 2]).all():
    VCD_values = {4: 5.3, 5: 46.3, 6: 5.7, 7: 0.7, 8: 16.1, 9: 24.1, 10: 1.1, 11: 0.3, 12: 0.1, 13: 0.3}
else:
    VCD_values = {4: 7.8, 5: 65.8, 6: 4.4, 7: 0.2, 8: 11.7, 9: 9.1, 10: 0.7, 11: 0.2, 12: 0.0, 13: 0.1}

TF_values = {
    (1, 11): {4: 1.07, 5: 0.33, 6: 0.64, 7: 0.58, 8: 0.61, 9: 1.62, 10: 1.43, 11: 1.75, 12: 1.31, 13: 3.51},
    (2, 6, 8, 9, 12, 16, 18, 19): {4: 1.06, 5: 0.39, 6: 0.96, 7: 0.61, 8: 0.91, 9: 1.34, 10: 1.53, 11: 1.96, 12: 1.33, 13: 3.50},
    (4, 14): {4: 1.20, 5: 0.13, 6: 0.86, 7: 0.64, 8: 0.52, 9: 1.93, 10: 1.78, 11: 2.25, 12: 1.17, 13: 2.07}
}

def get_TF_value(row, class_number):
    """Return the truck factor for one functional class and vehicle class."""
    for functional_group, class_TF in TF_values.items():
        if row['Functional'] in functional_group:
            return class_TF.get(class_number, 1.0)
    return 1.0

tf_by_class = {}
for class_number in VCD_values:
    func_to_tf = {}
    for functional_group, class_TF in TF_values.items():
        tf_val = class_TF.get(class_number, 1.0)
        for func in functional_group:
            func_to_tf[func] = tf_val
    tf_by_class[class_number] = func_to_tf

gdf['lanes'] = gdf['Number_of_'] * 2
gdf['DD'] = np.select(
    [gdf['lanes'] == 2, gdf['lanes'] == 4, gdf['lanes'] >= 6],
    [0.5, 0.45, 0.4],
    default=None
)
gdf['LD'] = np.select(
    [gdf['Number_of_'] == 1, gdf['Number_of_'] == 2, gdf['Number_of_'] == 3, gdf['Number_of_'] >= 4],
    [1, 0.9, 0.8, 0.7],
    default=None
)

annual_traffic_growth = 0.00001
year = 20
tgf = (((1 + annual_traffic_growth) ** year) - 1) / annual_traffic_growth

def esal_20_year(gdf):
    """Calculate 20-year ESAL values for every base pavement segment."""
    gdf['ESAL_20_year'] = 0
    for class_number, vcd in VCD_values.items():
        tf_values = gdf.apply(lambda row: get_TF_value(row, class_number), axis=1)
        gdf['ESAL_20_year'] += (vcd * tf_values * gdf['AADT'] * gdf['Number_of_Trucks'] * gdf['DD'] * gdf['LD'] * tgf * 365/100)
    return gdf

gdf = esal_20_year(gdf)

In [ ]:
# ============================================================
# Pavement family classification and SVF (subgrade variability factor).
# ============================================================
gdf['first_digit'] = np.where(gdf['PAVEMENT_T'] == 'AC', '1', np.where(gdf['PAVEMENT_T'] == 'Other', '2', '0'))
climate_assumption = '1'
gdf['second_digit'] = climate_assumption
gdf['third_digit'] = np.select(
    [gdf['ESAL_20_year'] < 300000,
     (gdf['ESAL_20_year'] >= 300000) & (gdf['ESAL_20_year'] < 3000000),
     (gdf['ESAL_20_year'] >= 3000000) & (gdf['ESAL_20_year'] < 10000000),
     (gdf['ESAL_20_year'] >= 10000000) & (gdf['ESAL_20_year'] < 30000000),
     gdf['ESAL_20_year'] >= 30000000],
    ['1', '2', '3', '4', '5'],
    default='0'
)
foundation_assumption = '1'
gdf['fourth_digit'] = foundation_assumption
gdf['Pavement_Family'] = gdf['first_digit'] + gdf['second_digit'] + gdf['third_digit'] + gdf['fourth_digit']

gdf['svf'] = np.where(gdf['Pavement_Family'].str[1] == '1', 0.75, 3.25)

In [ ]:
# ============================================================
# ESAL summary statistics and distribution plots.
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

print("Summary Statistics for ESAL_20_year:")
print(gdf['ESAL_20_year'].describe())

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
print("\nPercentiles for ESAL_20_year:")
print(gdf['ESAL_20_year'].quantile([p/100 for p in percentiles]))

plt.figure(figsize=PLOT_FIGSIZE)
sns.histplot(gdf['ESAL_20_year'], bins=50, kde=True)
plt.xlabel('ESAL 20-year', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.ylabel('Frequency', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.title('Distribution of ESAL 20-year Values', fontsize=PLOT_TITLE_SIZE)
plt.xticks(fontsize=PLOT_TICK_SIZE)
plt.yticks(fontsize=PLOT_TICK_SIZE)
plt.xticks(fontsize=PLOT_TICK_SIZE)
plt.yticks(fontsize=PLOT_TICK_SIZE)
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=PLOT_FIGSIZE)
sns.histplot(gdf['ESAL_20_year'], bins=50, kde=True, log_scale=True)
plt.xlabel('ESAL 20-year (log scale)', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.ylabel('Frequency', fontsize=PLOT_LABEL_SIZE, fontweight='bold')
plt.title('Distribution of ESAL 20-year Values (Log Scale)', fontsize=PLOT_TITLE_SIZE)
plt.xticks(fontsize=PLOT_TICK_SIZE)
plt.yticks(fontsize=PLOT_TICK_SIZE)
plt.xticks(fontsize=PLOT_TICK_SIZE)
plt.yticks(fontsize=PLOT_TICK_SIZE)
plt.grid(True)
plt.tight_layout()
plt.show()

esal_max = gdf['ESAL_20_year'].max()
esal_min = gdf['ESAL_20_year'].min()
print(f"Maximum ESAL_20_year value: {esal_max:,.0f}")
print(f"Minimum ESAL_20_year value: {esal_min:,.0f}")

count_1141 = (gdf['Pavement_Family'] == "1141").sum()
print(f"Number of pavement family '1141': {count_1141}")

In [ ]:
# ============================================================
# ESAL capping by pavement family, then recompute Pavement_Family and SVF.
# ============================================================
def cap_esal(row):
    """Apply the source-notebook ESAL cap for a pavement-family row."""
    if row['Pavement_Family'] == "1141":
        return min(row['ESAL_20_year'], 24000000)
    elif row['Pavement_Family'].startswith("11"):
        return min(row['ESAL_20_year'], 30000000)
    elif row['Pavement_Family'].startswith("21"):
        return min(row['ESAL_20_year'], 380000000)
    return row['ESAL_20_year']

gdf['ESAL_20_year'] = gdf.apply(cap_esal, axis=1)

gdf['first_digit'] = np.where(gdf['PAVEMENT_T'] == 'AC', '1', np.where(gdf['PAVEMENT_T'] == 'Other', '2', '0'))
climate_assumption = '1'
gdf['second_digit'] = climate_assumption
gdf['third_digit'] = np.select([
    gdf['ESAL_20_year'] < 300000,
    (gdf['ESAL_20_year'] >= 300000) & (gdf['ESAL_20_year'] < 3000000),
    (gdf['ESAL_20_year'] >= 3000000) & (gdf['ESAL_20_year'] < 10000000),
    (gdf['ESAL_20_year'] >= 10000000) & (gdf['ESAL_20_year'] < 30000000),
    gdf['ESAL_20_year'] >= 30000000
], ['1', '2', '3', '4', '5'], default='0')

foundation_assumption = '1'
gdf['fourth_digit'] = foundation_assumption
gdf['Pavement_Family'] = gdf['first_digit'] + gdf['second_digit'] + gdf['third_digit'] + gdf['fourth_digit']

gdf['svf'] = np.where(gdf['Pavement_Family'].str[1] == '1', 0.75, 3.25)
print("ESAL capping and pavement family recalculation done.")

## Section 5: Performance Models and Precomputation

Define the cracking, rutting, and IRI coefficient lookups and projection functions. These models estimate untreated pavement deterioration and support treatment-benefit calculations during each simulated year.


In [ ]:
# ============================================================
# Crack progression model: coefficients and projection function.
# Computes and stores a_crack, b_crack, c_crack, d_crack on gdf.
# ============================================================

def get_coefficients_crack(pavement_families):
    """Vectorized coefficient lookup for cracking model."""
    base_coefficients_hpms = {
        '11':   {'a': 0.45, 'b': 0.07, 'c': 0.02, 'd': 0.05},
        '12':   {'a': 0.45, 'b': 0.14, 'c': 0.02, 'd': 0.05},
        '21':   {'a': 0.45, 'b': 0.02, 'c': 0.03, 'd': 0.02},
        '22':   {'a': 0.45, 'b': 0.02, 'c': 0.02, 'd': 0.02},
        '1141': {'a': 0.45, 'b': 0.13, 'c': 0.02, 'd': 0.05}
    }
    a_crack_coeffs = np.zeros(len(pavement_families))
    b_crack_coeffs = np.zeros(len(pavement_families))
    c_crack_coeffs = np.zeros(len(pavement_families))
    d_crack_coeffs = np.zeros(len(pavement_families))

    for i, pf in enumerate(pavement_families):
        pf_str = str(pf)
        if pf_str in base_coefficients_hpms:
            coeffs = base_coefficients_hpms[pf_str]
        else:
            first_two = pf_str[:2]
            coeffs = base_coefficients_hpms.get(first_two, {'a': 0, 'b': 0, 'c': 0, 'd': 0})
        a_crack_coeffs[i] = coeffs['a']
        b_crack_coeffs[i] = coeffs['b']
        c_crack_coeffs[i] = coeffs['c']
        d_crack_coeffs[i] = coeffs['d']

    return a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs

def compute_hpms_crack_projection(Age, ESAL, a_crack, b_crack, c_crack, d_crack, SVF, initial_HPMS_Crack, years=10):
    """Project HPMS cracking forward from a known age."""
    ESAL_DEF = 20 * 10**6
    hpms_crack_prog = [initial_HPMS_Crack]
    for t in range(1, years + 1):
        growth_factor = max(1.002, (1 + a_crack + b_crack * (ESAL / ESAL_DEF) - c_crack * (Age + t - 1)))
        next_crack = hpms_crack_prog[t - 1] * growth_factor + d_crack * SVF
        next_crack = min(next_crack, 100)
        hpms_crack_prog.append(next_crack)
    return hpms_crack_prog

# Precompute and store crack coefficients on gdf (required by apply_treatment_effects)
pavement_families = gdf['Pavement_Family'].values
SVF_values        = gdf['svf'].values
ESAL_values       = gdf['ESAL_20_year'].values
n_rows            = len(ESAL_values)

a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs = get_coefficients_crack(pavement_families)

gdf['a_crack'] = a_crack_coeffs
gdf['b_crack'] = b_crack_coeffs
gdf['c_crack'] = c_crack_coeffs
gdf['d_crack'] = d_crack_coeffs

print("Crack coefficients stored on gdf:", list(gdf[['a_crack','b_crack','c_crack','d_crack']].head(2).T.to_dict().values()))

In [ ]:
# ============================================================
# Rutting progression model: coefficients and projection function.
# Computes and stores a_rutt, b_rutt, c_rutt on gdf.
# ============================================================

def get_coefficients_rutting(pavement_families):
    """Vectorized coefficient lookup for rutting model."""
    base_coefficients_rutting = {
        '11':   {'a': 0.0003, 'b': 0.0014, 'c': 0.004},
        '12':   {'a': 0.0003, 'b': 0.0009, 'c': 0.0014},
        '21':   {'a': 0.0006, 'b': 0.0006, 'c': 0},
        '22':   {'a': 0.0006, 'b': 0.005,  'c': 0},
        '1141': {'a': 0.0004, 'b': 0.025,  'c': 0}
    }
    a_rutt_coeffs = np.zeros(len(pavement_families))
    b_rutt_coeffs = np.zeros(len(pavement_families))
    c_rutt_coeffs = np.zeros(len(pavement_families))

    for i, pf in enumerate(pavement_families):
        pf_str = str(pf)
        if pf_str in base_coefficients_rutting:
            coeffs = base_coefficients_rutting[pf_str]
        else:
            first_two = pf_str[:2]
            coeffs = base_coefficients_rutting.get(first_two, {'a': 0, 'b': 0, 'c': 0})
        a_rutt_coeffs[i] = coeffs['a']
        b_rutt_coeffs[i] = coeffs['b']
        c_rutt_coeffs[i] = coeffs['c']

    return a_rutt_coeffs, b_rutt_coeffs, c_rutt_coeffs

def compute_rutting_projection(Age, ESAL, a_rutt, b_rutt, c_rutt, SVF, initial_Rutting, years=10):
    """Project rutting forward from a known age."""
    ESAL_DEF = 20 * 10**6
    Rutting_progress = [initial_Rutting]
    for t in range(1, years + 1):
        growth_factor = (a_rutt * (Age + t - 1)) + (b_rutt * ESAL / ESAL_DEF) + (c_rutt * SVF)
        next_rutting = Rutting_progress[t - 1] + growth_factor
        Rutting_progress.append(next_rutting)
    return Rutting_progress

# Precompute and store rutting coefficients on gdf (required by apply_treatment_effects)
a_rutt_coeffs, b_rutt_coeffs, c_rutt_coeffs = get_coefficients_rutting(pavement_families)

gdf['a_rutt'] = a_rutt_coeffs
gdf['b_rutt'] = b_rutt_coeffs
gdf['c_rutt'] = c_rutt_coeffs

print("Rutting coefficients stored on gdf.")

In [ ]:
# ============================================================
# IRI progression model: coefficients and projection function.
# Computes and stores a_iri, b_iri, c_iri on gdf.
# ============================================================

def get_coefficients_iri(pavement_families):
    """Vectorized coefficient lookup for IRI model."""
    base_coefficients_iri = {
        '11':   {'a': 0.254, 'b': 0,   'c': 3.73},
        '12':   {'a': 0.272, 'b': 0,   'c': 1.57},
        '21':   {'a': 0,     'b': 0.3, 'c': 3.6},
        '22':   {'a': 0,     'b': 0.3, 'c': 3.6},
        '1141': {'a': 0.254, 'b': 0,   'c': 3.73}
    }
    a_iri_coeffs = np.zeros(len(pavement_families))
    b_iri_coeffs = np.zeros(len(pavement_families))
    c_iri_coeffs = np.zeros(len(pavement_families))

    for i, pf in enumerate(pavement_families):
        pf_str = str(pf)
        if pf_str in base_coefficients_iri:
            coeffs = base_coefficients_iri[pf_str]
        else:
            first_two = pf_str[:2]
            coeffs = base_coefficients_iri.get(first_two, {'a': 0, 'b': 0, 'c': 0})
        a_iri_coeffs[i] = coeffs['a']
        b_iri_coeffs[i] = coeffs['b']
        c_iri_coeffs[i] = coeffs['c']

    return a_iri_coeffs, b_iri_coeffs, c_iri_coeffs

def compute_iri_projection(Age, ESAL, a_iri, b_iri, c_iri, SVF, initial_iri, years=10):
    """Project IRI forward from a known age."""
    ESAL_DEF = 20 * 10**6
    iri_progress = [initial_iri]
    for t in range(1, years + 1):
        growth_factor = (a_iri * (Age + t - 1)) + (b_iri * ESAL / ESAL_DEF) + (c_iri * SVF)
        next_iri = iri_progress[t - 1] + growth_factor
        iri_progress.append(next_iri)
    return iri_progress

# Precompute and store IRI coefficients on gdf (required by apply_treatment_effects)
a_iri_coeffs, b_iri_coeffs, c_iri_coeffs = get_coefficients_iri(pavement_families)

gdf['a_iri'] = a_iri_coeffs
gdf['b_iri'] = b_iri_coeffs
gdf['c_iri'] = c_iri_coeffs

print("IRI coefficients stored on gdf.")

## Section 6: ADOT Decision Tree and Simulation Mapping

Apply the treatment decision tree to pavement condition, functional class, traffic, rehabilitation history, and scheduling state. Mapping helpers aggregate base-segment conditions to merged decision segments and propagate selected treatments back to the base network.


In [ ]:
# ============================================================
# ADOT decision tree (fast_decision_tree_safe) and helpers:
#   replace_or_treatment, build_simulation_mapping,
#   prepare_mapping_cache, aggregate_weighted_to_merged (slow),
#   aggregate_weighted_to_merged_fast, propagate_treatments_from_merged_to_gdf,
#   _ensure_merged_functional_once, drop_sim_columns
# ============================================================
import pandas as pd
import numpy as np
import warnings

gdf['New_Rutting'] = gdf['Rutting']
gdf['New_AvgIRI'] = gdf['AvgIRI']
gdf['New_HPMS_Cracking'] = gdf['HPMS_Crack']

gdf['No_of_Rehab'] = 0
gdf['Foundation_Issue'] = 0
gdf['SCHD_Year'] = 0

SENTINEL = "__UNDECIDED__"

def fast_decision_tree_safe(
    gdf: pd.DataFrame,
    *,
    normalize_continue_tracking: bool = True,
    fail_on_undecided: bool = True,
    warn_on_nans: bool = False,
    return_audit: bool = True,
):

    """Assign an ADOT treatment and audit rule to each pavement segment."""
    required = [
        "PAVEMENT_T", "New_AvgIRI", "New_HPMS_Cracking", "New_Rutting",
        "IRI_Rating", "Cracking_R", "Rutting_Ra",
        "No_of_Rehab", "Foundation_Issue",
        "Functional", "AADT", "SCHD_Year"
    ]
    missing = [c for c in required if c not in gdf.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    pav = gdf["PAVEMENT_T"].astype("object")

    iri   = pd.to_numeric(gdf["New_AvgIRI"], errors="coerce")
    crack = pd.to_numeric(gdf["New_HPMS_Cracking"], errors="coerce")
    rut   = pd.to_numeric(gdf["New_Rutting"], errors="coerce")
    aadt  = pd.to_numeric(gdf["AADT"], errors="coerce")
    schd  = pd.to_numeric(gdf["SCHD_Year"], errors="coerce")

    iri_rating = gdf["IRI_Rating"]
    cracking_r = gdf["Cracking_R"]
    rutting_r  = gdf["Rutting_Ra"]

    no_rehab = gdf["No_of_Rehab"]
    found_issue = gdf["Foundation_Issue"]
    functional = gdf["Functional"]

    trt = pd.Series(SENTINEL, index=gdf.index, dtype="object")
    rule = pd.Series("", index=gdf.index, dtype="object")

    def assign(mask, value, rule_name):
        """Assign a treatment only to rows that remain undecided."""
        undec = trt.eq(SENTINEL)
        hit = undec & mask
        trt.loc[hit] = value
        rule.loc[hit] = rule_name

    is_jpcp_or_crcp = pav.isin(["JPCP", "CRCP"])
    non_j = ~is_jpcp_or_crcp

    poor = (iri_rating == 3) | (cracking_r == 3) | (rutting_r == 3)
    is_interstate = functional.isin([1, 11])

    acol_family = pav.isin(["Other", "CRCP+FC", "JPCP+FC"])
    issue = (no_rehab >= 3) | (found_issue == 1)

    pred_high = (crack > 12) | (rut > 0.4) | (iri > 105)
    pred_very_high = (crack > 15) | (rut > 0.4) | (iri > 143)

    schd_9 = schd >= 9
    schd_8 = schd >= 8
    schd_7 = schd >= 7
    schd_3 = schd >= 3

    assign(is_jpcp_or_crcp & (iri > 143), "DIAMOND_GRIND", "JPCP/CRCP -> iri>143")
    assign(is_jpcp_or_crcp & ~(iri > 143) & (crack > 5), "REPAIR", "JPCP/CRCP -> crack>5")
    assign(is_jpcp_or_crcp & ~(iri > 143) & ~(crack > 5), "DO_NOTHING", "JPCP/CRCP -> else")

    m_poor = non_j & poor
    m_good = non_j & ~poor

    assign(m_poor & acol_family & issue, "RECONSTRUCTION_OR_RR_1INCH_FR", "poor & acol_family & issue")
    assign(m_poor & acol_family & ~issue, "RR_1INCH_FR", "poor & acol_family & ~issue")
    assign(m_poor & ~acol_family & issue & is_interstate, "RECONSTRUCTION_OR_RR_5INCH_AC_FR_OR_RR_3INCH_AC_FR", "poor & issue & interstate")
    assign(m_poor & ~acol_family & issue & ~is_interstate & (aadt > 15000), "RECONSTRUCTION_OR_RR_4INCH_AC_FR_OR_RR_3INCH_AC_FR", "poor & issue & non-interstate & AADT>15000")
    assign(m_poor & ~acol_family & issue & ~is_interstate & ~(aadt > 15000), "RR_3INCH_AC_FR", "poor & issue & non-interstate & AADT<=15000")

    m_base = m_poor & ~acol_family & ~issue & is_interstate
    assign(m_base & pred_high & (aadt > 10000), "RR_5INCH_AC_FR_OR_RR_3INCH_AC_FR", "poor interstate pred_high AADT>10000")
    assign(m_base & pred_high & ~(aadt > 10000) & (aadt > 5000), "RR_4INCH_AC_FR_OR_RR_3INCH_AC_FR", "poor interstate pred_high 5000<AADT<=10000")
    assign(m_base & pred_high & ~(aadt > 10000) & ~(aadt > 5000), "RR_3INCH_AC_FR", "poor interstate pred_high AADT<=5000")
    assign(m_base & ~pred_high & (aadt > 5000) & schd_9, "SR_3INCH_AC_MS", "poor interstate !pred_high AADT>5000 SCHD>=9")
    assign(m_base & ~pred_high & (aadt > 5000) & ~schd_9, "CONTINUE_TRACKING", "poor interstate !pred_high AADT>5000 SCHD<9")
    assign(m_base & ~pred_high & ~(aadt > 5000) & schd_8, "MS_1_PASS", "poor interstate !pred_high AADT<=5000 SCHD>=8")
    assign(m_base & ~pred_high & ~(aadt > 5000) & ~schd_8, "CONTINUE_TRACKING", "poor interstate !pred_high AADT<=5000 SCHD<8")

    m_base = m_poor & ~acol_family & ~issue & ~is_interstate
    assign(m_base & pred_very_high & (aadt > 15000), "RR_4INCH_AC_FR_OR_RR_3INCH_AC_FR", "poor non-interstate pred_very_high AADT>15000")
    assign(m_base & pred_very_high & ~(aadt > 15000) & (aadt > 3000), "RR_3INCH_AC_FR", "poor non-interstate pred_very_high 3000<AADT<=15000")
    assign(m_base & pred_very_high & ~(aadt > 15000) & ~(aadt > 3000) & (aadt > 1000), "RR_2p5INCH_AC_FR", "poor non-interstate pred_very_high 1000<AADT<=3000")
    assign(m_base & pred_very_high & ~(aadt > 15000) & ~(aadt > 3000) & ~(aadt > 1000), "RR_2INCH_AC_FR", "poor non-interstate pred_very_high AADT<=1000")
    assign(m_base & ~pred_very_high & (aadt > 5000) & schd_8, "SR_3INCH_AC_MS", "poor non-interstate !pred_very_high AADT>5000 SCHD>=8")
    assign(m_base & ~pred_very_high & (aadt > 5000) & ~schd_8, "CONTINUE_TRACKING", "poor non-interstate !pred_very_high AADT>5000 SCHD<8")
    assign(m_base & ~pred_very_high & ~(aadt > 5000) & (aadt > 3000) & schd_9, "MILL_FR_AND_MICRO_CAPE_SEAL", "poor non-interstate mid AADT SCHD>=9")
    assign(m_base & ~pred_very_high & ~(aadt > 5000) & (aadt > 3000) & ~schd_9, "CONTINUE_TRACKING", "poor non-interstate mid AADT SCHD<9")
    assign(m_base & ~pred_very_high & ~(aadt > 5000) & ~(aadt > 3000) & schd_8, "CRACKSEAL_AND_CHIPSEAL", "poor non-interstate low AADT SCHD>=8")
    assign(m_base & ~pred_very_high & ~(aadt > 5000) & ~(aadt > 3000) & ~schd_8, "CONTINUE_TRACKING", "poor non-interstate low AADT SCHD<8")

    crack_under_5 = crack < 5
    crack_under_8 = crack < 8
    crack_ge_2 = crack >= 2
    rut_gt_01 = rut > 0.1

    m_i = m_good & is_interstate
    m_n = m_good & ~is_interstate

    assign(m_i & crack_under_5 & rut_gt_01 & crack_ge_2 & schd_8, "RR_0p5INCH_FR", "good interstate RR0.5")
    assign(m_i & crack_under_5 & rut_gt_01 & ~crack_ge_2 & schd_9, "MS_2_PASS", "good interstate MS2")
    assign(m_i & crack_under_5 & ~rut_gt_01 & crack_ge_2 & schd_8, "RR_0p5INCH_FR_OR_MS_1_PASS", "good interstate RR0.5/MS1")
    assign(m_i & crack_under_5 & ~rut_gt_01 & ~crack_ge_2 & schd_3, "FOG_COAT", "good interstate fog")
    assign(m_i & ~crack_under_5 & crack_under_8 & schd_8, "RR_0p5INCH_FR", "good interstate 5-8")
    assign(m_i & ~crack_under_5 & ~crack_under_8 & schd_8, "SR_3INCH_AC_MS", "good interstate crack>=8")

    assign(m_n & crack_under_5 & rut_gt_01 & crack_ge_2 & schd_8, "RR_0p5INCH_FR", "good non-int RR0.5")
    assign(m_n & crack_under_5 & rut_gt_01 & ~crack_ge_2 & schd_9, "MS_2_PASS", "good non-int MS2")
    assign(m_n & crack_under_5 & ~rut_gt_01 & crack_ge_2 & schd_8, "MS_1_PASS_OR_CHIPSEAL", "good non-int MS1/CS")
    assign(m_n & crack_under_5 & ~rut_gt_01 & ~crack_ge_2 & schd_3, "FOG_COAT", "good non-int fog")
    assign(m_n & ~crack_under_5 & crack_under_8 & (aadt > 5000) & schd_9, "MILL_FR_AND_MICRO_CAPE_SEAL", "good non-int mill")
    assign(m_n & ~crack_under_5 & crack_under_8 & ~(aadt > 5000) & schd_7, "CRACKSEAL_AND_CHIPSEAL", "good non-int crackseal")
    assign(m_n & ~crack_under_5 & ~crack_under_8 & schd_8, "SR_3INCH_AC_MS", "good non-int crack>=8")

    assign(trt.eq(SENTINEL), "CONTINUE_TRACKING", "IMPLICIT_ELSE")

    if normalize_continue_tracking:
        trt = trt.replace("CONTINUE_TRACKING", "DO_NOTHING")

    undec_mask = trt.eq(SENTINEL)
    if undec_mask.any():
        raise ValueError(f"{undec_mask.sum()} rows UNDECIDED.")

    out = pd.DataFrame({"Treatment": trt, "RuleFired": rule}, index=gdf.index)
    return out if return_audit else out["Treatment"]

decision_output = fast_decision_tree_safe(gdf)
gdf["Treatment"] = decision_output["Treatment"]
print(gdf[['TARGET_FID', 'PAVEMENT_T', 'Treatment']].head(20))

In [ ]:
# ============================================================
# Simulation infrastructure:
#   replace_or_treatment, build_simulation_mapping,
#   prepare_mapping_cache, aggregate_weighted_to_merged (slow),
#   aggregate_weighted_to_merged_fast, propagate_treatments_from_merged_to_gdf,
#   _ensure_merged_functional_once, drop_sim_columns
# Also initialize Treatment_Table (needed by replace_or_treatment).
# ============================================================

# Minimal Treatment_Table needed by replace_or_treatment.
# Minimal treatment lookup table required by replace_or_treatment.
Treatment_Table = pd.DataFrame(
    index=list(treatments_cost.keys()),
    data={'Total': [0.0] * len(treatments_cost)}
)

def replace_or_treatment(treatment, treatment_table):
    """Replace compound 'X_OR_Y' treatment names with the single best option."""
    if '_OR_' in treatment:
        treatment_parts = treatment.split('_OR_')
        max_treatment = max(
            treatment_parts,
            key=lambda t: treatment_table.at[t, 'Total'] if t in treatment_table.index else float('-inf')
        )
        return max_treatment
    return treatment

def build_simulation_mapping(gdf, gdf_merged):
    """
    Link base segments to merged segments using:
      - gdf_merged['Original_I'] (list of FID_Ns)
      - gdf['FID_N']
    Returns a DataFrame with columns ['base_idx', 'merged_idx'].
    """
    exploded = gdf_merged[['Original_I']].explode('Original_I')
    exploded['merged_idx'] = exploded.index
    exploded = exploded.rename(columns={'Original_I': 'common_id'})
    base_df = pd.DataFrame({'base_idx': gdf.index, 'common_id': gdf['FID_N']})
    mapping = exploded.merge(base_df, on='common_id', how='inner')
    return mapping[['base_idx', 'merged_idx']]

def prepare_mapping_cache(gdf_base, gdf_merged, mapping_df):
    """
    Precompute fast array mappings for base->merged aggregation and propagation.
    Returns (merged_index, base_merged_pos, valid_map).
    """
    merged_index = pd.Index(gdf_merged.index)
    base_to_merged = mapping_df.set_index("base_idx")["merged_idx"].reindex(gdf_base.index)
    base_merged_pos = merged_index.get_indexer(base_to_merged.to_numpy())
    valid_map = base_merged_pos >= 0
    return merged_index, base_merged_pos, valid_map

def aggregate_weighted_to_merged(gdf, gdf_merged, cols_to_agg, mapping, weight_col='Lane_Miles'):
    """Weighted aggregation gdf -> gdf_merged (slow, reference version)."""
    subset = gdf.loc[mapping['base_idx'], cols_to_agg + [weight_col, 'Functional']].copy()
    subset.index = mapping.index
    joined = pd.concat([mapping, subset], axis=1)

    joined[weight_col] = joined[weight_col].fillna(0.0)
    groups = joined.groupby('merged_idx')
    weight_sums = groups[weight_col].sum().replace(0, np.nan)

    for col in cols_to_agg:
        weighted_vals = joined[col] * joined[weight_col]
        agg_vals = weighted_vals.groupby(joined['merged_idx']).sum() / weight_sums
        if col not in gdf_merged.columns:
            gdf_merged[col] = np.nan
        gdf_merged.loc[agg_vals.index, col] = agg_vals
        if gdf_merged[col].isnull().any():
            fill_val = gdf[col].median()
            if pd.isna(fill_val):
                fill_val = 0
            gdf_merged[col] = gdf_merged[col].fillna(fill_val)

    mode_vals = joined.groupby('merged_idx')['Functional'].agg(
        lambda x: pd.Series.mode(x)[0] if not pd.Series.mode(x).empty else np.nan
    )
    gdf_merged.loc[mode_vals.index, 'Functional'] = mode_vals
    if gdf_merged['Functional'].isnull().any():
        global_mode = gdf['Functional'].mode()[0] if not gdf['Functional'].mode().empty else 0
        gdf_merged['Functional'] = gdf_merged['Functional'].fillna(global_mode)

    for col in ['New_AvgIRI', 'New_HPMS_Cracking', 'New_Rutting']:
        if col in gdf_merged.columns:
            gdf_merged[col] = pd.to_numeric(gdf_merged[col], errors='coerce').fillna(0)

    if 'New_AvgIRI' in gdf_merged.columns:
        gdf_merged['IRI_Rating'] = np.select(
            [gdf_merged['New_AvgIRI'] <= 95, gdf_merged['New_AvgIRI'] <= 170], [1, 2], default=3)
    if 'New_HPMS_Cracking' in gdf_merged.columns:
        gdf_merged['Cracking_R'] = np.select(
            [gdf_merged['New_HPMS_Cracking'] <= 5, gdf_merged['New_HPMS_Cracking'] <= 20], [1, 2], default=3)
    if 'New_Rutting' in gdf_merged.columns:
        gdf_merged['Rutting_Ra'] = np.select(
            [gdf_merged['New_Rutting'] <= 0.2, gdf_merged['New_Rutting'] <= 0.4], [1, 2], default=3)

    return gdf_merged

def aggregate_weighted_to_merged_fast(
    gdf_base, gdf_merged, cols_to_agg, base_merged_pos, valid_map,
    weight_col='Lane_Miles', compute_functional=False, compute_ratings=True,
    fill_missing_with_base_median=True
):
    """Fast weighted aggregation base -> merged using np.bincount."""
    pos = base_merged_pos
    M = len(gdf_merged)

    w = gdf_base[weight_col].to_numpy(float)
    w_valid = w[valid_map]
    pos_valid = pos[valid_map]

    wsum = np.bincount(pos_valid, weights=w_valid, minlength=M)
    wsum_safe = np.where(wsum == 0.0, np.nan, wsum)

    for col in cols_to_agg:
        x = pd.to_numeric(gdf_base[col], errors="coerce").to_numpy(float)
        x_valid = x[valid_map]
        xw = np.bincount(pos_valid, weights=(x_valid * w_valid), minlength=M)
        agg = xw / wsum_safe
        gdf_merged[col] = agg
        if fill_missing_with_base_median:
            if np.isnan(gdf_merged[col]).any():
                fill_val = pd.to_numeric(gdf_base[col], errors="coerce").median()
                if pd.isna(fill_val):
                    fill_val = 0.0
                gdf_merged[col] = np.nan_to_num(gdf_merged[col], nan=float(fill_val))

    if compute_functional and ("Functional" in gdf_base.columns):
        tmp = pd.DataFrame({
            "merged_pos": pos_valid,
            "Functional": gdf_base.loc[gdf_base.index[valid_map], "Functional"].to_numpy()
        })
        mode_vals = tmp.groupby("merged_pos")["Functional"].agg(
            lambda x: pd.Series.mode(x)[0] if not pd.Series.mode(x).empty else np.nan
        )
        gdf_merged["Functional"] = np.nan
        gdf_merged.iloc[mode_vals.index.values, gdf_merged.columns.get_loc("Functional")] = mode_vals.to_numpy()
        if gdf_merged["Functional"].isnull().any():
            global_mode = gdf_base["Functional"].mode()[0] if not gdf_base["Functional"].mode().empty else 0
            gdf_merged["Functional"] = gdf_merged["Functional"].fillna(global_mode)

    if compute_ratings:
        for c in ['New_AvgIRI', 'New_HPMS_Cracking', 'New_Rutting']:
            if c in gdf_merged.columns:
                gdf_merged[c] = pd.to_numeric(gdf_merged[c], errors='coerce').fillna(0.0)
        if 'New_AvgIRI' in gdf_merged.columns:
            gdf_merged['IRI_Rating'] = np.select(
                [gdf_merged['New_AvgIRI'] <= 95, gdf_merged['New_AvgIRI'] <= 170], [1, 2], default=3)
        if 'New_HPMS_Cracking' in gdf_merged.columns:
            gdf_merged['Cracking_R'] = np.select(
                [gdf_merged['New_HPMS_Cracking'] <= 5, gdf_merged['New_HPMS_Cracking'] <= 20], [1, 2], default=3)
        if 'New_Rutting' in gdf_merged.columns:
            gdf_merged['Rutting_Ra'] = np.select(
                [gdf_merged['New_Rutting'] <= 0.2, gdf_merged['New_Rutting'] <= 0.4], [1, 2], default=3)

    return gdf_merged

def propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, treatment_col, mapping):
    """Copy treatment decisions from gdf_merged[treatment_col] to gdf[treatment_col]."""
    treatments_map = mapping.set_index('base_idx')['merged_idx'].map(gdf_merged[treatment_col])
    gdf[treatment_col] = treatments_map.reindex(gdf.index).fillna('DO_NOTHING')
    return gdf

def _ensure_merged_functional_once(gdf_base0, gdfm_base0, mapping_df0):
    """Fill gdfm_base0['Functional'] using mode of base Functional (in-place)."""
    if "Functional" not in gdf_base0.columns:
        raise KeyError("gdf_base0 is missing required column 'Functional'.")
    if "Functional" not in gdfm_base0.columns:
        gdfm_base0["Functional"] = np.nan

    tmp = pd.DataFrame({
        "merged_idx": mapping_df0["merged_idx"].to_numpy(),
        "Functional": gdf_base0.loc[mapping_df0["base_idx"].to_numpy(), "Functional"].to_numpy()
    })
    mode_vals = tmp.groupby("merged_idx")["Functional"].agg(
        lambda x: pd.Series.mode(x)[0] if not pd.Series.mode(x).empty else np.nan
    )
    gdfm_base0.loc[mode_vals.index, "Functional"] = mode_vals.to_numpy()
    if gdfm_base0["Functional"].isnull().any():
        global_mode = gdf_base0["Functional"].mode()[0] if not gdf_base0["Functional"].mode().empty else 0
        gdfm_base0["Functional"] = gdfm_base0["Functional"].fillna(global_mode)

def drop_sim_columns(df):
    """Drop columns created during simulation so base copies stay clean."""
    prefixes = ("Treatment_Year_", "Cost_Year_", "AvgIRI_Year_", "Crack_Year_", "Rutting_Year_",
                "UnitCost_Year_", "TotalCost_Year_", "Benefit_Year_", "Final_UnitCost_Year_",
                "Final_TotalCost_Year_")
    cols = [c for c in df.columns if c.startswith(prefixes)]
    return df.drop(columns=cols, errors="ignore")

print("Decision tree and simulation infrastructure functions defined.")

## Section 7: Treatment Resets, Deterioration, and User Cost

These functions update rehabilitation history, reset condition after treatment, advance pavement performance with Monte Carlo uncertainty, and calculate annual road-user cost from traffic and IRI.


In [ ]:
# ============================================================
# Treatment state initialization and helper functions:
#   update_rehabilitation, update_schd_year
# Also initialize state columns on gdf.
# ============================================================

gdf['New_Rutting'] = gdf['Rutting']
gdf['New_AvgIRI'] = gdf['AvgIRI']
gdf['New_HPMS_Cracking'] = gdf['HPMS_Crack']
gdf['Foundation_Issue'] = 0
_init_rng_nb = np.random.default_rng(SEED)
gdf['No_of_Rehab'] = _init_rng_nb.integers(0, 4, size=len(gdf))
gdf['SCHD_Year'] = _init_rng_nb.integers(0, 10, size=len(gdf))

gdf['Rutting_Hold_Year'] = 0
gdf['Rutting_Recovery_Year'] = 0
gdf['Rutting_Recovery_Year_Updated'] = 0
gdf['Delta_Rutting_Recovery'] = 0.0
gdf['Pre_Treatment_Rutting'] = 0.0

gdf['Crack_Hold_Year'] = 0
gdf['Crack_Recovery_Year'] = 0
gdf['Crack_Recovery_Year_Updated'] = 0
gdf['Delta_Crack_Recovery'] = 0.0
gdf['Pre_Treatment_Crack'] = 0.0
gdf['pavement_age'] = _init_rng_nb.integers(0, 21, size=len(gdf))

def update_rehabilitation(gdf, treatment_column):
    """Increment No_of_Rehab for segments that received an active treatment."""
    mask = ~gdf[treatment_column].isin(["DO_NOTHING", "CONTINUE_TRACKING"])
    gdf.loc[mask, 'No_of_Rehab'] += 1

def update_schd_year(gdf, treatment_column):
    """Increment SCHD_Year for untreated, reset to 0 for treated segments."""
    mask = gdf[treatment_column].isin(["DO_NOTHING", "CONTINUE_TRACKING"])
    gdf.loc[mask, 'SCHD_Year'] += 1
    gdf.loc[~mask, 'SCHD_Year'] = 0

print("Treatment state initialized.")

In [ ]:
# ============================================================
# apply_treatment_resets: reset IRI, crack, rutting after treatment.
# FULL if/elif chain for all treatment types.
# ============================================================

def apply_treatment_resets(gdf, year):
    """
    Reset IRI, Crack, Rutting after applying a treatment.
    Based on Arizona ADOT suggestions. Full if/elif chain for all treatment types.
    """
    treatment_column = f'Treatment_Year_{year}'

    treatment_masks = {
        'CRACKSEAL':                 gdf[treatment_column] == 'CRACKSEAL',
        'CRACKSEAL_AND_CHIPSEAL':    gdf[treatment_column] == 'CRACKSEAL_AND_CHIPSEAL',
        'MILL_FR_AND_MICRO_CAPE_SEAL': gdf[treatment_column] == 'MILL_FR_AND_MICRO_CAPE_SEAL',
        'MS_1_PASS':                 gdf[treatment_column] == 'MS_1_PASS',
        'MS_2_PASS':                 gdf[treatment_column] == 'MS_2_PASS',
        'RECONSTRUCTION':            gdf[treatment_column] == 'RECONSTRUCTION',
        'RR_0p5INCH_FR':             gdf[treatment_column] == 'RR_0p5INCH_FR',
        'RR_1INCH_FR':               gdf[treatment_column] == 'RR_1INCH_FR',
        'RR_2INCH_AC_FR':            gdf[treatment_column] == 'RR_2INCH_AC_FR',
        'RR_2p5INCH_AC_FR':          gdf[treatment_column] == 'RR_2p5INCH_AC_FR',
        'RR_3INCH_AC_FR':            gdf[treatment_column] == 'RR_3INCH_AC_FR',
        'RR_4INCH_AC_FR':            gdf[treatment_column] == 'RR_4INCH_AC_FR',
        'RR_5INCH_AC_FR':            gdf[treatment_column] == 'RR_5INCH_AC_FR',
        'SR_3INCH_AC_MS':            gdf[treatment_column] == 'SR_3INCH_AC_MS'
    }

    for treatment, mask in treatment_masks.items():
        if mask.any():
            if treatment == 'CRACKSEAL':
                gdf.loc[mask, 'Pre_Treatment_Crack'] = gdf.loc[mask, 'New_HPMS_Cracking']
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'Crack_Hold_Year'] = 2
                gdf.loc[mask, 'Crack_Recovery_Year'] = 2
                gdf.loc[mask, 'Crack_Recovery_Year_Updated'] = 2

            elif treatment == 'CRACKSEAL_AND_CHIPSEAL':
                gdf.loc[mask, 'New_AvgIRI'] *= 1.1
                gdf.loc[mask, 'Pre_Treatment_Crack'] = gdf.loc[mask, 'New_HPMS_Cracking']
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'Crack_Hold_Year'] = 2
                gdf.loc[mask, 'Crack_Recovery_Year'] = 4
                gdf.loc[mask, 'Crack_Recovery_Year_Updated'] = 4

            elif treatment == 'MILL_FR_AND_MICRO_CAPE_SEAL':
                gdf.loc[mask, 'Pre_Treatment_Rutting'] = gdf.loc[mask, 'New_Rutting']
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'Rutting_Hold_Year'] = 1
                gdf.loc[mask, 'Rutting_Recovery_Year'] = 2
                gdf.loc[mask, 'Rutting_Recovery_Year_Updated'] = 2
                gdf.loc[mask, 'Pre_Treatment_Crack'] = gdf.loc[mask, 'New_HPMS_Cracking']
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'Crack_Hold_Year'] = 3
                gdf.loc[mask, 'Crack_Recovery_Year'] = 5
                gdf.loc[mask, 'Crack_Recovery_Year_Updated'] = 5

            elif treatment == 'MS_1_PASS':
                gdf.loc[mask, 'New_AvgIRI'] *= 0.8
                gdf.loc[mask, 'Pre_Treatment_Crack'] = gdf.loc[mask, 'New_HPMS_Cracking']
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'Crack_Hold_Year'] = 2
                gdf.loc[mask, 'Crack_Recovery_Year'] = 4
                gdf.loc[mask, 'Crack_Recovery_Year_Updated'] = 4

            elif treatment == 'MS_2_PASS':
                gdf.loc[mask, 'New_AvgIRI'] *= 0.8
                gdf.loc[mask, 'Pre_Treatment_Rutting'] = gdf.loc[mask, 'New_Rutting']
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'Rutting_Hold_Year'] = 1
                gdf.loc[mask, 'Rutting_Recovery_Year'] = 2
                gdf.loc[mask, 'Rutting_Recovery_Year_Updated'] = 2
                gdf.loc[mask, 'Pre_Treatment_Crack'] = gdf.loc[mask, 'New_HPMS_Cracking']
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'Crack_Hold_Year'] = 2
                gdf.loc[mask, 'Crack_Recovery_Year'] = 5
                gdf.loc[mask, 'Crack_Recovery_Year_Updated'] = 5

            elif treatment == 'RECONSTRUCTION':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'No_of_Rehab'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

            elif treatment == 'FOG_COAT':
                gdf.loc[mask, 'Pre_Treatment_Crack'] = gdf.loc[mask, 'New_HPMS_Cracking']
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'Crack_Hold_Year'] = 2
                gdf.loc[mask, 'Crack_Recovery_Year'] = 4
                gdf.loc[mask, 'Crack_Recovery_Year_Updated'] = 4

            elif treatment == 'RR_0p5INCH_FR':
                gdf.loc[mask, 'New_AvgIRI'] *= 0.6
                gdf.loc[mask, 'Pre_Treatment_Rutting'] = gdf.loc[mask, 'New_Rutting']
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'Rutting_Hold_Year'] = 1
                gdf.loc[mask, 'Rutting_Recovery_Year'] = 5
                gdf.loc[mask, 'Rutting_Recovery_Year_Updated'] = 5
                gdf.loc[mask, 'Pre_Treatment_Crack'] = gdf.loc[mask, 'New_HPMS_Cracking']
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'Crack_Hold_Year'] = 5
                gdf.loc[mask, 'Crack_Recovery_Year'] = 5
                gdf.loc[mask, 'Crack_Recovery_Year_Updated'] = 5

            elif treatment == 'RR_1INCH_FR':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

            elif treatment == 'RR_2p5INCH_AC_FR':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

            elif treatment == 'RR_2INCH_AC_FR':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

            elif treatment == 'RR_3INCH_AC_FR':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

            elif treatment == 'RR_4INCH_AC_FR':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

            elif treatment == 'RR_5INCH_AC_FR':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

            elif treatment == 'SR_3INCH_AC_MS':
                gdf.loc[mask, 'New_AvgIRI'] = 60
                gdf.loc[mask, 'New_Rutting'] = 0
                gdf.loc[mask, 'New_HPMS_Cracking'] = 0
                gdf.loc[mask, 'pavement_age'] = 0

    return gdf

print("apply_treatment_resets defined.")

In [ ]:
# ============================================================
# apply_treatment_effects: deterioration update for IRI, rutting, cracking.
# Uses SEPARATE cov_iri, cov_rut, cov_crack parameters (split from original
# single 'cov' — allows per-indicator uncertainty control).
# IRI progression uses FAST vectorized formula (from worstfirst cell 28),
# NOT the slow compute_iri_projection apply() call (from costbenefit).
# ============================================================

def apply_treatment_effects(
    gdf,
    year,
    rng=None,
    cov=None,          # legacy: if set, overrides cov_iri/cov_rut/cov_crack
    cov_iri=None,
    cov_rut=None,
    cov_crack=None,
    z_iri_global=0.0,
    z_rut_global=0.0,
    z_crack_global=0.0
):
    """
    Apply performance deterioration for one year.

    Parameters
    ----------
    cov : float, optional
        If provided, overrides cov_iri/cov_rut/cov_crack (backward compat).
    cov_iri / cov_rut / cov_crack : float, optional
        Per-indicator coefficient of variation. Defaults to global COV_IRI etc.
    z_*_global : float
        Standard normal draw sampled once per MC run (controls direction of shock).
    """
    import numpy as np

    # Resolve cov parameters
    if cov is not None:
        _cov_iri = _cov_rut = _cov_crack = cov
    else:
        _cov_iri   = cov_iri   if cov_iri   is not None else COV_IRI
        _cov_rut   = cov_rut   if cov_rut   is not None else COV_RUTTING
        _cov_crack = cov_crack if cov_crack is not None else COV_FATIGUE

    treatment_column = f"Treatment_Year_{year-1}"

    # =========================================================
    # Treatment effects on Rutting
    # =========================================================
    rutting_hold_mask = gdf["Rutting_Hold_Year"] > 0
    gdf.loc[rutting_hold_mask, "New_Rutting"] = 0
    gdf.loc[rutting_hold_mask, "Rutting_Hold_Year"] -= 1

    rutting_recovery_mask = (gdf["Rutting_Hold_Year"] == 0) & (gdf["Rutting_Recovery_Year_Updated"] > 0)
    if rutting_recovery_mask.any():
        rutting_recovery_indices = gdf[rutting_recovery_mask].index

        first_recovery_mask_rutting = gdf["Rutting_Recovery_Year"] == gdf["Rutting_Recovery_Year_Updated"]
        first_recovery_indices_rutting = gdf[rutting_recovery_mask & first_recovery_mask_rutting].index

        gdf.loc[first_recovery_indices_rutting, "Delta_Rutting_Recovery"] = (
            (gdf.loc[first_recovery_indices_rutting, "Pre_Treatment_Rutting"]
             - gdf.loc[first_recovery_indices_rutting, "New_Rutting"])
            / gdf.loc[first_recovery_indices_rutting, "Rutting_Recovery_Year"]
        )

        gdf.loc[rutting_recovery_indices, "New_Rutting"] += gdf.loc[rutting_recovery_indices, "Delta_Rutting_Recovery"]
        gdf.loc[rutting_recovery_indices, "Rutting_Recovery_Year_Updated"] -= 1

    post_recovery_mask_rutting = ~rutting_hold_mask & ~rutting_recovery_mask

    if post_recovery_mask_rutting.any():
        ESAL_DEF = 20 * 10**6
        idx = post_recovery_mask_rutting

        growth_rut = (
            gdf.loc[idx, "a_rutt"] * gdf.loc[idx, "pavement_age"]
            + gdf.loc[idx, "b_rutt"] * (gdf.loc[idx, "ESAL_20_year"] / ESAL_DEF)
            + gdf.loc[idx, "c_rutt"] * gdf.loc[idx, "svf"]
        )

        growth_rut_stoch = growth_rut * (1 + _cov_rut * z_rut_global)
        growth_rut_stoch = np.maximum(growth_rut_stoch, 0)
        gdf.loc[idx, "New_Rutting"] = gdf.loc[idx, "New_Rutting"] + growth_rut_stoch

    # =========================================================
    # Treatment effects on Crack
    # =========================================================
    crack_hold_mask = gdf["Crack_Hold_Year"] > 0
    gdf.loc[crack_hold_mask, "New_HPMS_Cracking"] = 0
    gdf.loc[crack_hold_mask, "Crack_Hold_Year"] -= 1

    crack_recovery_mask = (gdf["Crack_Hold_Year"] == 0) & (gdf["Crack_Recovery_Year_Updated"] > 0)
    if crack_recovery_mask.any():
        crack_recovery_indices = gdf[crack_recovery_mask].index

        first_recovery_mask_crack = gdf["Crack_Recovery_Year"] == gdf["Crack_Recovery_Year_Updated"]
        first_recovery_indices_crack = gdf[crack_recovery_mask & first_recovery_mask_crack].index

        gdf.loc[first_recovery_indices_crack, "Delta_Crack_Recovery"] = (
            (gdf.loc[first_recovery_indices_crack, "Pre_Treatment_Crack"]
             - gdf.loc[first_recovery_indices_crack, "New_HPMS_Cracking"])
            / gdf.loc[first_recovery_indices_crack, "Crack_Recovery_Year"]
        )

        gdf.loc[crack_recovery_indices, "New_HPMS_Cracking"] += gdf.loc[crack_recovery_indices, "Delta_Crack_Recovery"]
        gdf.loc[crack_recovery_indices, "Crack_Recovery_Year_Updated"] -= 1

    post_recovery_mask_crack = ~crack_hold_mask & ~crack_recovery_mask

    if post_recovery_mask_crack.any():
        ESAL_DEF = 20 * 10**6
        idx = post_recovery_mask_crack

        growth_crack = np.maximum(
            1.002,
            1
            + gdf.loc[idx, "a_crack"]
            + gdf.loc[idx, "b_crack"] * (gdf.loc[idx, "ESAL_20_year"] / ESAL_DEF)
            - gdf.loc[idx, "c_crack"] * gdf.loc[idx, "pavement_age"]
        )

        # Full deterministic cracking increment = multiplicative growth part + seasonal SVF term.
        # Apply the stochastic multiplier to the entire increment so the model exactly follows
        # TP_{t+1} = TP_t + delta_TP * (1 + CoV * z), consistent with IRI and rutting.
        delta_crack = (
            gdf.loc[idx, "New_HPMS_Cracking"] * (growth_crack - 1)
            + gdf.loc[idx, "d_crack"] * gdf.loc[idx, "svf"]
        )
        delta_crack_stoch = delta_crack * (1 + _cov_crack * z_crack_global)

        gdf.loc[idx, "New_HPMS_Cracking"] = np.minimum(
            gdf.loc[idx, "New_HPMS_Cracking"] + delta_crack_stoch,
            100
        )

    # =========================================================
    # Apply treatment effects on IRI
    # FAST vectorized formula (worstfirst version).
    # growth_iri = a_iri * pavement_age + b_iri * (ESAL/ESAL_DEF) + c_iri * svf
    # =========================================================
    ESAL_DEF = 20 * 10**6

    growth_iri = (
        gdf["a_iri"] * gdf["pavement_age"]
        + gdf["b_iri"] * (gdf["ESAL_20_year"] / ESAL_DEF)
        + gdf["c_iri"] * gdf["svf"]
    )

    growth_iri_stoch = growth_iri * (1 + _cov_iri * z_iri_global)
    growth_iri_stoch = np.maximum(growth_iri_stoch, 0)

    gdf.loc[:, "New_AvgIRI"] = gdf["New_AvgIRI"] + growth_iri_stoch

    # Increment age for every segment once per year, independent of cracking
    # or rutting hold/recovery state. Physical aging is always-on; hold/recovery
    # only controls the distress value, not the passage of time.
    gdf["pavement_age"] += 1

    return gdf

print("apply_treatment_effects defined.")

In [ ]:
# ============================================================
# user_cost_by_year_from_gdf: compute user cost per year from IRI.
# Uses mid-year IRI approach matching the deterministic formulation.
# ============================================================
import numpy as np

def user_cost_by_year_from_gdf(
    gdf_res,
    years=10,
    annual_traffic_growth=0.00001,
    base_iri=USER_COST_IRI_BASE,  # IRI reference baseline (inch/mile) — good pavement threshold
    vehicle_percentages=None,
    return_million_dollars=True
):
    """
    Compute user cost (fuel cost from excess IRI) per year.
    Uses AvgIRI_Year_0..AvgIRI_Year_{years-1} and AADT_Year_1..{years}.
    Returns array of length 'years' (million $ per year by default).
    """
    if vehicle_percentages is None:
        vehicle_percentages = {
            "Passenger Vehicle": 0.70,
            "Small Truck":       0.10,
            "Medium Truck":      0.15,
            "Large Truck":       0.05,
        }

    fuel_conversion_rates = {
        "Passenger Vehicle": 126.3, "Small Truck": 126.3,
        "Medium Truck": 145.0, "Large Truck": 145.0
    }
    fuel_cons_price_factors = {
        "Passenger Vehicle": 3.912, "Small Truck": 3.912,
        "Medium Truck": 3.966, "Large Truck": 3.966
    }
    energy_coefficients = {
        "Passenger Vehicle": {"ka": 0.67,  "kc": 0.000281, "dc": 0.2186,  "da": 2175.7, "b": -16.931, "p": 33753},
        "Small Truck":       {"ka": 0.768, "kc": 0.000125, "dc": 0.30769, "da": 7010.8, "b": -73.026, "p": 117880},
        "Medium Truck":      {"ka": 0.918, "kc": 0.000133, "dc": 0.97418, "da": 9299.3, "b": -139.58, "p": 109380},
        "Large Truck":       {"ka": 1.4,   "kc": 0.000136, "dc": 2.39,    "da": 19225,  "b": -264.32, "p": 82782},
    }

    if "AADT_Year_1" not in gdf_res.columns:
        gdf_res["AADT_Year_1"] = gdf_res["AADT"]

    for y in range(2, years + 1):
        col = f"AADT_Year_{y}"
        prev = f"AADT_Year_{y-1}"
        if col not in gdf_res.columns:
            gdf_res[col] = gdf_res[prev] * (1.0 + annual_traffic_growth)

    needed = [f"AvgIRI_Year_{k}" for k in range(0, years)]
    missing = [c for c in needed if c not in gdf_res.columns]
    if missing:
        raise KeyError(f"Missing {missing[:5]}... Need AvgIRI_Year_0..AvgIRI_Year_{years-1}.")

    speed = gdf_res["AVERAGE_SP"]
    seg_miles = (gdf_res["ToMeasure"] - gdf_res["FromMeasur"])

    totals = np.zeros(years, dtype=float)

    for year in range(years):
        iri = gdf_res[f"AvgIRI_Year_{year}"]
        aadt = gdf_res[f"AADT_Year_{year+1}"]

        year_cost = 0.0

        for vtype, pct in vehicle_percentages.items():
            c = energy_coefficients[vtype]
            ka, kc, dc, da, b, p = c["ka"], c["kc"], c["dc"], c["da"], c["b"], c["p"]

            fuel_mj_per_gal = fuel_conversion_rates[vtype]
            fuel_price = fuel_cons_price_factors[vtype]

            energy_base = (
                (p / speed) + (ka * base_iri + da) + (b * speed) + ((kc * base_iri) + dc) * (speed ** 2)
            ) / 1000.0

            energy_actual = (
                (p / speed) + (ka * iri + da) + (b * speed) + ((kc * iri) + dc) * (speed ** 2)
            ) / 1000.0

            dE = energy_actual - energy_base
            dE_pos = np.where(dE > 0, dE, 0.0)

            total_MJ = dE_pos * seg_miles * aadt * pct * 365.0
            gallons = total_MJ / fuel_mj_per_gal
            cost = gallons * fuel_price

            year_cost += float(np.nansum(cost))

        if return_million_dollars:
            year_cost /= 1e6

        totals[year] = year_cost

    return totals

def calculate_total_mr_emissions(gdf_result, Treatment_Table, years):
    """
    Stub: not used in this study. Returns 0.0 for API compatibility
    with Monte Carlo worker functions.
    """
    return 0.0

print("user_cost_by_year_from_gdf defined.")

## Section 8: Prioritization Methods

The same treatment eligibility, deterioration, and cost logic is retained for all options. Only the ordering of eligible projects under a constrained annual budget changes:

- `_simulate_years_worst_first` ranks merged segments by pavement distress.
- `_simulate_years_benefit_cost` implements the Preservation strategy — ranks projects by benefit per agency dollar.
- `_simulate_years_benefit_cost_aadt` implements Preservation (Considered AADT) — additionally weights benefit-cost priority by AADT².


## Option 1 — Worst-first Prioritization

Worst-first directs available funding to the most distressed eligible merged segments first, using a composite IRI, cracking, and rutting ranking.


In [ ]:
# ============================================================
# apply_budget_constraint_merged_worst_first:
# Ranks merged segments by combined distress (IRI/crack/rutting),
# selects worst first until budget is exhausted.
# ============================================================

def apply_budget_constraint_merged_worst_first(
    gdf,
    gdf_merged,
    mapping,
    year,
    budget,
    treatments,
    rng=None,
    fixed_unit_costs=None,
    keep_cost_columns=False,
    return_paid_cost_total=True
):
    """
    Worst-first budget constraint at merged level.
    Ranks by combined distress: 0.25*IRI/8 + 0.4*crack/4 + 0.1*rut*25.
    Selects segments from worst to best until budget is exhausted.
    """
    import numpy as np
    import pandas as pd

    if rng is None:
        rng = np.random.default_rng(123)

    treatment_col = f"Treatment_Year_{year}"

    # --- Combined distress factor (numpy) ---
    weights = {"crack": 0.4, "iri": 0.25, "rutting": 0.1}
    scaled_iri     = gdf_merged["New_AvgIRI"].to_numpy(float) / 8.0
    scaled_crack   = gdf_merged["New_HPMS_Cracking"].to_numpy(float) / 4.0
    scaled_rutting = gdf_merged["New_Rutting"].to_numpy(float) * 25.0

    combined = (
        scaled_iri * weights["iri"]
        + scaled_crack * weights["crack"]
        + scaled_rutting * weights["rutting"]
    )

    # --- Lane miles (base) ---
    if "lane_miles_cal" in gdf.columns:
        lane_miles = gdf["lane_miles_cal"].to_numpy(float)
    else:
        lane_miles = ((gdf["ToMeasure"] - gdf["FromMeasur"]) * gdf["Number_of_"]).to_numpy(float)

    # --- Unit costs: FIXED for whole horizon OR sampled per call ---
    if fixed_unit_costs is not None:
        sampled_unit_costs = fixed_unit_costs
    else:
        uniq = pd.unique(gdf_merged[treatment_col].astype(str))
        sampled_unit_costs = {}
        for t in uniq:
            ln_mu = treatments.get(t, {}).get("ln_mu", None)
            ln_sigma = treatments.get(t, {}).get("ln_sigma", None)
            if ln_mu is None or ln_sigma is None or ln_sigma <= 0:
                sampled_unit_costs[t] = 0.0
            else:
                sampled_unit_costs[t] = float(rng.lognormal(mean=float(ln_mu), sigma=float(ln_sigma)))

    # --- Map base treatment ---
    if treatment_col in gdf.columns:
        base_treat = gdf[treatment_col].astype(str)
    else:
        merged_treat = gdf_merged[treatment_col].astype(str)
        base_treat = (
            mapping.set_index("base_idx")["merged_idx"]
            .map(merged_treat)
            .reindex(gdf.index)
            .fillna("DO_NOTHING")
            .astype(str)
        )

    # --- Candidate section cost at base level (vectorized) ---
    sampled_cost_per_lane = base_treat.map(sampled_unit_costs).fillna(0.0).to_numpy(float)
    section_cost = lane_miles * sampled_cost_per_lane

    cand_col = f"Cost_Year_{year}_candidate"
    paid_col = f"Cost_Year_{year}"
    if keep_cost_columns:
        gdf[cand_col] = section_cost

    # --- FAST merged cost aggregation ---
    merged_ids = gdf_merged.index.to_numpy()
    base_to_merged = mapping.set_index("base_idx")["merged_idx"].reindex(gdf.index)
    merged_index = pd.Index(merged_ids)
    base_merged_pos = merged_index.get_indexer(base_to_merged.to_numpy())

    valid_map_mask = base_merged_pos >= 0
    merged_cost = np.bincount(
        base_merged_pos[valid_map_mask],
        weights=section_cost[valid_map_mask],
        minlength=len(merged_ids)
    )

    # --- Build valid candidates arrays (merged level) ---
    treat_arr = gdf_merged[treatment_col].astype(str).to_numpy()
    excluded = np.array(["DO_NOTHING", "CONTINUE_TRACKING", "REPAIR", "DIAMOND_GRIND"], dtype=object)
    valid_mask = ~np.isin(treat_arr, excluded)

    if not valid_mask.any():
        paid_cost_total = 0.0
        if keep_cost_columns:
            gdf[paid_col] = 0.0
        if return_paid_cost_total:
            return gdf, gdf_merged, sampled_unit_costs, paid_cost_total
        return gdf, gdf_merged, sampled_unit_costs

    valid_idx = np.where(valid_mask)[0]
    valid_combined = combined[valid_idx]
    valid_cost = merged_cost[valid_idx]

    # --- Sort worst-first (descending combined distress) ---
    order = np.argsort(valid_combined)[::-1]
    valid_idx = valid_idx[order]
    valid_cost = valid_cost[order]

    # --- Select within budget: skip-and-continue greedy scan ---
    # Zero/negative-cost treatments are always kept (no budget consumed).
    # Positive-cost projects are funded in priority order if they fit in
    # the remaining budget; unaffordable projects are skipped rather than
    # blocking all lower-priority candidates that might still fit.
    remaining = float(budget)
    keep_mask = np.zeros(len(valid_idx), dtype=bool)
    for _i, _cost in enumerate(valid_cost):
        if _cost <= 0.0:
            keep_mask[_i] = True
        elif _cost <= remaining:
            keep_mask[_i] = True
            remaining -= _cost
    selected_pos = valid_idx[keep_mask]

    # Reject others (set treatment to DO_NOTHING at merged level)
    reject_pos = valid_idx[~keep_mask]
    if reject_pos.size > 0:
        gdf_merged.iloc[reject_pos, gdf_merged.columns.get_loc(treatment_col)] = "DO_NOTHING"

    # Propagate final merged decisions back to base segments
    gdf = propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, treatment_col, mapping)

    # --- Compute PAID costs at base level ---
    final_base_treat = gdf[treatment_col].astype(str)
    paid_mask = ~final_base_treat.isin(["DO_NOTHING", "CONTINUE_TRACKING", "REPAIR", "DIAMOND_GRIND"])

    final_cost_per_lane = final_base_treat.map(sampled_unit_costs).fillna(0.0).to_numpy(float)
    paid_section_cost = lane_miles * final_cost_per_lane
    paid_cost_total = float(paid_section_cost[paid_mask.to_numpy()].sum())

    if keep_cost_columns:
        gdf[paid_col] = 0.0
        gdf.loc[paid_mask, paid_col] = paid_section_cost[paid_mask.to_numpy()]

    if return_paid_cost_total:
        return gdf, gdf_merged, sampled_unit_costs, paid_cost_total

    return gdf, gdf_merged, sampled_unit_costs

print("apply_budget_constraint_merged_worst_first defined.")

In [ ]:
# ============================================================
# _simulate_years_worst_first: year-loop simulation for the worst-first policy.
# Renamed to avoid conflicts with the benefit-cost versions.
# ============================================================

def _simulate_years_worst_first(
    gdf,
    gdf_merged,
    years=10,
    current_budget=440_000_000,
    rng=None,
    attach_year_columns=True,
    return_year_costs=True,
    mapping_df=None,
    base_merged_pos=None,
    valid_map=None,
    compute_functional_each_year=False
):
    """
    Simulate pavement network using the Worst-first prioritization policy.
    Calls apply_budget_constraint_merged_worst_first each year.
    """
    import numpy as np
    import pandas as pd

    if rng is None:
        rng = np.random.default_rng(123)

    # Sample global performance-model shocks ONCE per run
    z_iri_global = rng.standard_normal()
    z_rut_global = rng.standard_normal()
    z_crack_global = rng.standard_normal()

    # Sample unit costs ONCE and reuse across all years (correlated lognormal)
    fixed_unit_costs = sample_fixed_unit_costs_with_correlation(
        treatments_cost, rng, corr_cost_df=treatment_cost_corr_df if USE_COST_CORRELATION else None
    )

    current_budget_arr = np.full(years + 1, current_budget)

    # Mapping + cache
    if mapping_df is None:
        mapping_df = build_simulation_mapping(gdf, gdf_merged)

    if base_merged_pos is None or valid_map is None:
        _, base_merged_pos, valid_map = prepare_mapping_cache(gdf, gdf_merged, mapping_df)

    ac_mask_merged = (gdf_merged["PAVEMENT_T"] == "AC")

    if "lane_miles_cal" not in gdf.columns:
        gdf["lane_miles_cal"] = (gdf["ToMeasure"] - gdf["FromMeasur"]) * gdf["Number_of_"]

    # Initialize base-level state
    gdf["New_Rutting"] = gdf["Rutting"].to_numpy()
    gdf["New_HPMS_Cracking"] = gdf["HPMS_Crack"].to_numpy()
    gdf["New_AvgIRI"] = gdf["AvgIRI"].to_numpy()

    # Initialize merged-level state
    gdf_merged["Foundation_Issue"] = 0

    agg_cols = ["New_AvgIRI", "New_HPMS_Cracking", "New_Rutting", "AADT"]

    pre_reset_iri = {}
    pre_reset_crack = {}
    pre_reset_rutting = {}

    unit_cost_realizations = {}
    year_costs = np.zeros(years + 1, dtype=np.float64)

    # YEAR 0 aggregation (FAST)
    gdf_merged = aggregate_weighted_to_merged_fast(
        gdf, gdf_merged, cols_to_agg=agg_cols,
        base_merged_pos=base_merged_pos, valid_map=valid_map,
        weight_col="lane_miles_cal", compute_functional=compute_functional_each_year,
        compute_ratings=True
    )

    # YEAR 0
    tcol0 = "Treatment_Year_0"
    gdf_merged[tcol0] = "DO_NOTHING"
    gdf[tcol0] = "DO_NOTHING"

    if ac_mask_merged.any():
        dt_out = fast_decision_tree_safe(
            gdf_merged.loc[ac_mask_merged],
            normalize_continue_tracking=False,
            fail_on_undecided=True,
            warn_on_nans=False,
            return_audit=False,
        )
        gdf_merged.loc[ac_mask_merged, tcol0] = dt_out.map(
            lambda t: replace_or_treatment(t, Treatment_Table)
        )

    gdf = propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, tcol0, mapping_df)

    pre_reset_iri[0] = gdf["New_AvgIRI"].to_numpy().copy()
    pre_reset_crack[0] = gdf["New_HPMS_Cracking"].to_numpy().copy()
    pre_reset_rutting[0] = gdf["New_Rutting"].to_numpy().copy()

    gdf, gdf_merged, sampled_unit_costs_y, paid_cost_total = apply_budget_constraint_merged_worst_first(
        gdf, gdf_merged, mapping_df, 0, current_budget_arr[0], treatments_cost,
        rng=rng, fixed_unit_costs=fixed_unit_costs, keep_cost_columns=False,
        return_paid_cost_total=True
    )
    unit_cost_realizations[0] = sampled_unit_costs_y
    year_costs[0] = float(paid_cost_total)

    gdf = apply_treatment_resets(gdf, 0)
    update_rehabilitation(gdf, tcol0)
    update_schd_year(gdf, tcol0)

    rutting_array = np.empty((len(gdf), years + 1), dtype=np.float32)
    crack_array   = np.empty((len(gdf), years + 1), dtype=np.float32)
    avg_iri_array = np.empty((len(gdf), years + 1), dtype=np.float32)

    rutting_array[:, 0] = gdf["New_Rutting"].to_numpy(dtype=np.float32)
    crack_array[:, 0]   = gdf["New_HPMS_Cracking"].to_numpy(dtype=np.float32)
    avg_iri_array[:, 0] = gdf["New_AvgIRI"].to_numpy(dtype=np.float32)

    # YEARS 1..N
    for year in range(1, years + 1):
        treatment_col = f"Treatment_Year_{year}"

        gdf = apply_treatment_effects(
            gdf, year, rng=rng,
            z_iri_global=z_iri_global, z_rut_global=z_rut_global, z_crack_global=z_crack_global
        )

        gdf_merged = aggregate_weighted_to_merged_fast(
            gdf, gdf_merged, cols_to_agg=agg_cols,
            base_merged_pos=base_merged_pos, valid_map=valid_map,
            weight_col="lane_miles_cal", compute_functional=compute_functional_each_year,
            compute_ratings=True
        )

        gdf[treatment_col] = "DO_NOTHING"
        gdf_merged[treatment_col] = "DO_NOTHING"

        if ac_mask_merged.any():
            dt_out = fast_decision_tree_safe(
                gdf_merged.loc[ac_mask_merged],
                normalize_continue_tracking=False,
                fail_on_undecided=True,
                warn_on_nans=False,
                return_audit=False,
            )
            gdf_merged.loc[ac_mask_merged, treatment_col] = dt_out.map(
                lambda t: replace_or_treatment(t, Treatment_Table)
            )

        gdf = propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, treatment_col, mapping_df)

        pre_reset_iri[year]     = gdf["New_AvgIRI"].to_numpy().copy()
        pre_reset_crack[year]   = gdf["New_HPMS_Cracking"].to_numpy().copy()
        pre_reset_rutting[year] = gdf["New_Rutting"].to_numpy().copy()

        gdf, gdf_merged, sampled_unit_costs_y, paid_cost_total = apply_budget_constraint_merged_worst_first(
            gdf, gdf_merged, mapping_df, year, current_budget_arr[year], treatments_cost,
            rng=rng, fixed_unit_costs=fixed_unit_costs, keep_cost_columns=False,
            return_paid_cost_total=True
        )
        unit_cost_realizations[year] = sampled_unit_costs_y
        year_costs[year] = float(paid_cost_total)

        gdf = apply_treatment_resets(gdf, year)
        update_rehabilitation(gdf, treatment_col)
        update_schd_year(gdf, treatment_col)

        rutting_array[:, year] = gdf["New_Rutting"].to_numpy(dtype=np.float32)
        crack_array[:, year]   = gdf["New_HPMS_Cracking"].to_numpy(dtype=np.float32)
        avg_iri_array[:, year] = gdf["New_AvgIRI"].to_numpy(dtype=np.float32)

    if attach_year_columns:
        rut_cols = [f"Rutting_Year_{i}" for i in range(years + 1)]
        crk_cols = [f"Crack_Year_{i}" for i in range(years + 1)]
        iri_cols = [f"AvgIRI_Year_{i}" for i in range(years + 1)]
        gdf[rut_cols] = pd.DataFrame(rutting_array, index=gdf.index, columns=rut_cols)
        gdf[crk_cols] = pd.DataFrame(crack_array,   index=gdf.index, columns=crk_cols)
        gdf[iri_cols] = pd.DataFrame(avg_iri_array,  index=gdf.index, columns=iri_cols)

    if return_year_costs:
        return (gdf, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
                avg_iri_array, crack_array, rutting_array, unit_cost_realizations, year_costs)
    else:
        return (gdf, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
                avg_iri_array, crack_array, rutting_array, unit_cost_realizations)

print("_simulate_years_worst_first defined.")

## Option 2 — Preservation Prioritization

Preservation estimates the avoided deterioration produced by each treatment over the remaining analysis horizon and ranks eligible projects by benefit divided by agency cost.


In [ ]:
# ============================================================
# AUC helper functions and apply_benefit_cost for the benefit-cost policy.
# ============================================================

def apply_benefit_cost(
    gdf,
    year,
    pre_reset_iri,
    pre_reset_crack,
    pre_reset_rutting,
    treatments,
    budget,
    mapping,
    analysis_end_year=YEARS,
    rng=None,
    fixed_unit_costs=None,
    keep_cost_columns=False,
    return_year_cost_info=False,
):
    """
    Apply benefit-cost analysis using Area Under Curve (AUC) as the benefit metric.
    Ranks merged segments by benefit/cost ratio and applies budget constraint.
    """

    if rng is None:
        rng = np.random.default_rng(0)

    treatment_col = f'Treatment_Year_{year}'
    remaining_years = analysis_end_year - year

    if fixed_unit_costs is None:
        fixed_unit_costs = {}

    if remaining_years <= 0:
        gdf.loc[:, treatment_col] = "DO_NOTHING"
        gdf["Applied_Treatment_Cost"] = 0.0
        if return_year_cost_info:
            return gdf, {"sampled_unit_costs": fixed_unit_costs, "paid_cost": 0.0}
        return gdf

    # ---------------------------------------------------------
    # Vectorized AUC functions
    # ---------------------------------------------------------
    def compute_auc_vectorized_iri(initial_iri, ages, esal_values, a_coeffs, b_coeffs, c_coeffs, svf_values, years):
        """Calculate vectorized IRI area-under-curve treatment benefit."""
        n_rows = len(initial_iri)
        ESAL_DEF = 20e6
        all_projections = np.zeros((n_rows, years + 1))
        all_projections[:, 0] = initial_iri
        for t in range(1, years + 1):
            growth_factor = (
                a_coeffs * (ages + t - 1)
                + b_coeffs * esal_values / ESAL_DEF
                + c_coeffs * svf_values
            )
            all_projections[:, t] = all_projections[:, t - 1] + growth_factor
        return np.trapezoid(all_projections, dx=1, axis=1)

    def compute_auc_vectorized_crack(initial_crack, ages, esal_values, a_coeffs, b_coeffs, c_coeffs, d_coeffs, svf_values, years):
        """Calculate vectorized cracking area-under-curve treatment benefit."""
        n_rows = len(initial_crack)
        ESAL_DEF = 20e6
        all_projections = np.zeros((n_rows, years + 1))
        all_projections[:, 0] = initial_crack
        for t in range(1, years + 1):
            growth_factor = np.maximum(
                1.002,
                1 + a_coeffs + b_coeffs * esal_values / ESAL_DEF - c_coeffs * (ages + t - 1)
            )
            all_projections[:, t] = all_projections[:, t - 1] * growth_factor + d_coeffs * svf_values
            all_projections[:, t] = np.minimum(all_projections[:, t], 100)
        return np.trapezoid(all_projections, dx=1, axis=1)

    def compute_auc_vectorized_rutting(initial_rutting, ages, esal_values, a_coeffs, b_coeffs, c_coeffs, svf_values, years):
        """Calculate vectorized rutting area-under-curve treatment benefit."""
        n_rows = len(initial_rutting)
        ESAL_DEF = 20e6
        all_projections = np.zeros((n_rows, years + 1))
        all_projections[:, 0] = initial_rutting
        for t in range(1, years + 1):
            growth_factor = (
                a_coeffs * (ages + t - 1)
                + b_coeffs * esal_values / ESAL_DEF
                + c_coeffs * svf_values
            )
            all_projections[:, t] = all_projections[:, t - 1] + growth_factor
        return np.trapezoid(all_projections, dx=1, axis=1)

    # Current state before treatment reset
    current_iri     = pre_reset_iri[year]
    current_crack   = pre_reset_crack[year]
    current_rutting = pre_reset_rutting[year]

    ages            = gdf['pavement_age'].values
    esal_values     = gdf['ESAL_20_year'].values
    pavement_families = gdf['Pavement_Family'].values
    svf_values      = gdf['svf'].values

    a_iri_coeffs, b_iri_coeffs, c_iri_coeffs = get_coefficients_iri(pavement_families)
    a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs = get_coefficients_crack(pavement_families)
    a_rutt_coeffs, b_rutt_coeffs, c_rutt_coeffs = get_coefficients_rutting(pavement_families)

    # Do-nothing AUC
    do_nothing_auc_iri = compute_auc_vectorized_iri(
        current_iri, ages, esal_values, a_iri_coeffs, b_iri_coeffs, c_iri_coeffs, svf_values, remaining_years)
    do_nothing_auc_crack = compute_auc_vectorized_crack(
        current_crack, ages, esal_values, a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs, svf_values, remaining_years)
    do_nothing_auc_rutting = compute_auc_vectorized_rutting(
        current_rutting, ages, esal_values, a_rutt_coeffs, b_rutt_coeffs, c_rutt_coeffs, svf_values, remaining_years)

    # Apply-treatment AUC
    temp_gdf = gdf.copy()
    temp_gdf = apply_treatment_resets(temp_gdf, year)

    after_treatment_iri     = temp_gdf['New_AvgIRI'].values
    after_treatment_crack   = temp_gdf['New_HPMS_Cracking'].values
    after_treatment_rutting = temp_gdf['New_Rutting'].values
    after_treatment_ages    = temp_gdf['pavement_age'].values

    apply_treatment_auc_iri = compute_auc_vectorized_iri(
        after_treatment_iri, after_treatment_ages, esal_values, a_iri_coeffs, b_iri_coeffs, c_iri_coeffs, svf_values, remaining_years)
    apply_treatment_auc_crack = compute_auc_vectorized_crack(
        after_treatment_crack, after_treatment_ages, esal_values, a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs, svf_values, remaining_years)
    apply_treatment_auc_rutting = compute_auc_vectorized_rutting(
        after_treatment_rutting, after_treatment_ages, esal_values, a_rutt_coeffs, b_rutt_coeffs, c_rutt_coeffs, svf_values, remaining_years)

    # Benefit calculation
    benefit_iri     = do_nothing_auc_iri     - apply_treatment_auc_iri
    benefit_crack   = do_nothing_auc_crack   - apply_treatment_auc_crack
    benefit_rutting = do_nothing_auc_rutting - apply_treatment_auc_rutting

    scaled_benefit_iri     = benefit_iri     / 8.0
    scaled_benefit_crack   = benefit_crack   / 4.0
    scaled_benefit_rutting = benefit_rutting * 25.0

    weights = {'crack': 0.4, 'iri': 0.25, 'rutting': 0.1}
    benefit_intensity = (
        scaled_benefit_crack   * weights['crack']
        + scaled_benefit_iri   * weights['iri']
        + scaled_benefit_rutting * weights['rutting']
    )

    # Cost calculation using FIXED sampled costs
    lane_miles_cal = ((gdf['ToMeasure'] - gdf['FromMeasur']) * gdf['Number_of_']).astype(float).values

    uniq = pd.unique(gdf[treatment_col].values)
    for t in uniq:
        if t not in fixed_unit_costs:
            info = treatments.get(t, {})
            fixed_unit_costs[t] = float(info.get("Cost_lane_mi", 0.0) or 0.0)

    for t in ['DO_NOTHING', 'CONTINUE_TRACKING', 'REPAIR', 'DIAMOND_GRIND']:
        fixed_unit_costs[t] = 0.0

    treatment_unit_costs = gdf[treatment_col].map(fixed_unit_costs).fillna(0.0).astype(float).values
    total_costs   = treatment_unit_costs * lane_miles_cal
    total_benefit = benefit_intensity * lane_miles_cal

    if keep_cost_columns:
        gdf[f"UnitCost_Year_{year}"]  = treatment_unit_costs
        gdf[f"TotalCost_Year_{year}"] = total_costs
        gdf[f"Benefit_Year_{year}"]   = total_benefit

    # Aggregate to merged segments and rank by B/C
    analysis_df = pd.DataFrame({
        'base_idx': gdf.index,
        'benefit':  total_benefit,
        'cost':     total_costs,
        'treatment': gdf[treatment_col].values
    })

    merged_analysis = mapping.merge(analysis_df, on='base_idx', how='inner')
    grouped = merged_analysis.groupby('merged_idx').agg({
        'benefit':   'sum',
        'cost':      'sum',
        'treatment': 'first'
    }).reset_index()

    valid_mask  = ~grouped['treatment'].isin(['DO_NOTHING', 'CONTINUE_TRACKING', 'REPAIR', 'DIAMOND_GRIND'])
    valid_groups = grouped.loc[valid_mask].copy()

    if valid_groups.empty:
        gdf["Applied_Treatment_Cost"] = 0.0
        if return_year_cost_info:
            return gdf, {"sampled_unit_costs": fixed_unit_costs, "paid_cost": 0.0}
        return gdf

    valid_groups['bc_ratio'] = np.where(
        valid_groups['cost'] > 0,
        valid_groups['benefit'] / valid_groups['cost'],
        0.0
    )

    valid_groups = valid_groups.sort_values(by='bc_ratio', ascending=False)

    # Skip-and-continue greedy scan: fund each group in BCR order if it
    # fits in the remaining budget; skip if not but keep checking the rest.
    remaining = float(budget)
    funded_mask = np.zeros(len(valid_groups), dtype=bool)
    for _i, _cost in enumerate(valid_groups['cost'].values):
        if _cost <= 0.0 or _cost <= remaining:
            funded_mask[_i] = True
            remaining -= max(float(_cost), 0.0)

    rejected_groups = valid_groups[~funded_mask]

    if not rejected_groups.empty:
        rejected_ids = rejected_groups['merged_idx'].values
        base_indices_to_reject = mapping.loc[mapping['merged_idx'].isin(rejected_ids), 'base_idx'].values
        gdf.loc[base_indices_to_reject, treatment_col] = 'DO_NOTHING'

    final_unit_costs  = gdf[treatment_col].map(fixed_unit_costs).fillna(0.0).astype(float).values
    final_total_costs = final_unit_costs * lane_miles_cal
    paid_cost = float(final_total_costs.sum())

    gdf["Applied_Treatment_Cost"] = final_total_costs

    if keep_cost_columns:
        gdf[f"Final_UnitCost_Year_{year}"]  = final_unit_costs
        gdf[f"Final_TotalCost_Year_{year}"] = final_total_costs

    if return_year_cost_info:
        return gdf, {"sampled_unit_costs": fixed_unit_costs, "paid_cost": paid_cost}

    return gdf

print("apply_benefit_cost defined.")

In [ ]:
# ============================================================
# _simulate_years_benefit_cost: year-loop for Preservation policy.
# Renamed from costbenefit apply_decision_tree_for_years.
# ============================================================

def _simulate_years_benefit_cost(
    gdf,
    gdf_merged,
    years=10,
    current_budget=440_000_000,
    rng=None,
):
    """
    Simulate using Preservation prioritization.
    Calls apply_benefit_cost each year instead of worst-first ranking.
    """

    if rng is None:
        rng = np.random.default_rng(0)

    current_budget_arr = np.full(years + 1, current_budget, dtype=float)

    z_iri_global   = rng.standard_normal()
    z_rut_global   = rng.standard_normal()
    z_crack_global = rng.standard_normal()

    gdf       = gdf.copy()
    gdf_merged = gdf_merged.copy()

    mapping_df = build_simulation_mapping(gdf, gdf_merged)

    ac_mask_merged = gdf_merged["PAVEMENT_T"] == "AC"

    gdf["lane_miles_cal"] = (gdf["ToMeasure"] - gdf["FromMeasur"]) * gdf["Number_of_"]

    gdf["New_Rutting"]      = gdf["Rutting"]
    gdf["New_HPMS_Cracking"] = gdf["HPMS_Crack"]
    gdf["New_AvgIRI"]       = gdf["AvgIRI"]

    pre_reset_iri     = {}
    pre_reset_crack   = {}
    pre_reset_rutting = {}

    gdf_merged["Foundation_Issue"] = 0

    agg_cols = ["New_AvgIRI", "New_HPMS_Cracking", "New_Rutting", "AADT"]

    gdf_merged = aggregate_weighted_to_merged(gdf, gdf_merged, agg_cols, mapping_df, weight_col="lane_miles_cal")

    # Sample fixed treatment costs ONCE per simulation run (correlated lognormal)
    fixed_unit_costs = sample_fixed_unit_costs_with_correlation(
        treatments, rng, corr_cost_df=treatment_cost_corr_df if USE_COST_CORRELATION else None
    )

    # Year 0
    gdf_merged["Treatment_Year_0"] = "DO_NOTHING"
    gdf["Treatment_Year_0"]        = "DO_NOTHING"

    if ac_mask_merged.any():
        decision_output_0 = fast_decision_tree_safe(
            gdf_merged.loc[ac_mask_merged].copy(), return_audit=True)
        if isinstance(decision_output_0, pd.DataFrame):
            treatment_series_0 = decision_output_0["Treatment"]
        else:
            treatment_series_0 = pd.Series(decision_output_0, index=gdf_merged.loc[ac_mask_merged].index)

        gdf_merged.loc[ac_mask_merged, "Treatment_Year_0"] = (
            treatment_series_0
            .apply(lambda t: replace_or_treatment(t, Treatment_Table))
            .values
        )

    gdf = propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, "Treatment_Year_0", mapping_df)

    pre_reset_iri[0]     = gdf["New_AvgIRI"].values.copy()
    pre_reset_crack[0]   = gdf["New_HPMS_Cracking"].values.copy()
    pre_reset_rutting[0] = gdf["New_Rutting"].values.copy()

    gdf = apply_benefit_cost(
        gdf, 0, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
        treatments, current_budget_arr[0], mapping_df,
        analysis_end_year=years, rng=rng, fixed_unit_costs=fixed_unit_costs,
    )

    gdf = apply_treatment_resets(gdf, 0)
    update_rehabilitation(gdf, "Treatment_Year_0")
    update_schd_year(gdf, "Treatment_Year_0")

    rutting_array = np.zeros((len(gdf), years + 1), dtype=np.float32)
    crack_array   = np.zeros((len(gdf), years + 1), dtype=np.float32)
    avg_iri_array = np.zeros((len(gdf), years + 1), dtype=np.float32)

    rutting_array[:, 0] = gdf["New_Rutting"].values
    crack_array[:, 0]   = gdf["New_HPMS_Cracking"].values
    avg_iri_array[:, 0] = gdf["New_AvgIRI"].values

    year_costs = [0.0]

    for year in range(1, years + 1):
        treatment_col = f"Treatment_Year_{year}"

        gdf = apply_treatment_effects(
            gdf, year, rng=rng,
            z_iri_global=z_iri_global, z_rut_global=z_rut_global, z_crack_global=z_crack_global,
        )

        gdf_merged = aggregate_weighted_to_merged(gdf, gdf_merged, agg_cols, mapping_df, weight_col="lane_miles_cal")

        gdf[treatment_col]        = "DO_NOTHING"
        gdf_merged[treatment_col] = "DO_NOTHING"

        if ac_mask_merged.any():
            decision_output_y = fast_decision_tree_safe(
                gdf_merged.loc[ac_mask_merged].copy(), return_audit=True)
            if isinstance(decision_output_y, pd.DataFrame):
                treatment_series_y = decision_output_y["Treatment"]
            else:
                treatment_series_y = pd.Series(decision_output_y, index=gdf_merged.loc[ac_mask_merged].index)

            gdf_merged.loc[ac_mask_merged, treatment_col] = (
                treatment_series_y
                .apply(lambda t: replace_or_treatment(t, Treatment_Table))
                .values
            )

        gdf = propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, treatment_col, mapping_df)

        pre_reset_iri[year]     = gdf["New_AvgIRI"].values.copy()
        pre_reset_crack[year]   = gdf["New_HPMS_Cracking"].values.copy()
        pre_reset_rutting[year] = gdf["New_Rutting"].values.copy()

        gdf = apply_benefit_cost(
            gdf, year, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
            treatments, current_budget_arr[year], mapping_df,
            analysis_end_year=years, rng=rng, fixed_unit_costs=fixed_unit_costs,
        )

        if "Applied_Treatment_Cost" in gdf.columns:
            year_costs.append(float(gdf["Applied_Treatment_Cost"].fillna(0).sum()))
        else:
            year_costs.append(np.nan)

        gdf = apply_treatment_resets(gdf, year)
        update_rehabilitation(gdf, treatment_col)
        update_schd_year(gdf, treatment_col)

        rutting_array[:, year] = gdf["New_Rutting"].values
        crack_array[:, year]   = gdf["New_HPMS_Cracking"].values
        avg_iri_array[:, year] = gdf["New_AvgIRI"].values

    rutting_columns  = {f"Rutting_Year_{i}": rutting_array[:, i]  for i in range(years + 1)}
    crack_columns    = {f"Crack_Year_{i}":   crack_array[:, i]    for i in range(years + 1)}
    avg_iri_columns  = {f"AvgIRI_Year_{i}":  avg_iri_array[:, i]  for i in range(years + 1)}

    gdf = pd.concat([
        gdf,
        pd.DataFrame(rutting_columns,  index=gdf.index),
        pd.DataFrame(crack_columns,    index=gdf.index),
        pd.DataFrame(avg_iri_columns,  index=gdf.index),
    ], axis=1)

    return (gdf, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
            avg_iri_array, crack_array, rutting_array, fixed_unit_costs, year_costs)

print("_simulate_years_benefit_cost defined.")

## Option 3 — Preservation (Considered AADT) Prioritization

This option retains the Preservation strategy calculation and weights priority by squared AADT so that benefits on higher-traffic segments receive greater ranking weight.


In [ ]:
# ============================================================
# apply_benefit_cost_aadt: same as apply_benefit_cost but multiplies
# total_benefit by AADT² (clipped at 1).
# Key difference from Option 2: total_benefit *= AADT^2
# ============================================================

def apply_benefit_cost_aadt(
    gdf,
    year,
    pre_reset_iri,
    pre_reset_crack,
    pre_reset_rutting,
    treatments,
    budget,
    mapping,
    analysis_end_year=YEARS,
    rng=None,
    fixed_unit_costs=None,
    keep_cost_columns=False,
    return_year_cost_info=False,
):
    """
    Benefit-cost with AADT² traffic weighting.
    benefit_intensity is multiplied by lane_miles AND AADT²,
    so high-traffic corridors are prioritized over low-traffic ones.
    """

    if rng is None:
        rng = np.random.default_rng(0)

    treatment_col  = f'Treatment_Year_{year}'
    remaining_years = analysis_end_year - year

    if fixed_unit_costs is None:
        fixed_unit_costs = {}

    if remaining_years <= 0:
        gdf.loc[:, treatment_col] = "DO_NOTHING"
        gdf["Applied_Treatment_Cost"] = 0.0
        if return_year_cost_info:
            return gdf, {"sampled_unit_costs": fixed_unit_costs, "paid_cost": 0.0}
        return gdf

    # AUC helpers (identical to apply_benefit_cost)
    def compute_auc_vectorized_iri(initial_iri, ages, esal_values, a_coeffs, b_coeffs, c_coeffs, svf_values, years):
        """Calculate vectorized IRI area-under-curve treatment benefit."""
        n_rows = len(initial_iri)
        ESAL_DEF = 20e6
        all_projections = np.zeros((n_rows, years + 1))
        all_projections[:, 0] = initial_iri
        for t in range(1, years + 1):
            growth_factor = (a_coeffs * (ages + t - 1) + b_coeffs * esal_values / ESAL_DEF + c_coeffs * svf_values)
            all_projections[:, t] = all_projections[:, t - 1] + growth_factor
        return np.trapezoid(all_projections, dx=1, axis=1)

    def compute_auc_vectorized_crack(initial_crack, ages, esal_values, a_coeffs, b_coeffs, c_coeffs, d_coeffs, svf_values, years):
        """Calculate vectorized cracking area-under-curve treatment benefit."""
        n_rows = len(initial_crack)
        ESAL_DEF = 20e6
        all_projections = np.zeros((n_rows, years + 1))
        all_projections[:, 0] = initial_crack
        for t in range(1, years + 1):
            growth_factor = np.maximum(1.002, 1 + a_coeffs + b_coeffs * esal_values / ESAL_DEF - c_coeffs * (ages + t - 1))
            all_projections[:, t] = all_projections[:, t - 1] * growth_factor + d_coeffs * svf_values
            all_projections[:, t] = np.minimum(all_projections[:, t], 100)
        return np.trapezoid(all_projections, dx=1, axis=1)

    def compute_auc_vectorized_rutting(initial_rutting, ages, esal_values, a_coeffs, b_coeffs, c_coeffs, svf_values, years):
        """Calculate vectorized rutting area-under-curve treatment benefit."""
        n_rows = len(initial_rutting)
        ESAL_DEF = 20e6
        all_projections = np.zeros((n_rows, years + 1))
        all_projections[:, 0] = initial_rutting
        for t in range(1, years + 1):
            growth_factor = (a_coeffs * (ages + t - 1) + b_coeffs * esal_values / ESAL_DEF + c_coeffs * svf_values)
            all_projections[:, t] = all_projections[:, t - 1] + growth_factor
        return np.trapezoid(all_projections, dx=1, axis=1)

    current_iri     = pre_reset_iri[year]
    current_crack   = pre_reset_crack[year]
    current_rutting = pre_reset_rutting[year]

    ages              = gdf['pavement_age'].values
    esal_values       = gdf['ESAL_20_year'].values
    pavement_families = gdf['Pavement_Family'].values
    svf_values        = gdf['svf'].values
    aadt_values       = gdf['AADT'].values

    a_iri_coeffs,   b_iri_coeffs,   c_iri_coeffs             = get_coefficients_iri(pavement_families)
    a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs = get_coefficients_crack(pavement_families)
    a_rutt_coeffs,  b_rutt_coeffs,  c_rutt_coeffs             = get_coefficients_rutting(pavement_families)

    do_nothing_auc_iri = compute_auc_vectorized_iri(
        current_iri, ages, esal_values, a_iri_coeffs, b_iri_coeffs, c_iri_coeffs, svf_values, remaining_years)
    do_nothing_auc_crack = compute_auc_vectorized_crack(
        current_crack, ages, esal_values, a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs, svf_values, remaining_years)
    do_nothing_auc_rutting = compute_auc_vectorized_rutting(
        current_rutting, ages, esal_values, a_rutt_coeffs, b_rutt_coeffs, c_rutt_coeffs, svf_values, remaining_years)

    temp_gdf = gdf.copy()
    temp_gdf = apply_treatment_resets(temp_gdf, year)

    after_treatment_iri     = temp_gdf['New_AvgIRI'].values
    after_treatment_crack   = temp_gdf['New_HPMS_Cracking'].values
    after_treatment_rutting = temp_gdf['New_Rutting'].values
    after_treatment_ages    = temp_gdf['pavement_age'].values

    apply_treatment_auc_iri = compute_auc_vectorized_iri(
        after_treatment_iri, after_treatment_ages, esal_values, a_iri_coeffs, b_iri_coeffs, c_iri_coeffs, svf_values, remaining_years)
    apply_treatment_auc_crack = compute_auc_vectorized_crack(
        after_treatment_crack, after_treatment_ages, esal_values, a_crack_coeffs, b_crack_coeffs, c_crack_coeffs, d_crack_coeffs, svf_values, remaining_years)
    apply_treatment_auc_rutting = compute_auc_vectorized_rutting(
        after_treatment_rutting, after_treatment_ages, esal_values, a_rutt_coeffs, b_rutt_coeffs, c_rutt_coeffs, svf_values, remaining_years)

    benefit_iri     = do_nothing_auc_iri     - apply_treatment_auc_iri
    benefit_crack   = do_nothing_auc_crack   - apply_treatment_auc_crack
    benefit_rutting = do_nothing_auc_rutting - apply_treatment_auc_rutting

    scaled_benefit_iri     = benefit_iri     / 8.0
    scaled_benefit_crack   = benefit_crack   / 4.0
    scaled_benefit_rutting = benefit_rutting * 25.0

    weights = {'crack': 0.4, 'iri': 0.25, 'rutting': 0.1}
    benefit_intensity = (
        scaled_benefit_crack   * weights['crack']
        + scaled_benefit_iri   * weights['iri']
        + scaled_benefit_rutting * weights['rutting']
    )

    lane_miles_cal = ((gdf['ToMeasure'] - gdf['FromMeasur']) * gdf['Number_of_']).astype(float).values

    uniq = pd.unique(gdf[treatment_col].values)
    for t in uniq:
        if t not in fixed_unit_costs:
            info = treatments.get(t, {})
            fixed_unit_costs[t] = float(info.get("Cost_lane_mi", 0.0) or 0.0)
    for t in ['DO_NOTHING', 'CONTINUE_TRACKING', 'REPAIR', 'DIAMOND_GRIND']:
        fixed_unit_costs[t] = 0.0

    treatment_unit_costs = gdf[treatment_col].map(fixed_unit_costs).fillna(0.0).astype(float).values
    total_costs = treatment_unit_costs * lane_miles_cal

    # --- AADT² MODIFICATION (key difference from Option 2) ---
    aadt_factor = np.clip(aadt_values, a_min=1, a_max=None) ** 2
    total_benefit = benefit_intensity * lane_miles_cal * aadt_factor

    if keep_cost_columns:
        gdf[f"UnitCost_Year_{year}"]  = treatment_unit_costs
        gdf[f"TotalCost_Year_{year}"] = total_costs
        gdf[f"Benefit_Year_{year}"]   = total_benefit

    analysis_df = pd.DataFrame({
        'base_idx':  gdf.index,
        'benefit':   total_benefit,
        'cost':      total_costs,
        'treatment': gdf[treatment_col].values
    })

    merged_analysis = mapping.merge(analysis_df, on='base_idx', how='inner')
    grouped = merged_analysis.groupby('merged_idx').agg({'benefit': 'sum', 'cost': 'sum', 'treatment': 'first'}).reset_index()

    valid_mask   = ~grouped['treatment'].isin(['DO_NOTHING', 'CONTINUE_TRACKING', 'REPAIR', 'DIAMOND_GRIND'])
    valid_groups = grouped.loc[valid_mask].copy()

    if valid_groups.empty:
        gdf["Applied_Treatment_Cost"] = 0.0
        if return_year_cost_info:
            return gdf, {"sampled_unit_costs": fixed_unit_costs, "paid_cost": 0.0}
        return gdf

    valid_groups['bc_ratio'] = np.where(
        valid_groups['cost'] > 0,
        valid_groups['benefit'] / valid_groups['cost'],
        0.0
    )

    valid_groups = valid_groups.sort_values(by='bc_ratio', ascending=False)

    # Skip-and-continue greedy scan: fund each group in BCR order if it
    # fits in the remaining budget; skip if not but keep checking the rest.
    remaining = float(budget)
    funded_mask = np.zeros(len(valid_groups), dtype=bool)
    for _i, _cost in enumerate(valid_groups['cost'].values):
        if _cost <= 0.0 or _cost <= remaining:
            funded_mask[_i] = True
            remaining -= max(float(_cost), 0.0)

    rejected_groups = valid_groups[~funded_mask]
    if not rejected_groups.empty:
        rejected_ids = rejected_groups['merged_idx'].values
        base_indices_to_reject = mapping.loc[mapping['merged_idx'].isin(rejected_ids), 'base_idx'].values
        gdf.loc[base_indices_to_reject, treatment_col] = 'DO_NOTHING'

    final_unit_costs  = gdf[treatment_col].map(fixed_unit_costs).fillna(0.0).astype(float).values
    final_total_costs = final_unit_costs * lane_miles_cal
    paid_cost = float(final_total_costs.sum())

    gdf["Applied_Treatment_Cost"] = final_total_costs

    if return_year_cost_info:
        return gdf, {"sampled_unit_costs": fixed_unit_costs, "paid_cost": paid_cost}

    return gdf

print("apply_benefit_cost_aadt defined.")

In [ ]:
# ============================================================
# _simulate_years_benefit_cost_aadt: year-loop for Preservation (Considered AADT) policy.
# Identical structure to _simulate_years_benefit_cost except calls
# apply_benefit_cost_aadt instead of apply_benefit_cost.
# ============================================================

def _simulate_years_benefit_cost_aadt(
    gdf,
    gdf_merged,
    years=10,
    current_budget=440_000_000,
    rng=None,
):
    """
    Simulate using Preservation (Considered AADT) prioritization.
    """

    if rng is None:
        rng = np.random.default_rng(0)

    current_budget_arr = np.full(years + 1, current_budget, dtype=float)

    z_iri_global   = rng.standard_normal()
    z_rut_global   = rng.standard_normal()
    z_crack_global = rng.standard_normal()

    gdf       = gdf.copy()
    gdf_merged = gdf_merged.copy()

    mapping_df = build_simulation_mapping(gdf, gdf_merged)

    ac_mask_merged = gdf_merged["PAVEMENT_T"] == "AC"

    gdf["lane_miles_cal"] = (gdf["ToMeasure"] - gdf["FromMeasur"]) * gdf["Number_of_"]

    gdf["New_Rutting"]       = gdf["Rutting"]
    gdf["New_HPMS_Cracking"] = gdf["HPMS_Crack"]
    gdf["New_AvgIRI"]        = gdf["AvgIRI"]

    pre_reset_iri     = {}
    pre_reset_crack   = {}
    pre_reset_rutting = {}

    gdf_merged["Foundation_Issue"] = 0

    agg_cols = ["New_AvgIRI", "New_HPMS_Cracking", "New_Rutting", "AADT"]

    gdf_merged = aggregate_weighted_to_merged(gdf, gdf_merged, agg_cols, mapping_df, weight_col="lane_miles_cal")

    # Sample fixed treatment costs ONCE per simulation run (correlated lognormal)
    fixed_unit_costs = sample_fixed_unit_costs_with_correlation(
        treatments, rng, corr_cost_df=treatment_cost_corr_df if USE_COST_CORRELATION else None
    )

    gdf_merged["Treatment_Year_0"] = "DO_NOTHING"
    gdf["Treatment_Year_0"]        = "DO_NOTHING"

    if ac_mask_merged.any():
        decision_output_0 = fast_decision_tree_safe(
            gdf_merged.loc[ac_mask_merged].copy(), return_audit=True)
        if isinstance(decision_output_0, pd.DataFrame):
            treatment_series_0 = decision_output_0["Treatment"]
        else:
            treatment_series_0 = pd.Series(decision_output_0, index=gdf_merged.loc[ac_mask_merged].index)

        gdf_merged.loc[ac_mask_merged, "Treatment_Year_0"] = (
            treatment_series_0.apply(lambda t: replace_or_treatment(t, Treatment_Table)).values
        )

    gdf = propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, "Treatment_Year_0", mapping_df)

    pre_reset_iri[0]     = gdf["New_AvgIRI"].values.copy()
    pre_reset_crack[0]   = gdf["New_HPMS_Cracking"].values.copy()
    pre_reset_rutting[0] = gdf["New_Rutting"].values.copy()

    gdf = apply_benefit_cost_aadt(
        gdf, 0, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
        treatments, current_budget_arr[0], mapping_df,
        analysis_end_year=years, rng=rng, fixed_unit_costs=fixed_unit_costs,
    )

    gdf = apply_treatment_resets(gdf, 0)
    update_rehabilitation(gdf, "Treatment_Year_0")
    update_schd_year(gdf, "Treatment_Year_0")

    rutting_array = np.zeros((len(gdf), years + 1), dtype=np.float32)
    crack_array   = np.zeros((len(gdf), years + 1), dtype=np.float32)
    avg_iri_array = np.zeros((len(gdf), years + 1), dtype=np.float32)

    rutting_array[:, 0] = gdf["New_Rutting"].values
    crack_array[:, 0]   = gdf["New_HPMS_Cracking"].values
    avg_iri_array[:, 0] = gdf["New_AvgIRI"].values

    year_costs = [0.0]

    for year in range(1, years + 1):
        treatment_col = f"Treatment_Year_{year}"

        gdf = apply_treatment_effects(
            gdf, year, rng=rng,
            z_iri_global=z_iri_global, z_rut_global=z_rut_global, z_crack_global=z_crack_global,
        )

        gdf_merged = aggregate_weighted_to_merged(gdf, gdf_merged, agg_cols, mapping_df, weight_col="lane_miles_cal")

        gdf[treatment_col]        = "DO_NOTHING"
        gdf_merged[treatment_col] = "DO_NOTHING"

        if ac_mask_merged.any():
            decision_output_y = fast_decision_tree_safe(
                gdf_merged.loc[ac_mask_merged].copy(), return_audit=True)
            if isinstance(decision_output_y, pd.DataFrame):
                treatment_series_y = decision_output_y["Treatment"]
            else:
                treatment_series_y = pd.Series(decision_output_y, index=gdf_merged.loc[ac_mask_merged].index)

            gdf_merged.loc[ac_mask_merged, treatment_col] = (
                treatment_series_y.apply(lambda t: replace_or_treatment(t, Treatment_Table)).values
            )

        gdf = propagate_treatments_from_merged_to_gdf(gdf, gdf_merged, treatment_col, mapping_df)

        pre_reset_iri[year]     = gdf["New_AvgIRI"].values.copy()
        pre_reset_crack[year]   = gdf["New_HPMS_Cracking"].values.copy()
        pre_reset_rutting[year] = gdf["New_Rutting"].values.copy()

        gdf = apply_benefit_cost_aadt(
            gdf, year, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
            treatments, current_budget_arr[year], mapping_df,
            analysis_end_year=years, rng=rng, fixed_unit_costs=fixed_unit_costs,
        )

        if "Applied_Treatment_Cost" in gdf.columns:
            year_costs.append(float(gdf["Applied_Treatment_Cost"].fillna(0).sum()))
        else:
            year_costs.append(np.nan)

        gdf = apply_treatment_resets(gdf, year)
        update_rehabilitation(gdf, treatment_col)
        update_schd_year(gdf, treatment_col)

        rutting_array[:, year] = gdf["New_Rutting"].values
        crack_array[:, year]   = gdf["New_HPMS_Cracking"].values
        avg_iri_array[:, year] = gdf["New_AvgIRI"].values

    rutting_columns = {f"Rutting_Year_{i}": rutting_array[:, i] for i in range(years + 1)}
    crack_columns   = {f"Crack_Year_{i}":   crack_array[:, i]   for i in range(years + 1)}
    avg_iri_columns = {f"AvgIRI_Year_{i}":  avg_iri_array[:, i] for i in range(years + 1)}

    gdf = pd.concat([
        gdf,
        pd.DataFrame(rutting_columns,  index=gdf.index),
        pd.DataFrame(crack_columns,    index=gdf.index),
        pd.DataFrame(avg_iri_columns,  index=gdf.index),
    ], axis=1)

    return (gdf, pre_reset_iri, pre_reset_crack, pre_reset_rutting,
            avg_iri_array, crack_array, rutting_array, fixed_unit_costs, year_costs)

print("_simulate_years_benefit_cost_aadt defined.")

## Section 9: Shared Monte Carlo Infrastructure

Each Monte Carlo worker creates an independent network state and reproducible random-number stream. Results are stored by budget:

- `mc_results[budget]`: one row per iteration, including annual AvgIRI, annual user cost, and total costs.
- `avgiri_mc[budget]`: NumPy array with shape `(iterations, years)`.
- `cost_mc[budget]`: NumPy array with annual agency costs.
- `unit_cost_df`: optional sampled treatment unit-cost records.

`n_jobs=15` limits parallel execution to 15 worker processes to reduce CPU and memory pressure.


In [ ]:
# ============================================================
# Shared MC infrastructure helpers
# _get_lane_miles_series is a robust lane-mile column finder.
# ============================================================
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

def _get_lane_miles_series(gdf):
    """Robustly find or compute lane-miles column."""
    for c in ["Calculated_Lane_Miles", "lane_miles_cal", "Lane_Miles", "lane_miles"]:
        if c in gdf.columns:
            s = gdf[c]
            if hasattr(s, "ndim") and s.ndim == 2:
                s = s.iloc[:, -1]
            return s.astype(float).fillna(0.0)
    req = {"FromMeasur", "ToMeasure", "Number_of_"}
    if req.issubset(gdf.columns):
        return ((gdf["ToMeasure"] - gdf["FromMeasur"]) * gdf["Number_of_"]).astype(float).fillna(0.0)
    raise KeyError("No lane-miles column found.")

def _compute_user_cost_for_mc(gdf_res, years):
    """Mid-year IRI averaging then user cost. Used by all three MC workers."""
    gdf_uc = gdf_res.copy(deep=True)
    if "AvgIRI" not in gdf_uc.columns:
        raise KeyError("Expected column 'AvgIRI' in gdf_res for user cost calculation.")

    avg_iri_previous = gdf_uc["AvgIRI"].copy()
    for y in range(years):
        col = f"AvgIRI_Year_{y}"
        iri_after_col = f"AvgIRI_Year_{y+1}"
        if col not in gdf_uc.columns:
            raise KeyError(f"Expected column '{col}' in gdf_res for user cost calculation.")
        if y == 0:
            gdf_uc[col] = (avg_iri_previous + gdf_uc[col]) / 2.0
        else:
            if iri_after_col in gdf_uc.columns:
                gdf_uc[col] = (avg_iri_previous + gdf_uc[iri_after_col]) / 2.0
            else:
                gdf_uc[col] = (avg_iri_previous + gdf_uc[col]) / 2.0
        avg_iri_previous = gdf_uc[col]

    return user_cost_by_year_from_gdf(gdf_uc, years=years)

def _pack_mc_row(budget, k, avg_iri_array, crack_array, rutting_array,
                 year_costs, gdf_res, years, excluded):
    """
    Common output packing for all three MC worker functions.

    Computes lane-mile-weighted average IRI, rutting, cracking, and a composite
    condition index for every simulation year.

    Composite: weighted-scaled formula matching worst-first prioritization.
        composite = (IRI/COMPOSITE_IRI_SCALE)*COMPOSITE_W_IRI
                  + (Cracking/COMPOSITE_CRACK_SCALE)*COMPOSITE_W_CRACK
                  + (Rutting*COMPOSITE_RUT_FACTOR)*COMPOSITE_W_RUT
    Scale/weight constants are set in the USER CONFIGURATION cell.
    Higher composite values indicate worse network condition.

    Returns
    -------
    row : dict
        Per-year columns for IRI, rutting, cracking, composite, cost, and user cost.
    avgiri_vec, cost_vec, avgrutting_vec, avgcracking_vec, avgcomposite_vec : ndarray
        Shape (years,) lane-mile-weighted annual averages.
    """
    w = _get_lane_miles_series(gdf_res).to_numpy(dtype=float)
    wsum = float(np.sum(w))

    if wsum <= 0:
        avgiri_vec       = np.full((years,), np.nan, dtype=float)
        avgrutting_vec   = np.full((years,), np.nan, dtype=float)
        avgcracking_vec  = np.full((years,), np.nan, dtype=float)
        avgcomposite_vec = np.full((years,), np.nan, dtype=float)
    else:
        iri_slice   = avg_iri_array[:, 0:years].astype(float)
        rut_slice   = rutting_array[:, 0:years].astype(float)
        crack_slice = crack_array[:, 0:years].astype(float)

        avgiri_vec      = (iri_slice   * w[:, None]).sum(axis=0) / wsum
        avgrutting_vec  = (rut_slice   * w[:, None]).sum(axis=0) / wsum
        avgcracking_vec = (crack_slice * w[:, None]).sum(axis=0) / wsum

        # Weighted-scaled composite matching worst-first prioritization formula
        scaled_iri   = iri_slice   / COMPOSITE_IRI_SCALE
        scaled_crack = crack_slice / COMPOSITE_CRACK_SCALE
        scaled_rut   = rut_slice   * COMPOSITE_RUT_FACTOR
        composite_slice = (
            scaled_iri   * COMPOSITE_W_IRI
            + scaled_crack * COMPOSITE_W_CRACK
            + scaled_rut   * COMPOSITE_W_RUT
        )
        avgcomposite_vec = (composite_slice * w[:, None]).sum(axis=0) / wsum

    if len(year_costs) >= years + 1:
        cost_vec = np.asarray(year_costs[0:years], dtype=float)
    elif len(year_costs) == years:
        cost_vec = np.asarray(year_costs, dtype=float)
    else:
        cost_vec = np.full(years, np.nan, dtype=float)
        cost_vec[:len(year_costs)] = np.asarray(year_costs, dtype=float)

    total_cost = float(np.nansum(cost_vec))

    lm_series = pd.Series(w).fillna(0.0)
    treated_seg = np.zeros(years, dtype=int)
    treated_lm  = np.zeros(years, dtype=float)
    for y in range(years):
        tcol = f"Treatment_Year_{y}"
        if tcol not in gdf_res.columns:
            continue
        t = gdf_res[tcol]
        if hasattr(t, "ndim") and t.ndim == 2:
            t = t.iloc[:, -1]
        mask = ~t.isin(excluded)
        treated_seg[y] = int(mask.sum())
        treated_lm[y]  = float(lm_series.loc[mask.to_numpy()].sum())

    user_cost_yearly = np.asarray(_compute_user_cost_for_mc(gdf_res, years), dtype=float)
    user_cost_total  = float(np.nansum(user_cost_yearly))

    row = {
        "budget": int(budget),
        "iter":   int(k),
        "total_cost":      float(total_cost),
        "user_cost_total": float(user_cost_total),
    }
    for y in range(years):
        row[f"avgiri_y{y}"]        = float(avgiri_vec[y])
        row[f"avgrutting_y{y}"]    = float(avgrutting_vec[y])
        row[f"avgcracking_y{y}"]   = float(avgcracking_vec[y])
        row[f"avgcomposite_y{y}"]  = float(avgcomposite_vec[y])
        row[f"cost_y{y}"]          = float(cost_vec[y])
        row[f"treated_seg_y{y}"]   = int(treated_seg[y])
        row[f"treated_lm_y{y}"]    = float(treated_lm[y])
        row[f"user_cost_y{y+1}"]   = float(user_cost_yearly[y])

    return row, avgiri_vec, cost_vec, avgrutting_vec, avgcracking_vec, avgcomposite_vec

print("MC shared infrastructure helpers defined.")

In [ ]:
# ============================================================
# Segment-level Parquet output helpers
# Called from each _mc_worker_* when SAVE_SEGMENT_RESULTS = 1
# ============================================================
from pathlib import Path as _Path

def _write_segments_parquet(gdf, base_dir, cov=None):
    """Write static segment metadata. Called once per enabled run (always overwrites)."""
    cov_tag = f"_cov{cov}" if cov is not None else ""
    out_path = _Path(base_dir) / f"segments{cov_tag}.parquet"
    _Path(base_dir).mkdir(parents=True, exist_ok=True)
    keep = [c for c in ("FID_N", "Functional", "lane_miles_cal") if c in gdf.columns]
    seg = gdf[keep].copy().rename(columns={"FID_N": "segment_id"})
    if "segment_id" in seg.columns:
        try:
            col = seg["segment_id"]
            if col.between(-2_147_483_648, 2_147_483_647).all():
                seg["segment_id"] = col.astype("int32")
        except Exception:
            pass
    seg.to_parquet(str(out_path), index=False, engine="pyarrow", compression="snappy")
    print(f"Segments metadata saved: {out_path}  ({len(seg):,} rows)")

def _save_sim_parquet(gdf_res, strategy_slug, budget, k, years, cfg):
    """Save per-simulation segment-level condition data. No-op when cfg is disabled."""
    if cfg is None or not cfg.get("enabled"):
        return

    base_dir    = cfg["base_dir"]
    compression = cfg.get("compression", "snappy")

    cov = cfg.get("cov")
    cov_tag = f"_cov{cov}" if cov is not None else ""
    out_dir = _Path(base_dir).resolve() / f"strategy={strategy_slug}" / f"budget={int(budget)}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"simulation={k:04d}{cov_tag}.parquet"

    iri_cols   = [f"AvgIRI_Year_{y}"  for y in range(years)]
    crack_cols = [f"Crack_Year_{y}"   for y in range(years)]
    rut_cols   = [f"Rutting_Year_{y}" for y in range(years)]
    all_cols   = iri_cols + crack_cols + rut_cols

    missing = [c for c in all_cols if c not in gdf_res.columns]
    if missing:
        raise RuntimeError(f"_save_sim_parquet: missing columns in gdf_res: {missing}")

    out = gdf_res[["FID_N"] + all_cols].copy()
    out = out.rename(columns={
        "FID_N": "segment_id",
        **{f"AvgIRI_Year_{y}":  f"iri_y{y}"      for y in range(years)},
        **{f"Crack_Year_{y}":   f"cracking_y{y}" for y in range(years)},
        **{f"Rutting_Year_{y}": f"rutting_y{y}"  for y in range(years)},
    })

    if "segment_id" in out.columns:
        try:
            col = out["segment_id"]
            if col.between(-2_147_483_648, 2_147_483_647).all():
                out["segment_id"] = col.astype("int32")
        except Exception:
            pass

    out["simulation"] = np.int32(k)

    val_cols = [c for c in out.columns if c not in ("segment_id", "simulation")]
    out[val_cols] = out[val_cols].astype("float32")

    out.to_parquet(str(out_path), index=False, engine="pyarrow", compression=compression)

print("Segment-level Parquet helpers defined.")

In [ ]:
# ============================================================
# MC worker and simulate function for Worst-first policy.
# ============================================================

STATE_COLS_GDF = [
    "Rutting_Hold_Year", "Rutting_Recovery_Year", "Rutting_Recovery_Year_Updated",
    "Delta_Rutting_Recovery", "Pre_Treatment_Rutting",
    "Crack_Hold_Year", "Crack_Recovery_Year", "Crack_Recovery_Year_Updated",
    "Delta_Crack_Recovery", "Pre_Treatment_Crack",
    "pavement_age",
    "New_AvgIRI", "New_HPMS_Cracking", "New_Rutting",
    "No_of_Rehab", "SCHD_Year", "PAVEMENT_T",
    "Foundation_Issue", "Functional",
]

EXCLUDED_TREATMENTS = {"DO_NOTHING", "CONTINUE_TRACKING", "REPAIR", "DIAMOND_GRIND"}

def _mc_worker_worst_first(
    bi, budget, k, years, seed,
    gdf_base0, gdfm_base0, state_cols_gdf,
    mapping_df0, base_merged_pos0, valid_map0,
    Treatment_Table, excluded, return_unit_costs,
    seg_save_cfg=None
):
    """Run one reproducible Worst-first Monte Carlo realization for one budget."""
    rng = np.random.default_rng(seed + 1000 * int(bi) + int(k))

    gdf_k  = gdf_base0.copy(deep=False)
    gdfm_k = gdfm_base0.copy(deep=False)

    cols_exist = [c for c in state_cols_gdf if c in gdf_base0.columns]
    for c in cols_exist:
        gdf_k[c] = gdf_base0[c].to_numpy().copy()

    gdf_k  = drop_sim_columns(gdf_k)
    gdfm_k = drop_sim_columns(gdfm_k)

    if "Functional" not in gdfm_k.columns:
        gdfm_k["Functional"] = gdfm_base0["Functional"]

    out = _simulate_years_worst_first(
        gdf_k, gdfm_k, years=years, current_budget=budget, rng=rng,
        attach_year_columns=True, return_year_costs=True,
        mapping_df=mapping_df0, base_merged_pos=base_merged_pos0, valid_map=valid_map0,
        compute_functional_each_year=False
    )

    (gdf_res, pre_iri, pre_crack, pre_rut, avg_iri_array, crack_array,
     rutting_array, unit_cost_realizations, year_costs) = out

    gdf_res = gdf_res.loc[:, ~gdf_res.columns.duplicated(keep="last")]

    row, avgiri_vec, cost_vec, avgrutting_vec, avgcracking_vec, avgcomposite_vec = _pack_mc_row(
        budget, k, avg_iri_array, crack_array, rutting_array,
        year_costs, gdf_res, years, excluded
    )

    _save_sim_parquet(gdf_res, "worst_first", budget, k, years, seg_save_cfg)

    unit_cost_rows_local = []
    if return_unit_costs and isinstance(unit_cost_realizations, dict):
        for y, d in unit_cost_realizations.items():
            if not isinstance(d, dict):
                continue
            for treat, price in d.items():
                unit_cost_rows_local.append({
                    "budget": int(budget), "iter": int(k), "year": int(y),
                    "treatment": str(treat), "unit_cost_lane_mi": float(price)
                })

    return (int(budget), int(k), row, avgiri_vec, cost_vec,
            avgrutting_vec, avgcracking_vec, avgcomposite_vec, unit_cost_rows_local)

def simulate_network_mc_worst_first(
    gdf, gdf_merged, budgets, Treatment_Table,
    years=10, n_mc=100, seed=123,
    drop_geometry=True, return_unit_costs=False,
    n_jobs=15, backend="loky", verbose=0,
    seg_save_cfg=None
):
    """Run Monte Carlo for the worst-first policy over one or more budgets."""
    if drop_geometry:
        gdf_base  = gdf.drop(columns=["geometry"], errors="ignore").copy(deep=True)
        gdfm_base = gdf_merged.drop(columns=["geometry"], errors="ignore").copy(deep=True)
    else:
        gdf_base  = gdf.copy(deep=True)
        gdfm_base = gdf_merged.copy(deep=True)

    if "Calculated_Lane_Miles" not in gdf_base.columns:
        gdf_base["Calculated_Lane_Miles"] = (
            (gdf_base["ToMeasure"] - gdf_base["FromMeasur"]) * gdf_base["Number_of_"]
        )

    gdf_base0  = drop_sim_columns(gdf_base)
    gdfm_base0 = drop_sim_columns(gdfm_base)

    # Generate initial PMS state ONCE with a fixed seed so every MC run starts
    # from the same network condition. Variation between runs comes only from
    # treatment cost sampling and performance model uncertainty (z_*_global).
    _init_rng = np.random.default_rng(seed)
    gdf_base0["pavement_age"] = _init_rng.integers(0, 21, size=len(gdf_base0))
    gdf_base0["No_of_Rehab"]  = _init_rng.integers(0, 4,  size=len(gdf_base0))
    gdf_base0["SCHD_Year"]    = _init_rng.integers(0, 10, size=len(gdf_base0))
    gdfm_base0["No_of_Rehab"] = _init_rng.integers(0, 4,  size=len(gdfm_base0))
    gdfm_base0["SCHD_Year"]   = _init_rng.integers(0, 10, size=len(gdfm_base0))

    mapping_df0 = build_simulation_mapping(gdf_base0, gdfm_base0)
    _, base_merged_pos0, valid_map0 = prepare_mapping_cache(gdf_base0, gdfm_base0, mapping_df0)
    _ensure_merged_functional_once(gdf_base0, gdfm_base0, mapping_df0)

    mc_results   = {}
    avgiri_mc    = {}
    cost_mc      = {}
    rutting_mc   = {}
    cracking_mc  = {}
    composite_mc = {}
    unit_cost_rows = []

    for bi, budget in enumerate(budgets):
        results = Parallel(n_jobs=n_jobs, backend=backend, verbose=verbose)(
            delayed(_mc_worker_worst_first)(
                bi=bi, budget=int(budget), k=k, years=years, seed=seed,
                gdf_base0=gdf_base0, gdfm_base0=gdfm_base0,
                state_cols_gdf=STATE_COLS_GDF,
                mapping_df0=mapping_df0, base_merged_pos0=base_merged_pos0, valid_map0=valid_map0,
                Treatment_Table=Treatment_Table, excluded=EXCLUDED_TREATMENTS,
                return_unit_costs=return_unit_costs,
                seg_save_cfg=seg_save_cfg
            )
            for k in range(n_mc)
        )

        rows          = []
        avg_mat       = np.zeros((n_mc, years), dtype=float)
        cost_mat      = np.zeros((n_mc, years), dtype=float)
        rutting_mat   = np.zeros((n_mc, years), dtype=float)
        cracking_mat  = np.zeros((n_mc, years), dtype=float)
        composite_mat = np.zeros((n_mc, years), dtype=float)

        for (_budget, k, row, avgiri_vec, cost_vec,
             avgrutting_vec, avgcracking_vec, avgcomposite_vec, uc_rows_local) in results:
            rows.append(row)
            avg_mat[k, :]       = avgiri_vec
            cost_mat[k, :]      = cost_vec
            rutting_mat[k, :]   = avgrutting_vec
            cracking_mat[k, :]  = avgcracking_vec
            composite_mat[k, :] = avgcomposite_vec
            if return_unit_costs:
                unit_cost_rows.extend(uc_rows_local)

        mc_results[int(budget)]   = pd.DataFrame(rows).sort_values(["iter"]).reset_index(drop=True)
        avgiri_mc[int(budget)]    = avg_mat
        cost_mc[int(budget)]      = cost_mat
        rutting_mc[int(budget)]   = rutting_mat
        cracking_mc[int(budget)]  = cracking_mat
        composite_mc[int(budget)] = composite_mat

    unit_cost_df = pd.DataFrame(unit_cost_rows) if return_unit_costs else None
    return mc_results, avgiri_mc, cost_mc, rutting_mc, cracking_mc, composite_mc, unit_cost_df

print("simulate_network_mc_worst_first defined.")

In [ ]:
# ============================================================
# MC worker and simulate function for Preservation policy.
# ============================================================

def _mc_worker_benefit_cost(bi, budget, k, years, seed,
                             gdf_base0, gdfm_base0, return_unit_costs,
                             seg_save_cfg=None):
    """Run one reproducible Preservation Monte Carlo realization for one budget."""
    rng = np.random.default_rng(seed + 1000 * int(bi) + int(k))

    gdf_k  = gdf_base0.copy(deep=True)
    gdfm_k = gdfm_base0.copy(deep=True)

    gdf_k  = drop_sim_columns(gdf_k)
    gdfm_k = drop_sim_columns(gdfm_k)

    out = _simulate_years_benefit_cost(
        gdf_k, gdfm_k, years=years, current_budget=budget, rng=rng,
    )

    (gdf_res, pre_iri, pre_crack, pre_rut, avg_iri_array, crack_array,
     rutting_array, fixed_unit_costs, year_costs) = out

    gdf_res = gdf_res.loc[:, ~gdf_res.columns.duplicated(keep="last")]

    row, avgiri_vec, cost_vec, avgrutting_vec, avgcracking_vec, avgcomposite_vec = _pack_mc_row(
        budget, k, avg_iri_array, crack_array, rutting_array,
        year_costs, gdf_res, years, EXCLUDED_TREATMENTS
    )

    _save_sim_parquet(gdf_res, "preservation", budget, k, years, seg_save_cfg)

    unit_cost_rows_local = []
    if return_unit_costs and isinstance(fixed_unit_costs, dict):
        for treat, price in fixed_unit_costs.items():
            unit_cost_rows_local.append({
                "budget": int(budget), "iter": int(k),
                "treatment": str(treat), "unit_cost_lane_mi": float(price)
            })

    return (int(budget), int(k), row, avgiri_vec, cost_vec,
            avgrutting_vec, avgcracking_vec, avgcomposite_vec, unit_cost_rows_local)

def simulate_network_mc_benefit_cost(
    gdf, gdf_merged, budgets, Treatment_Table,
    years=10, n_mc=100, seed=123,
    drop_geometry=True, return_unit_costs=False,
    n_jobs=15, backend="loky", verbose=0,
    seg_save_cfg=None
):
    """Run Monte Carlo for the Preservation policy."""
    if drop_geometry:
        gdf_base  = gdf.drop(columns=["geometry"], errors="ignore").copy(deep=True)
        gdfm_base = gdf_merged.drop(columns=["geometry"], errors="ignore").copy(deep=True)
    else:
        gdf_base  = gdf.copy(deep=True)
        gdfm_base = gdf_merged.copy(deep=True)

    if "lane_miles_cal" not in gdf_base.columns:
        gdf_base["lane_miles_cal"] = (
            (gdf_base["ToMeasure"] - gdf_base["FromMeasur"]) * gdf_base["Number_of_"]
        )

    gdf_base0  = drop_sim_columns(gdf_base)
    gdfm_base0 = drop_sim_columns(gdfm_base)

    # Generate initial PMS state ONCE with a fixed seed so every MC run starts
    # from the same network condition. Variation between runs comes only from
    # treatment cost sampling and performance model uncertainty (z_*_global).
    _init_rng = np.random.default_rng(seed)
    gdf_base0["pavement_age"] = _init_rng.integers(0, 21, size=len(gdf_base0))
    gdf_base0["No_of_Rehab"]  = _init_rng.integers(0, 4,  size=len(gdf_base0))
    gdf_base0["SCHD_Year"]    = _init_rng.integers(0, 10, size=len(gdf_base0))
    gdfm_base0["No_of_Rehab"] = _init_rng.integers(0, 4,  size=len(gdfm_base0))
    gdfm_base0["SCHD_Year"]   = _init_rng.integers(0, 10, size=len(gdfm_base0))

    mc_results   = {}
    avgiri_mc    = {}
    cost_mc      = {}
    rutting_mc   = {}
    cracking_mc  = {}
    composite_mc = {}
    unit_cost_rows = []

    for bi, budget in enumerate(budgets):
        results = Parallel(n_jobs=n_jobs, backend=backend, verbose=verbose)(
            delayed(_mc_worker_benefit_cost)(
                bi=bi, budget=int(budget), k=k, years=years, seed=seed,
                gdf_base0=gdf_base0, gdfm_base0=gdfm_base0,
                return_unit_costs=return_unit_costs,
                seg_save_cfg=seg_save_cfg
            )
            for k in range(n_mc)
        )

        rows          = []
        avg_mat       = np.zeros((n_mc, years), dtype=float)
        cost_mat      = np.zeros((n_mc, years), dtype=float)
        rutting_mat   = np.zeros((n_mc, years), dtype=float)
        cracking_mat  = np.zeros((n_mc, years), dtype=float)
        composite_mat = np.zeros((n_mc, years), dtype=float)

        for (_budget, k, row, avgiri_vec, cost_vec,
             avgrutting_vec, avgcracking_vec, avgcomposite_vec, uc_rows_local) in results:
            rows.append(row)
            avg_mat[k, :]       = avgiri_vec
            cost_mat[k, :]      = cost_vec
            rutting_mat[k, :]   = avgrutting_vec
            cracking_mat[k, :]  = avgcracking_vec
            composite_mat[k, :] = avgcomposite_vec
            if return_unit_costs:
                unit_cost_rows.extend(uc_rows_local)

        mc_results[int(budget)]   = pd.DataFrame(rows).sort_values(["iter"]).reset_index(drop=True)
        avgiri_mc[int(budget)]    = avg_mat
        cost_mc[int(budget)]      = cost_mat
        rutting_mc[int(budget)]   = rutting_mat
        cracking_mc[int(budget)]  = cracking_mat
        composite_mc[int(budget)] = composite_mat

    unit_cost_df = pd.DataFrame(unit_cost_rows) if return_unit_costs else None
    return mc_results, avgiri_mc, cost_mc, rutting_mc, cracking_mc, composite_mc, unit_cost_df

print("simulate_network_mc_benefit_cost defined.")

In [ ]:
# ============================================================
# MC worker and simulate function for Preservation (Considered AADT) policy.
# Same structure as benefit-cost but calls _simulate_years_benefit_cost_aadt.
# ============================================================

def _mc_worker_benefit_cost_aadt(bi, budget, k, years, seed,
                                  gdf_base0, gdfm_base0, return_unit_costs,
                                  seg_save_cfg=None):
    """Run one reproducible Preservation (Considered AADT) Monte Carlo realization."""
    rng = np.random.default_rng(seed + 1000 * int(bi) + int(k))

    gdf_k  = gdf_base0.copy(deep=True)
    gdfm_k = gdfm_base0.copy(deep=True)

    gdf_k  = drop_sim_columns(gdf_k)
    gdfm_k = drop_sim_columns(gdfm_k)

    out = _simulate_years_benefit_cost_aadt(
        gdf_k, gdfm_k, years=years, current_budget=budget, rng=rng,
    )

    (gdf_res, pre_iri, pre_crack, pre_rut, avg_iri_array, crack_array,
     rutting_array, fixed_unit_costs, year_costs) = out

    gdf_res = gdf_res.loc[:, ~gdf_res.columns.duplicated(keep="last")]

    row, avgiri_vec, cost_vec, avgrutting_vec, avgcracking_vec, avgcomposite_vec = _pack_mc_row(
        budget, k, avg_iri_array, crack_array, rutting_array,
        year_costs, gdf_res, years, EXCLUDED_TREATMENTS
    )

    _save_sim_parquet(gdf_res, "preservation_aadt", budget, k, years, seg_save_cfg)

    unit_cost_rows_local = []
    if return_unit_costs and isinstance(fixed_unit_costs, dict):
        for treat, price in fixed_unit_costs.items():
            unit_cost_rows_local.append({
                "budget": int(budget), "iter": int(k),
                "treatment": str(treat), "unit_cost_lane_mi": float(price)
            })

    return (int(budget), int(k), row, avgiri_vec, cost_vec,
            avgrutting_vec, avgcracking_vec, avgcomposite_vec, unit_cost_rows_local)

def simulate_network_mc_benefit_cost_aadt(
    gdf, gdf_merged, budgets, Treatment_Table,
    years=10, n_mc=100, seed=123,
    drop_geometry=True, return_unit_costs=False,
    n_jobs=15, backend="loky", verbose=0,
    seg_save_cfg=None
):
    """Run Monte Carlo for the Preservation (Considered AADT) policy."""
    if drop_geometry:
        gdf_base  = gdf.drop(columns=["geometry"], errors="ignore").copy(deep=True)
        gdfm_base = gdf_merged.drop(columns=["geometry"], errors="ignore").copy(deep=True)
    else:
        gdf_base  = gdf.copy(deep=True)
        gdfm_base = gdf_merged.copy(deep=True)

    if "lane_miles_cal" not in gdf_base.columns:
        gdf_base["lane_miles_cal"] = (
            (gdf_base["ToMeasure"] - gdf_base["FromMeasur"]) * gdf_base["Number_of_"]
        )

    gdf_base0  = drop_sim_columns(gdf_base)
    gdfm_base0 = drop_sim_columns(gdfm_base)

    # Generate initial PMS state ONCE with a fixed seed so every MC run starts
    # from the same network condition. Variation between runs comes only from
    # treatment cost sampling and performance model uncertainty (z_*_global).
    _init_rng = np.random.default_rng(seed)
    gdf_base0["pavement_age"] = _init_rng.integers(0, 21, size=len(gdf_base0))
    gdf_base0["No_of_Rehab"]  = _init_rng.integers(0, 4,  size=len(gdf_base0))
    gdf_base0["SCHD_Year"]    = _init_rng.integers(0, 10, size=len(gdf_base0))
    gdfm_base0["No_of_Rehab"] = _init_rng.integers(0, 4,  size=len(gdfm_base0))
    gdfm_base0["SCHD_Year"]   = _init_rng.integers(0, 10, size=len(gdfm_base0))

    mc_results   = {}
    avgiri_mc    = {}
    cost_mc      = {}
    rutting_mc   = {}
    cracking_mc  = {}
    composite_mc = {}
    unit_cost_rows = []

    for bi, budget in enumerate(budgets):
        results = Parallel(n_jobs=n_jobs, backend=backend, verbose=verbose)(
            delayed(_mc_worker_benefit_cost_aadt)(
                bi=bi, budget=int(budget), k=k, years=years, seed=seed,
                gdf_base0=gdf_base0, gdfm_base0=gdfm_base0,
                return_unit_costs=return_unit_costs,
                seg_save_cfg=seg_save_cfg
            )
            for k in range(n_mc)
        )

        rows          = []
        avg_mat       = np.zeros((n_mc, years), dtype=float)
        cost_mat      = np.zeros((n_mc, years), dtype=float)
        rutting_mat   = np.zeros((n_mc, years), dtype=float)
        cracking_mat  = np.zeros((n_mc, years), dtype=float)
        composite_mat = np.zeros((n_mc, years), dtype=float)

        for (_budget, k, row, avgiri_vec, cost_vec,
             avgrutting_vec, avgcracking_vec, avgcomposite_vec, uc_rows_local) in results:
            rows.append(row)
            avg_mat[k, :]       = avgiri_vec
            cost_mat[k, :]      = cost_vec
            rutting_mat[k, :]   = avgrutting_vec
            cracking_mat[k, :]  = avgcracking_vec
            composite_mat[k, :] = avgcomposite_vec
            if return_unit_costs:
                unit_cost_rows.extend(uc_rows_local)

        mc_results[int(budget)]   = pd.DataFrame(rows).sort_values(["iter"]).reset_index(drop=True)
        avgiri_mc[int(budget)]    = avg_mat
        cost_mc[int(budget)]      = cost_mat
        rutting_mc[int(budget)]   = rutting_mat
        cracking_mc[int(budget)]  = cracking_mat
        composite_mc[int(budget)] = composite_mat

    unit_cost_df = pd.DataFrame(unit_cost_rows) if return_unit_costs else None
    return mc_results, avgiri_mc, cost_mc, rutting_mc, cracking_mc, composite_mc, unit_cost_df

print("simulate_network_mc_benefit_cost_aadt defined.")

## Section 10: Strategy Run Blocks

Each enabled block performs a single-budget simulation and a multi-budget simulation. Within every block, the four convergence figures are shown first:

1. Running mean by simulation year.
2. Running CoV by simulation year.
3. Running mean averaged over years.
4. Running CoV averaged over years.

The remaining condition, user-cost, CoV-table, multi-budget, violin, and CSV outputs follow those convergence figures.


In [ ]:
# ============================================================
# Shared plotting and export helpers
# ============================================================
# Computational summaries match the source notebooks. Formatting follows
# the shared constants copied from Code.ipynb in the import section.

def _resolve_strategy_style(policy_label):
    """Return the standard color and marker for a policy label."""
    aliases = {
        "Worst-first": "Worst-first",
        "Benefit-Cost / AUC": "Benefit-Cost / AUC",
        "Preservation": "Benefit-Cost / AUC",
        "Benefit-Cost / AUC with AADT": "Benefit-Cost / AUC with AADT",
        "Preservation (Considered AADT)": "Benefit-Cost / AUC with AADT",
    }
    return STRATEGY_STYLES[aliases[policy_label]]

def _format_plot(
    xlabel,
    ylabel,
    *,
    title=None,
    xticks=None,
    ylim=None,
    legend=False,
    legend_title=None,
    legend_fontsize=PLOT_LEGEND_SIZE,
):
    """Apply the Code.ipynb typography, axes, legend, and layout standard."""
    plt.xlabel(xlabel, fontsize=PLOT_LABEL_SIZE, fontweight="bold")
    plt.ylabel(ylabel, fontsize=PLOT_LABEL_SIZE, fontweight="bold")
    if title is not None:
        plt.title(title, fontsize=PLOT_TITLE_SIZE)
    if xticks is not None:
        plt.xticks(xticks, fontsize=PLOT_TICK_SIZE)
    else:
        plt.xticks(fontsize=PLOT_TICK_SIZE)
    plt.yticks(fontsize=PLOT_TICK_SIZE)
    if ylim is not None:
        plt.ylim(*ylim)
    if legend:
        plt.legend(
            title=legend_title,
            fontsize=legend_fontsize,
            title_fontsize=PLOT_LEGEND_TITLE_SIZE if legend_title else None,
        )
    plt.tight_layout()
    plt.show()

def _compute_running_convergence(avgiri_mc_dict, budget):
    """
    Compute running mean and coefficient of variation by simulation year.

    Parameters
    ----------
    avgiri_mc_dict : dict
        Mapping from budget to an `(iterations, years)` weighted-AvgIRI array.
    budget : int or float
        Budget key to evaluate.

    Returns
    -------
    dict
        Sample counts and running mean, standard deviation, and CoV arrays.
    """
    avgiri_mat = np.asarray(avgiri_mc_dict[budget], dtype=float)
    n_mc, years = avgiri_mat.shape
    m_vals = np.arange(1, n_mc + 1)
    running_mean = np.zeros((n_mc, years), dtype=float)
    running_std = np.zeros((n_mc, years), dtype=float)
    running_cov = np.zeros((n_mc, years), dtype=float)
    eps = 1e-12

    for m in range(1, n_mc + 1):
        sample = avgiri_mat[:m, :]
        mean = sample.mean(axis=0)
        std = sample.std(axis=0, ddof=1) if m > 1 else np.zeros(years)
        running_mean[m - 1, :] = mean
        running_std[m - 1, :] = std
        running_cov[m - 1, :] = std / np.maximum(np.abs(mean), eps)

    return {
        "m_vals": m_vals,
        "running_mean": running_mean,
        "running_std": running_std,
        "running_cov": running_cov,
        "running_mean_over_years": running_mean.mean(axis=1),
        "running_cov_over_years": running_cov.mean(axis=1),
    }

def _plot_convergence(avgiri_mc_dict, budget, policy_label):
    """
    Plot the first two convergence figures: running mean and CoV by year.

    Parameters
    ----------
    avgiri_mc_dict : dict
        Budget-indexed Monte Carlo AvgIRI matrices.
    budget : int or float
        Budget selected for convergence evaluation.
    policy_label : str
        Strategy name displayed in the figure titles.
    """
    convergence = _compute_running_convergence(avgiri_mc_dict, budget)
    m_vals = convergence["m_vals"]
    running_mean = convergence["running_mean"]
    running_cov = convergence["running_cov"]
    years = running_mean.shape[1]

    plt.figure(figsize=PLOT_FIGSIZE)
    for year in range(years):
        plt.plot(
            m_vals,
            running_mean[:, year],
            linewidth=PLOT_LINEWIDTH,
            label=f"Year {year + 1}",
        )
    plt.grid(True)
    _format_plot(
        "Number of MC samples used (m)",
        "Running mean of weighted AvgIRI",
        title=f"[{policy_label}] Convergence of mean by year — budget={budget:,}",
        legend=True,
        legend_fontsize=8,
    )

    plt.figure(figsize=PLOT_FIGSIZE)
    for year in range(years):
        plt.plot(
            m_vals,
            running_cov[:, year],
            linewidth=PLOT_LINEWIDTH,
            label=f"Year {year + 1}",
        )
    plt.grid(True)
    _format_plot(
        "Number of MC samples used (m)",
        "Running CoV of weighted AvgIRI (std/mean)",
        title=f"[{policy_label}] Convergence of CoV by year — budget={budget:,}",
        legend=True,
        legend_fontsize=8,
    )

def _plot_scalar_convergence(avgiri_mc_dict, budget, strategy_name):
    """
    Plot running mean and CoV averaged across all simulation years.

    Parameters
    ----------
    avgiri_mc_dict : dict
        Budget-indexed Monte Carlo AvgIRI matrices.
    budget : int or float
        Budget selected for convergence evaluation.
    strategy_name : str
        Canonical strategy key used to select the standard color.

    Returns
    -------
    dict
        `m_vals`, `running_mean_over_years`, and `running_cov_over_years`.
    """
    convergence = _compute_running_convergence(avgiri_mc_dict, budget)
    style = STRATEGY_STYLES[strategy_name]
    m_vals = convergence["m_vals"]

    plt.figure(figsize=PLOT_FIGSIZE)
    plt.plot(
        m_vals,
        convergence["running_mean_over_years"],
        color=style["color"],
        linewidth=PLOT_LINEWIDTH,
    )
    plt.grid(True)
    _format_plot(
        "Number of MC samples used (m)",
        "Running mean of weighted AvgIRI (avg over years)",
        title=f"Convergence of mean (Avg over years) — budget={budget:,}",
    )

    plt.figure(figsize=PLOT_FIGSIZE)
    plt.plot(
        m_vals,
        convergence["running_cov_over_years"],
        color=style["color"],
        linewidth=PLOT_LINEWIDTH,
    )
    plt.grid(True)
    _format_plot(
        "Number of MC samples used (m)",
        "Running CoV of weighted AvgIRI (avg over years)",
        title=f"Convergence of CoV (Avg over years) — budget={budget:,}",
    )

    return {
        "m_vals": m_vals,
        "running_mean_over_years": convergence["running_mean_over_years"],
        "running_cov_over_years": convergence["running_cov_over_years"],
    }

def _plot_avgiri_fan(avgiri_mc_dict, budget, policy_label):
    """Plot mean weighted AvgIRI and its P10–P90 interval over time."""
    avgiri_mat = np.asarray(avgiri_mc_dict[budget], dtype=float)
    years_axis = np.arange(1, avgiri_mat.shape[1] + 1)
    low, high = PLOT_PERCENTILE_BAND
    mean_avgiri = np.nanmean(avgiri_mat, axis=0)
    p_low = np.nanpercentile(avgiri_mat, low, axis=0)
    p_high = np.nanpercentile(avgiri_mat, high, axis=0)
    style = _resolve_strategy_style(policy_label)

    plt.figure(figsize=PLOT_FIGSIZE)
    plt.fill_between(
        years_axis,
        p_low,
        p_high,
        color=style["color"],
        alpha=PLOT_BAND_ALPHA,
        label=f"P{low}–P{high}",
    )
    plt.plot(
        years_axis,
        mean_avgiri,
        color=style["color"],
        marker=style["marker"],
        linewidth=PLOT_LINEWIDTH,
        markersize=PLOT_MARKERSIZE,
        label="Mean weighted AvgIRI",
    )
    plt.grid(True)
    _format_plot(
        "Year",
        "Average Weighted IRI (inch/mi)",
        title=f"[{policy_label}] Average Weighted IRI at $440M Budget",
        xticks=years_axis,
        ylim=(60, 140),
        legend=True,
    )

def _plot_user_cost(mc_results_dict, budget, policy_label):
    """Plot mean annual user cost and its P10–P90 interval."""
    df = mc_results_dict[budget]
    year_cols = sorted(
        [column for column in df.columns if re.fullmatch(r"user_cost_y\d+", column)],
        key=lambda column: int(column.split("y")[1]),
    )
    user_cost_matrix = df[year_cols].to_numpy(dtype=float)
    years_axis = np.arange(1, len(year_cols) + 1)
    low, high = PLOT_PERCENTILE_BAND
    mean = np.nanmean(user_cost_matrix, axis=0)
    p_low = np.nanpercentile(user_cost_matrix, low, axis=0)
    p_high = np.nanpercentile(user_cost_matrix, high, axis=0)
    style = _resolve_strategy_style(policy_label)

    plt.figure(figsize=PLOT_FIGSIZE)
    plt.fill_between(
        years_axis,
        p_low,
        p_high,
        color=style["color"],
        alpha=PLOT_BAND_ALPHA,
        label=f"P{low}–P{high}",
    )
    plt.plot(
        years_axis,
        mean,
        color=style["color"],
        marker=style["marker"],
        linewidth=PLOT_LINEWIDTH,
        markersize=PLOT_MARKERSIZE,
        label="Mean user cost",
    )
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    _format_plot(
        "Year",
        "User Cost (Million $)",
        title=f"[{policy_label}] User Cost at $440M Budget",
        xticks=years_axis,
        ylim=(80, 200),
        legend=True,
    )

def _plot_cov_table(avgiri_mc_dict, budget, policy_label):
    """Display the sample CoV of weighted AvgIRI for each simulation year."""
    avgiri_mat = np.asarray(avgiri_mc_dict[budget], dtype=float)
    years = avgiri_mat.shape[1]
    avgiri_df = pd.DataFrame(
        avgiri_mat,
        columns=[f"Year_{year + 1}" for year in range(years)],
    )
    cov_per_year = avgiri_df.std(axis=0, ddof=1) / avgiri_df.mean(axis=0)
    cov_df = pd.DataFrame({"CoV": cov_per_year})
    cov_df.index.name = "Year"
    print(f"[{policy_label}] CoV of weighted AvgIRI per year (budget={budget:,}):")
    display(cov_df)

def _plot_metric_fan(metric_mc_dict, budget, policy_label,
                     ylabel, metric_label, ylim=None):
    """
    Plot mean and P10–P90 band for any lane-mile-weighted MC metric.

    Mirrors _plot_avgiri_fan but accepts any (n_mc, years) matrix.

    Parameters
    ----------
    metric_mc_dict : dict
        Budget-indexed Monte Carlo metric matrices, shape (n_mc, years).
    budget : int or float
        Budget key to select from the dict.
    policy_label : str
        Strategy name for the title and color lookup.
    ylabel : str
        Vertical-axis label.
    metric_label : str
        Short metric name used in the legend and title.
    ylim : tuple or None
        Vertical-axis limits; None lets matplotlib auto-scale.
    """
    metric_mat = np.asarray(metric_mc_dict[budget], dtype=float)
    years_axis = np.arange(1, metric_mat.shape[1] + 1)
    low, high = PLOT_PERCENTILE_BAND
    mean_val = np.nanmean(metric_mat, axis=0)
    p_low = np.nanpercentile(metric_mat, low, axis=0)
    p_high = np.nanpercentile(metric_mat, high, axis=0)
    style = _resolve_strategy_style(policy_label)

    plt.figure(figsize=PLOT_FIGSIZE)
    plt.fill_between(
        years_axis,
        p_low,
        p_high,
        color=style["color"],
        alpha=PLOT_BAND_ALPHA,
        label=f"P{low}–P{high}",
    )
    plt.plot(
        years_axis,
        mean_val,
        color=style["color"],
        marker=style["marker"],
        linewidth=PLOT_LINEWIDTH,
        markersize=PLOT_MARKERSIZE,
        label=f"Mean {metric_label}",
    )
    plt.grid(True)
    _format_plot(
        "Year",
        ylabel,
        title=f"[{policy_label}] {metric_label} at ${budget / 1e6:.0f}M Budget",
        xticks=years_axis,
        ylim=ylim,
        legend=True,
    )

def _plot_multi_budget_avgiri(mc_results_dict, budgets, years, policy_label):
    """Plot mean weighted AvgIRI and P10–P90 intervals for every budget."""
    years_axis = np.arange(1, years + 1)
    low, high = PLOT_PERCENTILE_BAND
    plt.figure(figsize=PLOT_FIGSIZE)

    for index, budget in enumerate(budgets):
        df = mc_results_dict[budget]
        columns = [f"avgiri_y{year}" for year in range(years)]
        if not all(column in df.columns for column in columns):
            continue
        values = df[columns].to_numpy(dtype=float)
        color = PLOT_PALETTE[index % len(PLOT_PALETTE)]
        marker = MULTI_BUDGET_MARKERS[index % len(MULTI_BUDGET_MARKERS)]
        plt.fill_between(
            years_axis,
            np.nanpercentile(values, low, axis=0),
            np.nanpercentile(values, high, axis=0),
            color=color,
            alpha=PLOT_BAND_ALPHA,
        )
        plt.plot(
            years_axis,
            np.nanmean(values, axis=0),
            color=color,
            marker=marker,
            linewidth=PLOT_LINEWIDTH,
            markersize=PLOT_MARKERSIZE,
            label=f"Budget ${budget / 1e6:.0f}M",
        )

    plt.grid(True)
    _format_plot(
        "Year",
        "Average Weighted IRI (inch/mi)",
        title=f"Budget Sensitivity: Average Weighted IRI — {policy_label}",
        xticks=years_axis,
        ylim=(40, 180),
        legend=True,
        legend_title=f"Budget Level: {policy_label}",
    )

def _plot_multi_budget_user_cost(mc_results_dict, budgets, years, policy_label):
    """Plot mean user cost and P10–P90 intervals for every budget."""
    years_axis = np.arange(1, years + 1)
    low, high = PLOT_PERCENTILE_BAND
    plt.figure(figsize=PLOT_FIGSIZE)

    for index, budget in enumerate(budgets):
        df = mc_results_dict[budget]
        columns = [f"user_cost_y{year}" for year in range(1, years + 1)]
        if not all(column in df.columns for column in columns):
            continue
        values = df[columns].to_numpy(dtype=float)
        color = PLOT_PALETTE[index % len(PLOT_PALETTE)]
        marker = MULTI_BUDGET_MARKERS[index % len(MULTI_BUDGET_MARKERS)]
        plt.fill_between(
            years_axis,
            np.nanpercentile(values, low, axis=0),
            np.nanpercentile(values, high, axis=0),
            color=color,
            alpha=PLOT_BAND_ALPHA,
        )
        plt.plot(
            years_axis,
            np.nanmean(values, axis=0),
            color=color,
            marker=marker,
            linewidth=PLOT_LINEWIDTH,
            markersize=PLOT_MARKERSIZE,
            label=f"Budget ${budget / 1e6:.0f}M",
        )

    plt.grid(axis="y", linestyle="--", alpha=0.6)
    _format_plot(
        "Year",
        "User Cost (Million $)",
        title=f"Budget Sensitivity: User Cost — {policy_label}",
        xticks=years_axis,
        ylim=(60, 260),
        legend=True,
        legend_title=f"Budget Level: {policy_label}",
    )

def _plot_multi_budget_metric(
    mc_results_dict, budgets, years, policy_label,
    col_prefix, ylabel, metric_label, ylim=None,
):
    """
    Plot mean and P10–P90 band for a per-year column across all budget levels.

    Mirrors _plot_multi_budget_avgiri for any metric stored as
    ``{col_prefix}_y{year}`` columns in the result DataFrames.

    Parameters
    ----------
    mc_results_dict : dict
        Budget-indexed Monte Carlo result DataFrames.
    budgets : sequence
        Budget levels to plot.
    years : int
        Simulation horizon.
    policy_label : str
        Strategy name for the title.
    col_prefix : str
        Column prefix; columns are expected as ``{col_prefix}_y{year}``.
    ylabel : str
        Vertical-axis label.
    metric_label : str
        Short metric name for the figure title.
    ylim : tuple or None
        Vertical-axis limits; None lets matplotlib auto-scale.
    """
    years_axis = np.arange(1, years + 1)
    low, high = PLOT_PERCENTILE_BAND
    plt.figure(figsize=PLOT_FIGSIZE)

    for index, budget in enumerate(budgets):
        df = mc_results_dict[budget]
        columns = [f"{col_prefix}_y{year}" for year in range(years)]
        if not all(c in df.columns for c in columns):
            continue
        values = df[columns].to_numpy(dtype=float)
        color = PLOT_PALETTE[index % len(PLOT_PALETTE)]
        marker = MULTI_BUDGET_MARKERS[index % len(MULTI_BUDGET_MARKERS)]
        plt.fill_between(
            years_axis,
            np.nanpercentile(values, low, axis=0),
            np.nanpercentile(values, high, axis=0),
            color=color,
            alpha=PLOT_BAND_ALPHA,
        )
        plt.plot(
            years_axis,
            np.nanmean(values, axis=0),
            color=color,
            marker=marker,
            linewidth=PLOT_LINEWIDTH,
            markersize=PLOT_MARKERSIZE,
            label=f"Budget ${budget / 1e6:.0f}M",
        )

    plt.grid(True)
    _format_plot(
        "Year",
        ylabel,
        title=f"Budget Sensitivity: {metric_label} — {policy_label}",
        xticks=years_axis,
        ylim=ylim,
        legend=True,
        legend_title=f"Budget Level: {policy_label}",
    )

def _plot_final_year_avgiri_violin(mc_results_dict, budgets, years, strategy_name):
    """
    Plot final-year AvgIRI distributions by budget with mean and CoV labels.

    Parameters
    ----------
    mc_results_dict : dict
        Budget-indexed Monte Carlo result DataFrames.
    budgets : sequence
        Ordered budget levels shown on the x-axis.
    years : int
        Simulation horizon; the final column is `avgiri_y{years - 1}`.
    strategy_name : str
        Canonical strategy key used for color and title formatting.
    """
    final_column = f"avgiri_y{years - 1}"
    data_per_budget = [
        mc_results_dict[budget][final_column].to_numpy(dtype=float)
        for budget in budgets
    ]
    means = [np.nanmean(values) for values in data_per_budget]
    covs_percent = [
        np.nanstd(values) / mean * 100 if mean > 0 else np.nan
        for values, mean in zip(data_per_budget, means)
    ]
    style = STRATEGY_STYLES[strategy_name]

    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_VIOLIN)
    parts = ax.violinplot(data_per_budget, showmeans=True, showextrema=True)
    for body in parts["bodies"]:
        body.set_facecolor(style["color"])
        body.set_edgecolor("black")
        body.set_alpha(0.75)

    ax.set_xticks(
        range(1, len(budgets) + 1),
        [f"${budget / 1e6:.0f}M" for budget in budgets],
        fontsize=PLOT_TICK_SIZE,
    )
    ax.tick_params(axis="y", labelsize=PLOT_TICK_SIZE)
    ax.set_xlabel("Budget Level", fontsize=PLOT_LABEL_SIZE, fontweight="bold")
    ax.set_ylabel(
        "Average Weighted IRI for 10-Years (inch/mi)",
        fontsize=PLOT_LABEL_SIZE,
        fontweight="bold",
    )
    ax.set_title(
        f"{STRATEGY_STYLES[strategy_name]['label']}: Distribution of Final-Year Average Weighted IRI by Budget",
        fontsize=PLOT_TITLE_SIZE,
    )
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    all_values = np.concatenate(data_per_budget)
    y_min = np.nanmin(all_values)
    y_max = np.nanmax(all_values)
    y_range = y_max - y_min
    ax.set_ylim(y_min - 0.12 * y_range, y_max + 0.18 * y_range)

    for index, (mean, cov_percent, values) in enumerate(
        zip(means, covs_percent, data_per_budget),
        start=1,
    ):
        ax.text(
            index + 0.25,
            np.nanmax(values),
            f"Mean: {mean:.1f}\nCoV: {cov_percent:.1f}%",
            ha="left",
            va="top",
            fontsize=20,
            color="black",
            fontweight="bold",
            bbox={"facecolor": "white", "alpha": 0.6, "edgecolor": "none"},
        )

    plt.tight_layout()
    plt.show()

def _export_single_budget_csv(
    mc_results_dict,
    avgiri_mc_dict,
    rutting_mc_dict,
    cracking_mc_dict,
    composite_mc_dict,
    budget,
    n_mc,
    years,
    filename,
):
    """Export one row per Monte Carlo iteration and year for one budget.

    Columns: simulation, year, weighted_avg_iri, weighted_avg_rutting,
    weighted_avg_cracking, weighted_avg_composite, user_cost.
    """
    df = mc_results_dict[budget]
    avgiri_mat    = avgiri_mc_dict[budget]
    rutting_mat   = rutting_mc_dict[budget]
    cracking_mat  = cracking_mc_dict[budget]
    composite_mat = composite_mc_dict[budget]
    rows = []
    for simulation in range(n_mc):
        for year in range(1, years + 1):
            user_cost_column = f"user_cost_y{year}"
            rows.append(
                {
                    "simulation":             simulation,
                    "year":                   year,
                    "weighted_avg_iri":        avgiri_mat[simulation, year - 1],
                    "weighted_avg_rutting":    rutting_mat[simulation, year - 1],
                    "weighted_avg_cracking":   cracking_mat[simulation, year - 1],
                    "weighted_avg_composite":  composite_mat[simulation, year - 1],
                    "user_cost": (
                        df.loc[simulation, user_cost_column]
                        if user_cost_column in df.columns
                        else np.nan
                    ),
                }
            )
    result_df = pd.DataFrame(rows)
    result_df.to_csv(filename, index=False)
    print(f"Saved: {filename}  shape: {result_df.shape}")
    display(result_df.head())

def _export_multi_budget_csv(mc_results_dict, budgets, years, filename):
    """Export long-format annual results for all four metrics and all budgets.

    Columns: budget, simulation, year, weighted_avg_iri, weighted_avg_rutting,
    weighted_avg_cracking, weighted_avg_composite, user_cost.
    """
    rows = []
    for budget in budgets:
        df = mc_results_dict[budget].copy()
        iri_cols       = [f"avgiri_y{y}"       for y in range(years)]
        rutting_cols   = [f"avgrutting_y{y}"   for y in range(years)]
        cracking_cols  = [f"avgcracking_y{y}"  for y in range(years)]
        composite_cols = [f"avgcomposite_y{y}" for y in range(years)]
        user_cost_cols = [f"user_cost_y{y}"    for y in range(1, years + 1)]
        for row_index, result in df.iterrows():
            simulation = int(result["iter"]) + 1 if "iter" in df.columns else row_index + 1
            for year in range(years):
                rows.append(
                    {
                        "budget":                  budget,
                        "simulation":              simulation,
                        "year":                    year + 1,
                        "weighted_avg_iri":        result[iri_cols[year]],
                        "weighted_avg_rutting":    result[rutting_cols[year]] if rutting_cols[year] in df.columns else np.nan,
                        "weighted_avg_cracking":   result[cracking_cols[year]] if cracking_cols[year] in df.columns else np.nan,
                        "weighted_avg_composite":  result[composite_cols[year]] if composite_cols[year] in df.columns else np.nan,
                        "user_cost":               result[user_cost_cols[year]],
                    }
                )
    result_df = pd.DataFrame(rows)
    result_df.to_csv(filename, index=False)
    print(f"Saved: {filename}  shape: {result_df.shape}")
    display(result_df.head())

convergence_results = {}
print("Shared plotting and CSV export helpers defined.")

### Run Block: Option 1 — Worst-first

Produces `mc_results_wf` and `avgiri_mc_wf` for the configured single budget, plus `mc_results_wf_multi` for the multi-budget analysis.


In [ ]:
# ============================================================
# RUN BLOCK: WORST-FIRST POLICY
# Single-budget + plots + multi-budget + plots + CSV exports
# ============================================================
if RUN_WORST_FIRST:
    import time
    print("=" * 60)
    print("RUNNING: Worst-first Policy")
    print("=" * 60)

    _seg_cfg = {
        "enabled": bool(SAVE_SEGMENT_RESULTS),
        "base_dir": SEGMENT_RESULTS_DIR,
        "compression": SEGMENT_RESULTS_COMPRESSION,
        "cov": COV_IRI,
    }
    if SAVE_SEGMENT_RESULTS:
        _write_segments_parquet(gdf, SEGMENT_RESULTS_DIR, cov=COV_IRI)

    # ---- Single-budget simulation ----
    print(f"\nSingle-budget run: N_MC={N_MC}, Budget=${BUDGET:,}")
    _t0_wf = time.perf_counter()
    mc_results_wf, avgiri_mc_wf, cost_mc_wf, rutting_mc_wf, cracking_mc_wf, composite_mc_wf, unit_cost_df_wf = simulate_network_mc_worst_first(
        gdf, gdf_merged,
        budgets=[BUDGET],
        Treatment_Table=Treatment_Table,
        years=YEARS,
        n_mc=N_MC,
        seed=SEED,
        return_unit_costs=True,
        n_jobs=15,
        backend="loky",
        seg_save_cfg=_seg_cfg,
    )
    runtime_wf_single = time.perf_counter() - _t0_wf
    print(f"Single-budget run complete. Runtime: {runtime_wf_single:.1f} s")
    print("Results shape:", mc_results_wf[BUDGET].shape)

    # ========================================================
    # Convergence diagnostics — always display these four first
    # ========================================================
    # Figures 1–2 show running mean and CoV separately for each year.
    _plot_convergence(avgiri_mc_wf, BUDGET, "Worst-first")

    # Figures 3–4 average the running statistics across all years.
    convergence_results["worst_first"] = _plot_scalar_convergence(
        avgiri_mc_wf, BUDGET, "Worst-first"
    )

    # ---- Single-budget IRI and user-cost outputs ----
    _plot_avgiri_fan(avgiri_mc_wf, BUDGET, "Worst-first")
    _plot_user_cost(mc_results_wf, BUDGET, "Worst-first")
    _plot_cov_table(avgiri_mc_wf, BUDGET, "Worst-first")

    # ---- Single-budget rutting, cracking, and composite outputs ----
    _plot_metric_fan(
        rutting_mc_wf, BUDGET, "Worst-first",
        ylabel="Average Weighted Rutting",
        metric_label="Avg Rutting",
    )
    _plot_metric_fan(
        cracking_mc_wf, BUDGET, "Worst-first",
        ylabel="Average Weighted Fatigue Cracking (%)",
        metric_label="Avg Fatigue Cracking",
    )
    _plot_metric_fan(
        composite_mc_wf, BUDGET, "Worst-first",
        ylabel="Average Weighted Composite Condition Index",
        metric_label="Avg Composite Index",
    )

    # ---- CSV exports ----
    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    # ---- CSV: single budget ----
    _export_single_budget_csv(
        mc_results_wf, avgiri_mc_wf, rutting_mc_wf, cracking_mc_wf, composite_mc_wf,
        BUDGET, N_MC, YEARS,
        filename=Path(OUTPUT_DIR) / f"single_budget_simulation_worstfirst_cov{COV_IRI}.csv"
    )

    # ---- Multi-budget simulation ----
    print(f"\nMulti-budget run: N_MC_MULTI={N_MC_MULTI}, budgets={BUDGETS_MULTI}")
    mc_results_wf_multi, avgiri_mc_wf_multi, cost_mc_wf_multi, rutting_mc_wf_multi, cracking_mc_wf_multi, composite_mc_wf_multi, _ = simulate_network_mc_worst_first(
        gdf, gdf_merged,
        budgets=BUDGETS_MULTI,
        Treatment_Table=Treatment_Table,
        years=YEARS,
        n_mc=N_MC_MULTI,
        seed=SEED,
        return_unit_costs=False,
        n_jobs=15,
        backend="loky",
        seg_save_cfg=_seg_cfg,
    )
    print("Multi-budget run complete.")

    # ---- Multi-budget IRI and user-cost plots ----
    _plot_multi_budget_avgiri(mc_results_wf_multi, BUDGETS_MULTI, YEARS, "Worst-first")
    _plot_multi_budget_user_cost(mc_results_wf_multi, BUDGETS_MULTI, YEARS, "Worst-first")
    _plot_final_year_avgiri_violin(mc_results_wf_multi, BUDGETS_MULTI, YEARS, "Worst-first")

    # ---- Multi-budget rutting, cracking, and composite plots ----
    _plot_multi_budget_metric(
        mc_results_wf_multi, BUDGETS_MULTI, YEARS, "Worst-first",
        col_prefix="avgrutting",
        ylabel="Average Weighted Rutting",
        metric_label="Average Weighted Rutting",
    )
    _plot_multi_budget_metric(
        mc_results_wf_multi, BUDGETS_MULTI, YEARS, "Worst-first",
        col_prefix="avgcracking",
        ylabel="Average Weighted Fatigue Cracking (%)",
        metric_label="Average Weighted Fatigue Cracking",
    )
    _plot_multi_budget_metric(
        mc_results_wf_multi, BUDGETS_MULTI, YEARS, "Worst-first",
        col_prefix="avgcomposite",
        ylabel="Average Weighted Composite Condition Index",
        metric_label="Average Weighted Composite Index",
    )

    # ---- CSV: multi budget ----
    _export_multi_budget_csv(
        mc_results_wf_multi, BUDGETS_MULTI, YEARS,
        filename=Path(OUTPUT_DIR) / f"multi_budget_simulation_results_worstfirst_cov{COV_IRI}.csv"
    )
else:
    print("Skipping Worst-first (RUN_WORST_FIRST = 0)")

### Run Block: Option 2 — Preservation

Produces `mc_results_bc` and `avgiri_mc_bc` for the configured single budget, plus `mc_results_bc_multi` for the multi-budget analysis.


In [ ]:
# ============================================================
# RUN BLOCK: PRESERVATION POLICY
# Single-budget + plots + multi-budget + plots + CSV exports
# ============================================================
if RUN_BENEFIT_COST:
    import time
    print("=" * 60)
    print("RUNNING: Preservation Policy")
    print("=" * 60)

    _seg_cfg = {
        "enabled": bool(SAVE_SEGMENT_RESULTS),
        "base_dir": SEGMENT_RESULTS_DIR,
        "compression": SEGMENT_RESULTS_COMPRESSION,
        "cov": COV_IRI,
    }
    if SAVE_SEGMENT_RESULTS:
        _write_segments_parquet(gdf, SEGMENT_RESULTS_DIR, cov=COV_IRI)

    # ---- Single-budget simulation ----
    print(f"\nSingle-budget run: N_MC={N_MC}, Budget=${BUDGET:,}")
    _t0_bc = time.perf_counter()
    mc_results_bc, avgiri_mc_bc, cost_mc_bc, rutting_mc_bc, cracking_mc_bc, composite_mc_bc, unit_cost_df_bc = simulate_network_mc_benefit_cost(
        gdf, gdf_merged,
        budgets=[BUDGET],
        Treatment_Table=Treatment_Table,
        years=YEARS,
        n_mc=N_MC,
        seed=SEED,
        return_unit_costs=True,
        n_jobs=15,
        backend="loky",
        seg_save_cfg=_seg_cfg,
    )
    runtime_bc_single = time.perf_counter() - _t0_bc
    print(f"Single-budget run complete. Runtime: {runtime_bc_single:.1f} s")
    print("Results shape:", mc_results_bc[BUDGET].shape)

    # ========================================================
    # Convergence diagnostics — always display these four first
    # ========================================================
    # Figures 1–2 show running mean and CoV separately for each year.
    _plot_convergence(avgiri_mc_bc, BUDGET, "Preservation")

    # Figures 3–4 average the running statistics across all years.
    convergence_results["costbenefit_auc"] = _plot_scalar_convergence(
        avgiri_mc_bc, BUDGET, "Benefit-Cost / AUC"
    )

    # ---- Single-budget IRI and user-cost outputs ----
    _plot_avgiri_fan(avgiri_mc_bc, BUDGET, "Preservation")
    _plot_user_cost(mc_results_bc, BUDGET, "Preservation")
    _plot_cov_table(avgiri_mc_bc, BUDGET, "Preservation")

    # ---- Single-budget rutting, cracking, and composite outputs ----
    _plot_metric_fan(
        rutting_mc_bc, BUDGET, "Preservation",
        ylabel="Average Weighted Rutting",
        metric_label="Avg Rutting",
    )
    _plot_metric_fan(
        cracking_mc_bc, BUDGET, "Preservation",
        ylabel="Average Weighted Fatigue Cracking (%)",
        metric_label="Avg Fatigue Cracking",
    )
    _plot_metric_fan(
        composite_mc_bc, BUDGET, "Preservation",
        ylabel="Average Weighted Composite Condition Index",
        metric_label="Avg Composite Index",
    )

    # ---- CSV exports ----
    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    # ---- CSV: single budget ----
    _export_single_budget_csv(
        mc_results_bc, avgiri_mc_bc, rutting_mc_bc, cracking_mc_bc, composite_mc_bc,
        BUDGET, N_MC, YEARS,
        filename=Path(OUTPUT_DIR) / f"single_budget_simulation_benefit_cov{COV_IRI}.csv"
    )

    # ---- Multi-budget simulation ----
    print(f"\nMulti-budget run: N_MC_MULTI={N_MC_MULTI}, budgets={BUDGETS_MULTI}")
    mc_results_bc_multi, avgiri_mc_bc_multi, cost_mc_bc_multi, rutting_mc_bc_multi, cracking_mc_bc_multi, composite_mc_bc_multi, _ = simulate_network_mc_benefit_cost(
        gdf, gdf_merged,
        budgets=BUDGETS_MULTI,
        Treatment_Table=Treatment_Table,
        years=YEARS,
        n_mc=N_MC_MULTI,
        seed=SEED,
        return_unit_costs=False,
        n_jobs=15,
        backend="loky",
        seg_save_cfg=_seg_cfg,
    )
    print("Multi-budget run complete.")

    # ---- Multi-budget IRI and user-cost plots ----
    _plot_multi_budget_avgiri(mc_results_bc_multi, BUDGETS_MULTI, YEARS, "Preservation")
    _plot_multi_budget_user_cost(mc_results_bc_multi, BUDGETS_MULTI, YEARS, "Preservation")
    _plot_final_year_avgiri_violin(mc_results_bc_multi, BUDGETS_MULTI, YEARS, "Benefit-Cost / AUC")

    # ---- Multi-budget rutting, cracking, and composite plots ----
    _plot_multi_budget_metric(
        mc_results_bc_multi, BUDGETS_MULTI, YEARS, "Preservation",
        col_prefix="avgrutting",
        ylabel="Average Weighted Rutting",
        metric_label="Average Weighted Rutting",
    )
    _plot_multi_budget_metric(
        mc_results_bc_multi, BUDGETS_MULTI, YEARS, "Preservation",
        col_prefix="avgcracking",
        ylabel="Average Weighted Fatigue Cracking (%)",
        metric_label="Average Weighted Fatigue Cracking",
    )
    _plot_multi_budget_metric(
        mc_results_bc_multi, BUDGETS_MULTI, YEARS, "Preservation",
        col_prefix="avgcomposite",
        ylabel="Average Weighted Composite Condition Index",
        metric_label="Average Weighted Composite Index",
    )

    # ---- CSV: multi budget ----
    _export_multi_budget_csv(
        mc_results_bc_multi, BUDGETS_MULTI, YEARS,
        filename=Path(OUTPUT_DIR) / f"multi_budget_simulation_results_benefit_cov{COV_IRI}.csv"
    )
else:
    print("Skipping Preservation (RUN_BENEFIT_COST = 0)")

### Run Block: Option 3 — Preservation (Considered AADT)

Produces `mc_results_bca` and `avgiri_mc_bca` for the configured single budget, plus `mc_results_bca_multi` for the multi-budget analysis.


In [ ]:
# ============================================================
# RUN BLOCK: PRESERVATION (CONSIDERED AADT) POLICY
# Single-budget + plots + multi-budget + plots + CSV exports
# ============================================================
if RUN_BENEFIT_COST_AADT:
    import time
    print("=" * 60)
    print("RUNNING: Preservation (Considered AADT) Policy")
    print("=" * 60)

    _seg_cfg = {
        "enabled": bool(SAVE_SEGMENT_RESULTS),
        "base_dir": SEGMENT_RESULTS_DIR,
        "compression": SEGMENT_RESULTS_COMPRESSION,
        "cov": COV_IRI,
    }
    if SAVE_SEGMENT_RESULTS:
        _write_segments_parquet(gdf, SEGMENT_RESULTS_DIR, cov=COV_IRI)

    # ---- Single-budget simulation ----
    print(f"\nSingle-budget run: N_MC={N_MC}, Budget=${BUDGET:,}")
    _t0_bca = time.perf_counter()
    mc_results_bca, avgiri_mc_bca, cost_mc_bca, rutting_mc_bca, cracking_mc_bca, composite_mc_bca, unit_cost_df_bca = simulate_network_mc_benefit_cost_aadt(
        gdf, gdf_merged,
        budgets=[BUDGET],
        Treatment_Table=Treatment_Table,
        years=YEARS,
        n_mc=N_MC,
        seed=SEED,
        return_unit_costs=True,
        n_jobs=15,
        backend="loky",
        seg_save_cfg=_seg_cfg,
    )
    runtime_bca_single = time.perf_counter() - _t0_bca
    print(f"Single-budget run complete. Runtime: {runtime_bca_single:.1f} s")
    print("Results shape:", mc_results_bca[BUDGET].shape)

    # ========================================================
    # Convergence diagnostics — always display these four first
    # ========================================================
    # Figures 1–2 show running mean and CoV separately for each year.
    _plot_convergence(avgiri_mc_bca, BUDGET, "Preservation (Considered AADT)")

    # Figures 3–4 average the running statistics across all years.
    convergence_results["costbenefit_auc_aadt"] = _plot_scalar_convergence(
        avgiri_mc_bca, BUDGET, "Benefit-Cost / AUC with AADT"
    )

    # ---- Single-budget IRI and user-cost outputs ----
    _plot_avgiri_fan(avgiri_mc_bca, BUDGET, "Preservation (Considered AADT)")
    _plot_user_cost(mc_results_bca, BUDGET, "Preservation (Considered AADT)")
    _plot_cov_table(avgiri_mc_bca, BUDGET, "Preservation (Considered AADT)")

    # ---- Single-budget rutting, cracking, and composite outputs ----
    _plot_metric_fan(
        rutting_mc_bca, BUDGET, "Preservation (Considered AADT)",
        ylabel="Average Weighted Rutting",
        metric_label="Avg Rutting",
    )
    _plot_metric_fan(
        cracking_mc_bca, BUDGET, "Preservation (Considered AADT)",
        ylabel="Average Weighted Fatigue Cracking (%)",
        metric_label="Avg Fatigue Cracking",
    )
    _plot_metric_fan(
        composite_mc_bca, BUDGET, "Preservation (Considered AADT)",
        ylabel="Average Weighted Composite Condition Index",
        metric_label="Avg Composite Index",
    )

    # ---- CSV exports ----
    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    # ---- CSV: single budget ----
    _export_single_budget_csv(
        mc_results_bca, avgiri_mc_bca, rutting_mc_bca, cracking_mc_bca, composite_mc_bca,
        BUDGET, N_MC, YEARS,
        filename=Path(OUTPUT_DIR) / f"single_budget_simulation_benefit_aadt_cov{COV_IRI}.csv"
    )

    # ---- Multi-budget simulation ----
    print(f"\nMulti-budget run: N_MC_MULTI={N_MC_MULTI}, budgets={BUDGETS_MULTI}")
    mc_results_bca_multi, avgiri_mc_bca_multi, cost_mc_bca_multi, rutting_mc_bca_multi, cracking_mc_bca_multi, composite_mc_bca_multi, _ = simulate_network_mc_benefit_cost_aadt(
        gdf, gdf_merged,
        budgets=BUDGETS_MULTI,
        Treatment_Table=Treatment_Table,
        years=YEARS,
        n_mc=N_MC_MULTI,
        seed=SEED,
        return_unit_costs=False,
        n_jobs=15,
        backend="loky",
        seg_save_cfg=_seg_cfg,
    )
    print("Multi-budget run complete.")

    # ---- Multi-budget IRI and user-cost plots ----
    _plot_multi_budget_avgiri(mc_results_bca_multi, BUDGETS_MULTI, YEARS, "Preservation (Considered AADT)")
    _plot_multi_budget_user_cost(mc_results_bca_multi, BUDGETS_MULTI, YEARS, "Preservation (Considered AADT)")
    _plot_final_year_avgiri_violin(mc_results_bca_multi, BUDGETS_MULTI, YEARS, "Benefit-Cost / AUC with AADT")

    # ---- Multi-budget rutting, cracking, and composite plots ----
    _plot_multi_budget_metric(
        mc_results_bca_multi, BUDGETS_MULTI, YEARS, "Preservation (Considered AADT)",
        col_prefix="avgrutting",
        ylabel="Average Weighted Rutting",
        metric_label="Average Weighted Rutting",
    )
    _plot_multi_budget_metric(
        mc_results_bca_multi, BUDGETS_MULTI, YEARS, "Preservation (Considered AADT)",
        col_prefix="avgcracking",
        ylabel="Average Weighted Fatigue Cracking (%)",
        metric_label="Average Weighted Fatigue Cracking",
    )
    _plot_multi_budget_metric(
        mc_results_bca_multi, BUDGETS_MULTI, YEARS, "Preservation (Considered AADT)",
        col_prefix="avgcomposite",
        ylabel="Average Weighted Composite Condition Index",
        metric_label="Average Weighted Composite Index",
    )

    # ---- CSV: multi budget ----
    _export_multi_budget_csv(
        mc_results_bca_multi, BUDGETS_MULTI, YEARS,
        filename=Path(OUTPUT_DIR) / f"multi_budget_simulation_results_benefit_aadt_cov{COV_IRI}.csv"
    )
else:
    print("Skipping Preservation (Considered AADT) (RUN_BENEFIT_COST_AADT = 0)")

## Comparison of Prioritization Strategies

This section uses the in-memory results produced by the three run blocks; it does not rerun any simulation. It reports single-budget and multi-budget summary tables, compares annual weighted AvgIRI and user cost, and displays the final-year AvgIRI distributions by strategy and budget.


In [ ]:
# ============================================================
# Comparison of Prioritization Strategies
# ============================================================
# All objects below come from the three run blocks.
# The timing experiment at the end re-runs each strategy at small
# N_MC values to produce a runtime scaling plot.
import time

required_comparison_results = {
    "Worst-first": (mc_results_wf, mc_results_wf_multi),
    "Benefit-Cost / AUC": (mc_results_bc, mc_results_bc_multi),
    "Benefit-Cost / AUC with AADT": (mc_results_bca, mc_results_bca_multi),
}

# ---- Single-budget summary: one row per strategy ----
single_budget_summary_rows = []
for strategy in STRATEGY_ORDER:
    single_results, _ = required_comparison_results[strategy]
    result = single_results[BUDGET]
    final_iri   = result[f"avgiri_y{YEARS - 1}"].to_numpy(dtype=float)
    final_rut   = result[f"avgrutting_y{YEARS - 1}"].to_numpy(dtype=float)
    final_crack = result[f"avgcracking_y{YEARS - 1}"].to_numpy(dtype=float)
    final_comp  = result[f"avgcomposite_y{YEARS - 1}"].to_numpy(dtype=float)
    iri_mean   = np.nanmean(final_iri)
    rut_mean   = np.nanmean(final_rut)
    crack_mean = np.nanmean(final_crack)
    comp_mean  = np.nanmean(final_comp)
    single_budget_summary_rows.append(
        {
            "Strategy":                     STRATEGY_STYLES[strategy]["label"],
            "Budget":                       BUDGET,
            "Mean Total Agency Cost":        result["total_cost"].mean(),
            "Mean Total User Cost":          result["user_cost_total"].mean(),
            "Mean Final-Year Avg IRI":       iri_mean,
            "Final-Year IRI CoV (%)":        np.nanstd(final_iri,   ddof=1) / iri_mean   * 100 if iri_mean   > 0 else np.nan,
            "Mean Final-Year Avg Rutting":   rut_mean,
            "Final-Year Rutting CoV (%)":    np.nanstd(final_rut,   ddof=1) / rut_mean   * 100 if rut_mean   > 0 else np.nan,
            "Mean Final-Year Avg Cracking":  crack_mean,
            "Final-Year Cracking CoV (%)":   np.nanstd(final_crack, ddof=1) / crack_mean * 100 if crack_mean > 0 else np.nan,
            "Mean Final-Year Composite":     comp_mean,
            "Final-Year Composite CoV (%)":  np.nanstd(final_comp,  ddof=1) / comp_mean  * 100 if comp_mean  > 0 else np.nan,
        }
    )

comparison_summary = pd.DataFrame(single_budget_summary_rows)
print("Single-budget strategy comparison:")
display(comparison_summary.round(3))

# ---- Multi-budget summary: one row per strategy and budget ----
multi_budget_summary_rows = []
for strategy in STRATEGY_ORDER:
    _, multi_results = required_comparison_results[strategy]
    for budget in BUDGETS_MULTI:
        final_iri   = multi_results[budget][f"avgiri_y{YEARS - 1}"].to_numpy(dtype=float)
        final_rut   = multi_results[budget][f"avgrutting_y{YEARS - 1}"].to_numpy(dtype=float)
        final_crack = multi_results[budget][f"avgcracking_y{YEARS - 1}"].to_numpy(dtype=float)
        final_comp  = multi_results[budget][f"avgcomposite_y{YEARS - 1}"].to_numpy(dtype=float)
        iri_mean   = np.nanmean(final_iri)
        rut_mean   = np.nanmean(final_rut)
        crack_mean = np.nanmean(final_crack)
        comp_mean  = np.nanmean(final_comp)
        multi_budget_summary_rows.append(
            {
                "Strategy":                     STRATEGY_STYLES[strategy]["label"],
                "Budget":                       budget,
                "Mean Final-Year Avg IRI":       iri_mean,
                "Final-Year IRI CoV (%)":        np.nanstd(final_iri,   ddof=1) / iri_mean   * 100 if iri_mean   > 0 else np.nan,
                "Mean Final-Year Avg Rutting":   rut_mean,
                "Final-Year Rutting CoV (%)":    np.nanstd(final_rut,   ddof=1) / rut_mean   * 100 if rut_mean   > 0 else np.nan,
                "Mean Final-Year Avg Cracking":  crack_mean,
                "Final-Year Cracking CoV (%)":   np.nanstd(final_crack, ddof=1) / crack_mean * 100 if crack_mean > 0 else np.nan,
                "Mean Final-Year Composite":     comp_mean,
                "Final-Year Composite CoV (%)":  np.nanstd(final_comp,  ddof=1) / comp_mean  * 100 if comp_mean  > 0 else np.nan,
            }
        )

comparison_multi_budget_summary = pd.DataFrame(multi_budget_summary_rows)
print("Multi-budget final-year strategy comparison:")
display(comparison_multi_budget_summary.round(3))

def _plot_strategy_metric_comparison(
    result_map,
    column_builder,
    ylabel,
    y_limits,
    title,
):
    """
    Compare annual Monte Carlo distributions for all three strategies.

    Parameters
    ----------
    result_map : dict
        Strategy names mapped to `(single_budget, multi_budget)` result pairs.
    column_builder : callable
        Function that returns the result-column name for a zero-based year.
    ylabel : str
        Vertical-axis label.
    y_limits : tuple or None
        Lower and upper vertical-axis limits; None auto-scales.
    title : str
        Figure title.
    """
    years_axis = np.arange(1, YEARS + 1)
    low, high = PLOT_PERCENTILE_BAND
    plt.figure(figsize=PLOT_FIGSIZE)

    for strategy in STRATEGY_ORDER:
        single_results, _ = result_map[strategy]
        result = single_results[BUDGET]
        columns = [column_builder(year) for year in range(YEARS)]
        values = result[columns].to_numpy(dtype=float)
        style = STRATEGY_STYLES[strategy]
        plt.fill_between(
            years_axis,
            np.nanpercentile(values, low, axis=0),
            np.nanpercentile(values, high, axis=0),
            color=style["color"],
            alpha=PLOT_BAND_ALPHA,
        )
        plt.plot(
            years_axis,
            np.nanmean(values, axis=0),
            color=style["color"],
            marker=style["marker"],
            linewidth=PLOT_LINEWIDTH,
            markersize=PLOT_MARKERSIZE,
            label=style["label"],
        )

    plt.grid(True)
    _format_plot(
        "Year",
        ylabel,
        title=title,
        xticks=years_axis,
        ylim=y_limits,
        legend=True,
        legend_title="Strategy",
        legend_fontsize=20,
    )

# ---- Annual metric comparison plots (single budget) ----
_plot_strategy_metric_comparison(
    required_comparison_results,
    lambda year: f"avgiri_y{year}",
    "Average Weighted IRI (inch/mi)",
    (60, 140),
    "Average Weighted IRI at $440M Budget",
)
_plot_strategy_metric_comparison(
    required_comparison_results,
    lambda year: f"user_cost_y{year + 1}",
    "User Cost (Million $)",
    (80, 200),
    "User Cost at $440M Budget",
)
_plot_strategy_metric_comparison(
    required_comparison_results,
    lambda year: f"avgrutting_y{year}",
    "Average Weighted Rutting",
    None,
    "Average Weighted Rutting at $440M Budget",
)
_plot_strategy_metric_comparison(
    required_comparison_results,
    lambda year: f"avgcracking_y{year}",
    "Average Weighted Fatigue Cracking (%)",
    None,
    "Average Weighted Fatigue Cracking at $440M Budget",
)
_plot_strategy_metric_comparison(
    required_comparison_results,
    lambda year: f"avgcomposite_y{year}",
    "Average Weighted Composite Condition Index",
    None,
    "Average Weighted Composite Condition Index at $440M Budget",
)

# ---- Final-year multi-budget violin comparison (IRI), matching Code.ipynb ----
comparison_violin_rows = []
for strategy in STRATEGY_ORDER:
    _, multi_results = required_comparison_results[strategy]
    for budget in BUDGETS_MULTI:
        values = multi_results[budget][
            f"avgiri_y{YEARS - 1}"
        ].to_numpy(dtype=float)
        comparison_violin_rows.extend(
            {
                "Strategy": STRATEGY_STYLES[strategy]["label"],
                "Budget": f"${budget / 1e6:.0f}M",
                "Weighted AvgIRI": value,
            }
            for value in values
        )

comparison_violin_df = pd.DataFrame(comparison_violin_rows)
strategy_labels = [STRATEGY_STYLES[name]["label"] for name in STRATEGY_ORDER]
strategy_palette = {
    STRATEGY_STYLES[name]["label"]: STRATEGY_STYLES[name]["color"]
    for name in STRATEGY_ORDER
}

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_VIOLIN)
sns.violinplot(
    data=comparison_violin_df,
    x="Budget",
    y="Weighted AvgIRI",
    hue="Strategy",
    hue_order=strategy_labels,
    palette=strategy_palette,
    dodge=True,
    inner=None,
    cut=0,
    linewidth=1.5,
    ax=ax,
)
n_hue = len(STRATEGY_ORDER)
dodge_width = 0.8 / n_hue
offsets = [(j - (n_hue - 1) / 2) * dodge_width for j in range(n_hue)]
for bi, budget in enumerate(BUDGETS_MULTI):
    for si, strategy in enumerate(STRATEGY_ORDER):
        label = STRATEGY_STYLES[strategy]["label"]
        mask = (
            (comparison_violin_df["Budget"] == f"${budget / 1e6:.0f}M")
            & (comparison_violin_df["Strategy"] == label)
        )
        vals = comparison_violin_df.loc[mask, "Weighted AvgIRI"].to_numpy(dtype=float)
        mean_val = np.nanmean(vals)
        cov_val = np.nanstd(vals, ddof=1) / mean_val * 100 if mean_val > 0 else np.nan
        x_pos = bi + offsets[si]
        y_pos = np.nanmax(vals)
        ax.text(
            x_pos, y_pos,
            f"Mean: {mean_val:.1f}\nCoV: {cov_val:.1f}%",
            ha="center", va="bottom",
            fontsize=20, color="black", fontweight="bold",
            bbox={"facecolor": "white", "alpha": 0.6, "edgecolor": "none"},
        )
y_min = comparison_violin_df["Weighted AvgIRI"].min()
y_max = comparison_violin_df["Weighted AvgIRI"].max()
y_range = y_max - y_min
ax.set_ylim(y_min - 0.12 * y_range, y_max + 0.18 * y_range)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_xlabel("Budget Level", fontsize=PLOT_LABEL_SIZE, fontweight="bold")
ax.set_ylabel("Average Weighted IRI for 10-Years (inch/mi)", fontsize=PLOT_LABEL_SIZE, fontweight="bold")
ax.set_title("Distribution of Final-Year Average Weighted IRI by Budget and Strategy", fontsize=PLOT_TITLE_SIZE)
ax.tick_params(axis="x", labelsize=PLOT_TICK_SIZE)
ax.tick_params(axis="y", labelsize=PLOT_TICK_SIZE)
ax.legend(
    loc="lower left",
    bbox_to_anchor=(0.02, 0.02),
    fontsize=17,
    title="Strategy",
    title_fontsize=19,
)
plt.tight_layout()
plt.show()

# ============================================================
# Runtime Scaling Experiment (single-budget only)
# ============================================================
# Times each active strategy for the N_MC values in TIMING_N_MC_VALUES
# and plots a runtime vs. number-of-simulations comparison.
# ============================================================
# Runtime scaling derived from the single-budget runtimes already measured
# above — no extra simulations. Smaller N_MC points are estimated by linear
# scaling; the actual measured runtime at N_MC is the final anchor point.
_timing_n_vals = sorted(n for n in TIMING_N_MC_VALUES if n < N_MC) + [N_MC]

_strategy_ref_runtimes = {}
if RUN_WORST_FIRST:
    _strategy_ref_runtimes["Worst-first"] = runtime_wf_single
if RUN_BENEFIT_COST:
    _strategy_ref_runtimes["Benefit-Cost / AUC"] = runtime_bc_single
if RUN_BENEFIT_COST_AADT:
    _strategy_ref_runtimes["Benefit-Cost / AUC with AADT"] = runtime_bca_single

timing_data = {}
print(f"\nRuntime scaling estimated from single-budget runs (N_MC={N_MC}, no re-runs).")
for _strategy, _ref_time in _strategy_ref_runtimes.items():
    timing_data[_strategy] = [(_n, _ref_time * _n / N_MC) for _n in _timing_n_vals]
    print(f"  [{_strategy}] measured at N_MC={N_MC}: {_ref_time:.1f}s  (smaller points linearly estimated)")

# Runtime comparison plot
plt.figure(figsize=PLOT_FIGSIZE)
for _strategy in STRATEGY_ORDER:
    if _strategy not in timing_data:
        continue
    _style = STRATEGY_STYLES[_strategy]
    _n_vals = [t[0] for t in timing_data[_strategy]]
    _t_vals = [t[1] for t in timing_data[_strategy]]
    plt.plot(
        _n_vals, _t_vals,
        color=_style["color"],
        marker=_style["marker"],
        linewidth=PLOT_LINEWIDTH,
        markersize=PLOT_MARKERSIZE,
        label=_style["label"],
    )
plt.grid(True)
_format_plot(
    "Number of Monte Carlo Simulations",
    "Runtime (seconds)",
    title=f"Runtime Scaling: Single-Budget Simulation (measured at N_MC={N_MC}, smaller points estimated)",
    xticks=_timing_n_vals,
    legend=True,
    legend_title="Strategy",
    legend_fontsize=PLOT_LEGEND_SIZE,
)